# INF6083 - Projet P1
## Analyse du dataset Amazon Reviews 2023 - Books
### Équipe 7


## 0.1 Importation des bibliothèques

In [ ]:
import pandas as pd
import numpy as np
import polars as pl
import duckdb
import dask.dataframe as dd
from dask.distributed import Client, LocalCluster
import random
import gzip
import json
import matplotlib.pyplot as plt
import scipy.sparse as sp
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import KMeans
import networkx as nx
import time
import cudf
import cupy as cp
import rmm
import gc
import matplotlib.ticker as ticker
import seaborn as sns
import glob
from pathlib import Path
import os
import numpy as np
import matplotlib.cm as cm
from scipy.sparse import csr_matrix, coo_matrix
from sklearn.preprocessing import LabelEncoder
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score, silhouette_samples
import torch
from scipy.sparse import save_npz, load_npz
from itertools import combinations
from matplotlib.colors import TwoSlopeNorm
from scipy.cluster.hierarchy import linkage, dendrogram
import matplotlib.patches as mpatches
from collections import Counter
from networkx.algorithms import bipartite


## 0.2 Chargement des données (JSONL → Parquet)
Transformation du dataset en base Parquet pour un chargement rapide (10–50×). Fichier source : `data/Books.jsonl`.

In [ ]:
# Conversion JSONL → Parquet (DuckDB + Polars)
pl.scan_ndjson("data/Books.jsonl").sink_parquet("data/Books-polars.parquet")

con = duckdb.connect()
print("Converting JSONL → Parquet (this takes ~3-5 min)...")
con.execute("""
    COPY (
        SELECT *
        FROM read_json_auto(
            'data/Books.jsonl',
            format='newline_delimited',
            maximum_object_size=10485760
        )
    ) TO 'data/Books.parquet' (FORMAT PARQUET, ROW_GROUP_SIZE 1000000)
""")
print("Done!")
con.close()

# 3.1 Tâche 0 - Chargement et échantillonnage des données

## 3.1.1 Échantillonnage stratégique

### 1) Echantillonnage des utilisateurs actifs (≥ 20 reviews)
- Filtrage des utilisateurs actifs (≥ 20 reviews)
- Sélection aléatoire de 50,000 utilisateurs
- Conservation de toutes leurs interactions

In [ ]:
DATA_PATH = "data/Books.jsonl"
MIN_REVIEWS = 20
NUM_USERS = 50_000
SEED = 42

# ── Helper: flush everything ────────────────────────────────────
def flush_memory():
    """Aggressively free both RAM and VRAM."""
    # 1. Python garbage collector — release unreferenced objects
    gc.collect()
    
    # 2. CuPy memory pool — free cached GPU blocks
    mempool = cp.get_default_memory_pool()
    pinned_mempool = cp.get_default_pinned_memory_pool()
    mempool.free_all_blocks()
    pinned_mempool.free_all_blocks()
    
    # 3. Force CUDA synchronization (ensure all GPU ops finish first)
    cp.cuda.runtime.deviceSynchronize()

def print_memory_status(label=""):
    """Show current GPU memory usage."""
    import pynvml
    pynvml.nvmlInit()
    handle = pynvml.nvmlDeviceGetHandleByIndex(0)
    info = pynvml.nvmlDeviceGetMemoryInfo(handle)
    print(f"  [{label}] GPU: {info.used/1e9:.2f}/{info.total/1e9:.2f} GB "
          f"(free: {info.free/1e9:.2f} GB)")
    pynvml.nvmlShutdown()


# ── Configure RMM memory pool ──────────────────────────────────
rmm.reinitialize(
    managed_memory=True,
    pool_allocator=True,
)

print_memory_status("Before start")

# ================================================================
# PHASE 1: Load JSONL, count per user, find active users
# ================================================================
start = time.time()
print("Phase 1: Chargement en GPU memory...")

gdf = cudf.read_json(DATA_PATH, lines=True)
gdf['rating'] = gdf['rating'].astype('int8')

print_memory_status("After load")
print(f"  GPU DataFrame: {gdf.memory_usage(deep=True).sum() / 1e9:.2f} GB")

# Comptage sur GPU
user_counts = gdf['user_id'].value_counts()
active_users = user_counts[user_counts >= MIN_REVIEWS].index

# ── Transfer active user IDs to CPU immediately ────────────────
active_list = active_users.to_pandas().tolist()
print(f"  Active users: {len(active_list):,}")

# ── Sample 50,000 randomly (CPU) ───────────────────────────────
random.seed(SEED)
selected_users = random.sample(active_list, min(NUM_USERS, len(active_list)))

# ── FLUSH: delete the counts, we only need the user ID list now ─
del user_counts, active_users, active_list
flush_memory()
print_memory_status("After count flush")

# ================================================================
# PHASE 2: Filter the full DataFrame for sampled users
# ================================================================
print("\nPhase 2: Filtrage...")

# Convert back to cudf Series for GPU-side isin()
selected_series = cudf.Series(selected_users)
mask = gdf['user_id'].isin(selected_series)
sample_gdf = gdf[mask]

print(f"  Reviews matched: {len(sample_gdf):,}")

# ── FLUSH: delete the full DataFrame — we no longer need it ────
del gdf, mask, selected_series
flush_memory()
print_memory_status("After filter flush")

# ================================================================
# PHASE 3: Transfer to CPU and save
# ================================================================
print("\nPhase 3: Transfert vers CPU et sauvegarde...")

sample = sample_gdf.to_pandas()

# ── FLUSH: delete the GPU DataFrame — data is on CPU now ───────
del sample_gdf
flush_memory()
print_memory_status("After GPU->CPU flush")

elapsed = time.time() - start
print(f"\nTemps d'execution: {elapsed:.2f}s")
print(f"Reviews echantillonnees: {len(sample):,}")
print(f"Utilisateurs uniques: {sample['user_id'].nunique():,}")

# Save
sample.to_parquet('sample-cudf-activ-users/sample_gpu_active_users.parquet', compression='snappy')

# ── FINAL FLUSH: free everything including the pandas DataFrame ─
del sample
gc.collect()
print_memory_status("Final cleanup")

### 2) Echantillonage temporel
- Filtrage des reviews ayant eu lieu entre 2020-01-01 et 2023-12-31
- Sélection aléatoire de user ayant au moins 20 reviews
- Conservation de toutes les interactions


In [ ]:
DATA_PATH = "data/Books.jsonl"
CHUNK_SIZE = 2_000_000
MIN_REVIEWS = 20
NUM_USERS = 50_000
SEED = 42
TARGET_TOTAL = 2_000_000
TARGET_YEARS = [2020, 2021, 2022, 2023]


# Configuration mémoire GPU
rmm.reinitialize(
    managed_memory=True,  # Unified memory CPU/GPU
    pool_allocator=True
)

def sample_with_gpu(DATA_PATH):
    """
    Échantillonnage GPU-accéléré avec cuDF (RAPIDS)
    
    Performance: 10-50x plus rapide que pandas
    Requis: GPU NVIDIA avec 8GB+ VRAM
    
    """
    flush_memory()
    print("Chargement en GPU memory...")
    monitor_gpu_memory()
    gdf = cudf.read_json(DATA_PATH, lines=True)
    monitor_gpu_memory()
    
    gdf['rating'] = gdf['rating'].astype('int8')

    print(f"GPU memory utilisée: {gdf.memory_usage(deep=True).sum() / 1e9:.2f} GB")

    gdf['timestamp'] = cudf.to_datetime(gdf['timestamp'], unit='ms')
    gdf['year'] = gdf['timestamp'].dt.year

    # ── Year-by-year diagnostics ──────────────────────────────────
    gdf_target = gdf[gdf['year'].isin(TARGET_YEARS)]
    year_stats = (
        gdf_target.groupby('year')
        .agg({
            'user_id': ['count', 'nunique'],
            'rating': 'mean'
        })
    )
    # Transfer to pandas for display
    ys = year_stats.to_pandas()
    ys.columns = ['review_count', 'unique_users', 'avg_rating']
    ys = ys.sort_index()
    print(f"\n── Year-by-Year Breakdown ({TARGET_YEARS[0]}–{TARGET_YEARS[-1]}) ──")
    print(ys.to_string())
    print(f"\nTotal reviews: {ys['review_count'].sum():,}")
    print(f"Total unique users (with overlap): {ys['unique_users'].sum():,}")

    # ① Restrict to target period
    gdf_period = gdf[gdf['year'].isin(TARGET_YEARS)]
    del gdf
    monitor_gpu_memory()

    print(f"Reviews in {TARGET_YEARS[0]}–{TARGET_YEARS[-1]}: {len(gdf_period):,}")

    # ② Count reviews per user within the period
    user_counts = gdf_period['user_id'].value_counts().reset_index()
    user_counts.columns = ['user_id', 'review_count']

    # ③ Keep only users with >= MIN_REVIEWS in this period
    active_in_period = user_counts[user_counts['review_count'] >= MIN_REVIEWS]
    print(f"Active users (>= {MIN_REVIEWS} reviews in period): {len(active_in_period):,}")

    # ④ Sample up to 50,000 users
    active_list = active_in_period['user_id'].to_pandas().tolist()
    random.seed(SEED)
    n_to_sample = min(NUM_USERS, len(active_list))
    sampled_users = random.sample(active_list, n_to_sample)
    print(f"Sampled users: {n_to_sample:,} / {len(active_list):,}")

    # ⑤ Collect ALL their reviews in the period (no volume cap)
    sampled_series = cudf.Series(sampled_users)
    sample_gdf = gdf_period[gdf_period['user_id'].isin(sampled_series)]
    monitor_gpu_memory()

    sample = sample_gdf.to_pandas()
    print(f"Reviews: {len(sample):,} from {sample['user_id'].nunique():,} users")
    print(f"Avg reviews/user: {len(sample) / sample['user_id'].nunique():.1f}")
    del sample_gdf, gdf_period
    flush_memory()
    monitor_gpu_memory()  # after transferring to CPU and freeing GPU
    
    return sample
    
# Monitoring GPU
def monitor_gpu_memory():
    """Affiche utilisation GPU"""
    import pynvml
    
    pynvml.nvmlInit()
    handle = pynvml.nvmlDeviceGetHandleByIndex(0)
    info = pynvml.nvmlDeviceGetMemoryInfo(handle)
    
    print(f"GPU Memory: {info.used/1e9:.2f}/{info.total/1e9:.2f} GB")
    pynvml.nvmlShutdown()

# ── Helper: flush everything ────────────────────────────────────
def flush_memory():
    """Aggressively free both RAM and VRAM."""
    # 1. Python garbage collector — release unreferenced objects
    gc.collect()
    
    # 2. CuPy memory pool — free cached GPU blocks
    mempool = cp.get_default_memory_pool()
    pinned_mempool = cp.get_default_pinned_memory_pool()
    mempool.free_all_blocks()
    pinned_mempool.free_all_blocks()
    # 3. Force CUDA synchronization (ensure all GPU ops finish first)
    cp.cuda.runtime.deviceSynchronize()

# Utilisation
if __name__ == '__main__':
    import time
    start = time.time()
    sample = sample_with_gpu(DATA_PATH)
    elapsed = time.time() - start
    
    print(f"\n⚡ Temps d'exécution GPU: {elapsed:.2f}s")
    print(f"   Reviews échantillonnées: {len(sample):,}")
    
    # Sauvegarde
    sample.to_parquet('sample-cudf-tempor/sample_gpu_temporal.parquet', compression='snappy')

### 3) Justification

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# 3) JUSTIFICATION DE LA STRATÉGIE D'ÉCHANTILLONNAGE
#
# Ce document justifie les choix d'échantillonnage appliqués aux données
# Amazon Books 2023, selon trois critères : représentativité, volumétrie
# cible, et préservation de la structure des données.
#
# Deux stratégies complémentaires ont été implémentées :
#   • Stratégie 1 (cellule 8) : Utilisateurs actifs — 50 000 users ≥ 20 reviews
#   • Stratégie 2 (cellule 10) : Temporelle — période 2020–2023, users actifs
# ══════════════════════════════════════════════════════════════════════════

print("╔══════════════════════════════════════════════════════════════════════╗")
print("║  3) JUSTIFICATION DE LA STRATÉGIE D'ÉCHANTILLONNAGE                 ║")
print("╚══════════════════════════════════════════════════════════════════════╝")

print("""
  ══════════════════════════════════════════════════════════════════════
  A) REPRÉSENTATIVITÉ
  ══════════════════════════════════════════════════════════════════════

  L'objectif est d'obtenir un échantillon statistiquement représentatif
  du comportement des lecteurs sur Amazon Books, tout en restant
  exploitable pour les algorithmes de recommandation.

  ── Stratégie 1 : Utilisateurs actifs (≥ 20 reviews) ─────────────────

  • Justification du seuil MIN_REVIEWS = 20 :
    Les utilisateurs avec moins de 20 reviews fournissent un profil
    trop sparse pour le filtrage collaboratif : les similarités
    (cosinus, Pearson, Jaccard) sont instables et peu fiables.
    La littérature (Koren 2008, McAuley et al. 2015) recommande
    typiquement 15–25 reviews minimum pour des profils exploitables.
    Le seuil de 20 garantit un signal suffisant pour les mesures
    de similarité et le k-NN collaboratif.

  • Sélection aléatoire de 50 000 utilisateurs :
    Parmi les ~137 305 utilisateurs actifs (≥ 20 reviews) identifiés
    dans le dataset complet (~27M reviews, ~10,3M users), on tire
    aléatoirement 50 000 avec random.seed(42) pour reproductibilité.
    Ce tirage uniforme évite tout biais de sélection (pas de sur-
    représentation des « super-lecteurs » ni des niches thématiques).
    L'échantillon couvre ~36 % des utilisateurs actifs, ce qui assure
    une diversité suffisante de profils (mainstream, niche, éclectique).

  • Collecte exhaustive des reviews par utilisateur :
    Pour chaque utilisateur sélectionné, on conserve TOUTES ses
    reviews (pas de sous-échantillonnage intra-utilisateur). Cela
    préserve l'intégralité du profil de goûts et évite un biais
    temporel ou thématique artificiel.

  ── Stratégie 2 : Temporelle (2020–2023) ─────────────────────────────

  • Filtrage sur la période 2020–2023 :
    On restreint aux reviews récentes pour capturer les tendances
    actuelles du marché du livre et les comportements de notation
    les plus à jour. Les données antérieures à 2020 peuvent refléter
    des habitudes obsolètes (évolution des genres, des formats).

  • Utilisateurs actifs DANS LA PÉRIODE :
    Le comptage des reviews se fait UNIQUEMENT sur 2020–2023. Un
    utilisateur avec 50 reviews dont 15 dans la période est retenu
    s'il atteint ≥ 20 reviews dans cette fenêtre. Cela garantit
    des profils riches sur la période ciblée, adaptés aux analyses
    temporelles et à l'évaluation de modèles sur des données récentes.

  • Limite de représentativité temporelle :
    L'échantillon temporel ne préserve pas les profils complets
    historiques : un utilisateur peut n'avoir qu'une fraction de
    ses reviews dans l'échantillon. C'est un compromis volontaire
    pour privilégier la représentativité temporelle sur la
    représentativité des profils individuels.

  ══════════════════════════════════════════════════════════════════════
  B) VOLUMÉTRIE CIBLE (500K – 2M reviews)
  ══════════════════════════════════════════════════════════════════════

  La fourchette 500K–2M reviews est un compromis entre :
    - Suffisamment de données pour des analyses statistiquement
      robustes (distributions, similarités, clustering, prédictions)
    - Un volume gérable en mémoire pour des traitements interactifs
      (matrices CSR, graphes NetworkX, entraînement de modèles)

  ── Stratégie 1 : ~2,44M reviews ─────────────────────────────────────

  • Résultat : 2 442 267 reviews (ou 2 442 098 selon l'itération)
  • Position par rapport à la cible : légèrement AU-DESSUS de 2M
  • Justification du dépassement :
    La collecte exhaustive des reviews par utilisateur est prioritaire.
    Tronquer arbitrairement (ex. garder 40 reviews max par user)
    biaiserait les profils : on perdrait l'information sur les
    « power users » et on déformerait les distributions de degré.
    Un volume de ~2,4M reste raisonnable : la matrice CSR et le
    graphe biparti sont exploitables sur une machine standard
    (16–32 Go RAM). Pour ramener strictement dans 500K–2M, on
    pourrait réduire NUM_USERS à ~35 000–42 000 (volume estimé
    ~1,7M–2,0M) sans altérer la logique d'échantillonnage.

  ── Stratégie 2 : ~856K reviews ──────────────────────────────────────

  • Résultat : 856 620 reviews (18 097 utilisateurs)
  • Position par rapport à la cible : DANS la fourchette (500K–2M)
  • Le nombre d'utilisateurs (18 097) est inférieur à 50 000 car
    seuls 18 097 users ont ≥ 20 reviews dans la période 2020–2023.
    On a donc pris TOUS les utilisateurs actifs disponibles dans
    la fenêtre temporelle. Le volume résultant est naturellement
    dans la cible et adapté aux analyses temporelles.

  ══════════════════════════════════════════════════════════════════════
  C) PRÉSERVATION DE LA STRUCTURE DES DONNÉES
  ══════════════════════════════════════════════════════════════════════

  La structure du dataset Amazon Reviews comporte trois niveaux
  essentiels à préserver pour les analyses ultérieures :

  ── 1. Structure utilisateur → reviews ──────────────────────────────

  • Stratégie 1 : PRÉSERVÉE INTÉGRALEMENT
    Chaque utilisateur sélectionné conserve l'intégralité de ses
    reviews. Aucun sous-échantillonnage intra-utilisateur. Les
    profils sont complets pour le calcul des similarités, le
    clustering, et la prédiction.

  • Stratégie 2 : PARTIELLEMENT PRÉSERVÉE
    Un utilisateur ne conserve que ses reviews dans la période
    2020–2023. La relation historique (reviews avant 2020) est
    volontairement exclue. Acceptable pour des analyses centrées
    sur les tendances récentes.

  ── 2. Structure review → produit (parent_asin) ───────────────────────

  • Les deux stratégies : PRÉSERVÉE
    Chaque review conserve son lien vers le livre (parent_asin).
    Aucune agrégation ni perte de granularité. Les analyses
    produit-centriques (popularité, centralité, distribution des
    notes par livre) restent possibles.

  ── 3. Champs et types de données ─────────────────────────────────────

  • Colonnes conservées : user_id, parent_asin, asin, rating,
    timestamp, title, text, helpful_vote, verified_purchase, etc.
    Aucune projection (SELECT colonnes) qui supprimerait des champs.
  • Seule transformation : rating → int8 (sans perte, notes 1–5)
  • Timestamps : préservés en millisecondes Unix (stratégie 1) ou
    convertis en datetime pour le filtrage annuel (stratégie 2).
    La conversion cudf.to_datetime(..., unit='ms') est correcte
    pour le format Amazon.

  ── Garanties opérationnelles ────────────────────────────────────────

  • Gestion mémoire (flush_memory, RMM) : évite les troncatures
    silencieuses dues à l'OOM GPU lors du chargement ou du filtrage.
  • Seed fixe (SEED=42) : reproductibilité complète de l'échantillon.
  • Validation : le nombre de reviews et d'utilisateurs uniques
    est cohérent avec les attentes (sanity checks en aval).

  ══════════════════════════════════════════════════════════════════════
  RÉSUMÉ
  ══════════════════════════════════════════════════════════════════════

  ┌────────────────────┬─────────────────────┬─────────────────────┐
  │ Critère            │ Stratégie 1 (actifs)│ Stratégie 2 (temp.) │
  ├────────────────────┼─────────────────────┼─────────────────────┤
  │ Représentativité   │ Profils complets,   │ Période récente,    │
  │                    │ tirage uniforme     │ actifs dans fenêtre │
  │ Volumétrie         │ ~2,44M (légèrement  │ ~856K (dans cible)  │
  │                    │ > 2M, justifié)     │                     │
  │ Structure          │ Intégrale           │ Partielle (temp.)   │
  │ Utilisation        │ Similarités, k-NN,  │ Tendances, modèles  │
  │ recommandée        │ graphe, clustering  │ sur données récentes│
  └────────────────────┴─────────────────────┴─────────────────────┘
""")

print("✓ Justification documentée.")

## 3.1.2 Analyse exploratoire

- Statistiques descriptives
- Distribution des ratings
- Sparsité
- Visualisations

### 1) Statistiques de base :

#### A) Statistiques de base:

- Nombre total d’utilisateurs, de livres et d’évaluations
- Distribution des évaluations (histogramme)
- Nombre moyen d’évaluations par utilisateur et par livre
- Identification des 10 utilisateurs les plus actifs et des 10 livres les plus populaires

In [ ]:
gc.collect()
sns.set_theme(style="whitegrid")

# Pick whichever sample you want to analyze
SAMPLE_PATHS = sorted(glob.glob("sample-*/*.parquet"))


for path in SAMPLE_PATHS:
    print(f"\n{'=' * 60}")
    print(f"  {path}")
    print(f"{'=' * 60}")
    df = pd.read_parquet(path)
    print(f"  Reviews: {len(df):,}  |  Users: {df['user_id'].nunique():,}  |  Books: {df['parent_asin'].nunique():,}")
    if len(df)== 0:
        print("  Empty file")
        print(f"  {path}")
        continue

    n_users   = df["user_id"].nunique()
    n_books   = df["parent_asin"].nunique()
    n_reviews = len(df)

    print("=" * 50)
    print("       STATISTIQUES DE BASE DE L'ÉCHANTILLON")
    print("=" * 50)
    print(f"  Nombre total de reviews  : {n_reviews:>12,}")
    print(f"  Nombre d'utilisateurs    : {n_users:>12,}")
    print(f"  Nombre de livres (ASINs) : {n_books:>12,}")
    print(f"  Reviews / utilisateur    : {n_reviews / n_users:>12.1f}")
    print(f"  Reviews / livre          : {n_reviews / n_books:>12.1f}")
    print("=" * 50)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # ── 2a. Distribution of ratings ────────────────────────────────
    rating_counts = df["rating"].value_counts().sort_index()
    axes[0].bar(rating_counts.index, rating_counts.values, color="steelblue", edgecolor="white")
    axes[0].set_xlabel("Note (rating)")
    axes[0].set_ylabel("Nombre de reviews")
    axes[0].set_title("Distribution des notes")
    axes[0].set_xticks([1, 2, 3, 4, 5])
    for i, v in enumerate(rating_counts.values):
        axes[0].text(rating_counts.index[i], v + v * 0.02, f"{v:,}", ha="center", fontsize=9)

    # ── 2b. Distribution of reviews per user ───────────────────────
    reviews_per_user = df.groupby("user_id").size()
    axes[1].hist(reviews_per_user, bins=50, color="darkorange", edgecolor="white", log=True)
    axes[1].set_xlabel("Nombre de reviews par utilisateur")
    axes[1].set_ylabel("Nombre d'utilisateurs (log)")
    axes[1].set_title("Distribution des reviews par utilisateur")
    axes[1].axvline(reviews_per_user.mean(), color="red", linestyle="--", label=f"Moyenne: {reviews_per_user.mean():.1f}")
    axes[1].legend()

    # ── 2c. Distribution of reviews per book ───────────────────────
    reviews_per_book = df.groupby("parent_asin").size()
    axes[2].hist(reviews_per_book, bins=50, color="seagreen", edgecolor="white", log=True)
    axes[2].set_xlabel("Nombre de reviews par livre")
    axes[2].set_ylabel("Nombre de livres (log)")
    axes[2].set_title("Distribution des reviews par livre")
    axes[2].axvline(reviews_per_book.mean(), color="red", linestyle="--", label=f"Moyenne: {reviews_per_book.mean():.1f}")
    axes[2].legend()

    plt.tight_layout()
    plt.show()

    print("── Moyennes ──────────────────────────────────────")
    print(f"  Moyenne de reviews par utilisateur : {reviews_per_user.mean():.2f}")
    print(f"  Médiane de reviews par utilisateur : {reviews_per_user.median():.1f}")
    print(f"  Écart-type (utilisateur)           : {reviews_per_user.std():.2f}")
    print()
    print(f"  Moyenne de reviews par livre       : {reviews_per_book.mean():.2f}")
    print(f"  Médiane de reviews par livre       : {reviews_per_book.median():.1f}")
    print(f"  Écart-type (livre)                 : {reviews_per_book.std():.2f}")
    print()
    print(f"  Note moyenne globale               : {df['rating'].mean():.2f}")
    print(f"  Note médiane                       : {df['rating'].median():.1f}")

    # ── Top 10 utilisateurs les plus actifs ────────────────────────
    top_users = (
        df.groupby("user_id")
        .agg(
            nb_reviews=("rating", "size"),
            note_moyenne=("rating", "mean"),
            helpful_total=("helpful_vote", "sum"),
        )
        .sort_values("nb_reviews", ascending=False)
        .head(10)
    )
    top_users["note_moyenne"] = top_users["note_moyenne"].round(2)

    print("── Top 10 utilisateurs les plus actifs ───────────")
    print(top_users.to_string())
    print()

    # ── Top 10 livres les plus appréciés ──────────────────────────
    # (highest average rating with at least 10 reviews to avoid noise)
    MIN_REVIEWS_BOOK = 10
    book_stats = (
        df.groupby("parent_asin")
        .agg(
            nb_reviews=("rating", "size"),
            note_moyenne=("rating", "mean"),
            helpful_total=("helpful_vote", "sum"),
            titre_exemple=("title", "first"),
        )
    )
    qualified_books = book_stats[book_stats["nb_reviews"] >= MIN_REVIEWS_BOOK]

    top_liked = qualified_books.sort_values("note_moyenne", ascending=False).head(10)
    top_liked["note_moyenne"] = top_liked["note_moyenne"].round(2)

    print(f"── Top 10 livres les plus appréciés (>= {MIN_REVIEWS_BOOK} reviews) ──")
    print(top_liked[["titre_exemple", "nb_reviews", "note_moyenne", "helpful_total"]].to_string())
    print()

    # ── Top 10 livres les plus reviewés ───────────────────────────
    top_reviewed = book_stats.sort_values("nb_reviews", ascending=False).head(10)
    top_reviewed["note_moyenne"] = top_reviewed["note_moyenne"].round(2)

    print("── Top 10 livres les plus reviewés ──────────────")
    print(top_reviewed[["titre_exemple", "nb_reviews", "note_moyenne", "helpful_total"]].to_string())



**Justification et explication des visualisations :**

**1. Popularite des livres — Longue traine (log-log scatter plot)**
Ce graphique represente le nombre de reviews par livre, classe par rang decroissant, sur une echelle log-log. Il met en evidence le phenomene de *longue traine* (long tail) : une tres petite minorite de livres concentre l'essentiel des reviews, tandis que la grande majorite des livres n'en a que tres peu. Ce phenomene est fondamental pour les systemes de recommandation car il implique que la plupart des items souffrent d'un probleme de *cold-start* (peu ou pas de donnees disponibles pour generer des recommandations fiables).

**2. Distribution temporelle des evaluations (line chart)**
Ce graphique montre le volume mensuel de reviews au fil du temps. Il permet d'identifier des tendances (croissance/decroissance de l'activite), des saisonnalites (pics durant les fetes, les soldes) et des evenements ponctuels. C'est crucial pour valider que l'echantillon couvre bien la plage temporelle attendue et pour decider si une stratification temporelle est necessaire lors de l'entrainement du modele.

**3. Distribution des votes d'utilite (histogram, echelle log)**
Ce graphique montre la distribution du nombre de votes "helpful" par review, en excluant les zeros pour une meilleure lisibilite. La tres grande majorite des reviews ne recoit aucun vote d'utilite, et parmi celles qui en recoivent, la distribution est fortement asymetrique (skewed). Ce champ pourrait servir de signal de qualite pour ponderer les reviews dans un systeme de recommandation, mais sa forte asymetrie impose un traitement adapte (ex. transformation log, binarisation).

**4. Proportion d'achats verifies (pie chart)**
Ce graphique montre la repartition entre achats verifies et non verifies. Les achats verifies sont plus fiables car ils confirment que l'utilisateur a reellement achete le produit. Une forte proportion d'achats verifies renforce la credibilite du dataset. Cette information pourrait aussi etre utilisee comme feature ou comme filtre pour ameliorer la qualite des donnees d'entrainement.

#### B) Comparaison des echantillons

In [ ]:
sns.set_theme(style="whitegrid")

SAMPLE_PATHS = sorted(glob.glob("sample-*/*.parquet", recursive=False))

# Collect summary stats + per-sample distributions
summary = []
distributions = {}

for path in SAMPLE_PATHS:
    if os.path.getsize(path) < 1024:  # skip files smaller than 1KB
        print(f"  Skipping {path} (only {os.path.getsize(path)} bytes, likely corrupt)")
        continue
    df = pd.read_parquet(path)
    print(f"  {path}: {len(df)} rows, columns: {list(df.columns)}")    
    label = Path(path).parent.name  # e.g. "sample-cudf-claude"
    if len(df) == 0:
        continue

    reviews_per_user = df.groupby("user_id").size()
    reviews_per_book = df.groupby("parent_asin").size()

    summary.append({
        "sample": label,
        "file": Path(path).stem,
        "n_reviews": len(df),
        "n_users": df["user_id"].nunique(),
        "n_books": df["parent_asin"].nunique(),
        "avg_rating": df["rating"].mean(),
        "median_rating": df["rating"].median(),
        "avg_reviews_per_user": reviews_per_user.mean(),
        "median_reviews_per_user": reviews_per_user.median(),
        "avg_reviews_per_book": reviews_per_book.mean(),
        "median_reviews_per_book": reviews_per_book.median(),
        "rating_dist": df["rating"].value_counts().sort_index(),
        "sparsity": 1 - len(df) / (df["user_id"].nunique() * df["parent_asin"].nunique()),
    })

    distributions[f"{label}/{Path(path).stem}"] = {
        "reviews_per_user": reviews_per_user,
        "reviews_per_book": reviews_per_book,
        "ratings": df["rating"],
    }
    print(f"Loaded {label}/{Path(path).stem}: {len(df):,} reviews")

stats_df = pd.DataFrame(summary)
print(f"\n{len(stats_df)} samples loaded.")
stats_df[["sample", "file", "n_reviews", "n_users", "n_books", "avg_rating"]].to_string(index=False)

#### Sparsity ####
stats_df["sparsity"] = 1 - stats_df["n_reviews"] / (stats_df["n_users"] * stats_df["n_books"])
stats_df["label"] = stats_df["sample"].astype(str) + "/" + stats_df["file"].astype(str)
sparsity_df = stats_df[["label", "n_reviews", "n_users", "n_books", "sparsity"]].copy()
sparsity_df["density (%)"] = (1 - sparsity_df["sparsity"]) * 100
sparsity_df["sparsity (%)"] = sparsity_df["sparsity"] * 100
sparsity_df.columns = ["Échantillon", "|R|", "|U|", "|I|", "ρ", "Densité (%)", "Sparsité (%)"]

print(sparsity_df.to_string(index=False, float_format="%.6f"))

fig, axes = plt.subplots(2, 3, figsize=(22, 14))
fig.suptitle("Comparaison des échantillons", fontsize=18, fontweight="bold")

labels = stats_df["sample"] + "/" + stats_df["file"]

x = range(len(labels))
colors = sns.color_palette("husl", len(labels))

# Row 1, Col 1: Total reviews
axes[0, 0].barh(labels, stats_df["n_reviews"], color=colors)
axes[0, 0].set_xlabel("Nombre de reviews")
axes[0, 0].set_title("Volume total de reviews")
for i, v in enumerate(stats_df["n_reviews"]):
    axes[0, 0].text(v + v * 0.01, i, f"{v:,}", va="center", fontsize=9)

# Row 1, Col 2: Total users
axes[0, 1].barh(labels, stats_df["n_users"], color=colors)
axes[0, 1].set_xlabel("Nombre d'utilisateurs")
axes[0, 1].set_title("Utilisateurs uniques")
for i, v in enumerate(stats_df["n_users"]):
    axes[0, 1].text(v + v * 0.01, i, f"{v:,}", va="center", fontsize=9)

# Row 1, Col 3: Total books
axes[0, 2].barh(labels, stats_df["n_books"], color=colors)
axes[0, 2].set_xlabel("Nombre de livres")
axes[0, 2].set_title("Livres uniques (ASINs)")
for i, v in enumerate(stats_df["n_books"]):
    axes[0, 2].text(v + v * 0.01, i, f"{v:,}", va="center", fontsize=9)

# Row 2, Col 1: Avg rating
axes[1, 0].barh(labels, stats_df["avg_rating"], color=colors)
axes[1, 0].set_xlabel("Note moyenne")
axes[1, 0].set_title("Note moyenne")
axes[1, 0].set_xlim(1, 5)

# Row 2, Col 2: Avg reviews per user
axes[1, 1].barh(labels, stats_df["avg_reviews_per_user"], color=colors)
axes[1, 1].set_xlabel("Reviews / utilisateur")
axes[1, 1].set_title("Moyenne reviews par utilisateur")

# Row 2, Col 3: Avg reviews per book
axes[1, 2].barh(labels, stats_df["avg_reviews_per_book"], color=colors)
axes[1, 2].set_xlabel("Reviews / livre")
axes[1, 2].set_title("Moyenne reviews par livre")

plt.tight_layout()
plt.show()

fig, axes = plt.subplots(3, 1, figsize=(14, 15))
fig.suptitle("Distributions comparées", fontsize=18, fontweight="bold", y=1.01)

sample_names = list(distributions.keys())
colors = sns.color_palette("husl", len(sample_names))

# Row 1: Rating distributions (grouped bar chart)
width = 0.8 / len(sample_names)
for i, (name, data) in enumerate(distributions.items()):
    counts = data["ratings"].value_counts().sort_index()
    counts_pct = counts / counts.sum() * 100  # normalize to % for fair comparison
    offset = (i - len(sample_names) / 2) * width + width / 2
    axes[0].bar(counts_pct.index + offset, counts_pct.values, width=width,
                label=name, color=colors[i], edgecolor="white", alpha=0.85)
axes[0].set_xlabel("Note (rating)")
axes[0].set_ylabel("Pourcentage des reviews (%)")
axes[0].set_title("Distribution des notes (normalisée)")
axes[0].set_xticks([1, 2, 3, 4, 5])
axes[0].legend(fontsize=8, loc="upper left")

# Row 2: Reviews per user (overlaid histograms)
for i, (name, data) in enumerate(distributions.items()):
    axes[1].hist(data["reviews_per_user"], bins=50, color=colors[i],
                 alpha=0.4, label=name, log=True, density=True)
axes[1].set_xlabel("Nombre de reviews par utilisateur")
axes[1].set_ylabel("Densité (log)")
axes[1].set_title("Distribution des reviews par utilisateur")
axes[1].legend(fontsize=8)

# Row 3: Reviews per book (overlaid histograms)
for i, (name, data) in enumerate(distributions.items()):
    axes[2].hist(data["reviews_per_book"], bins=50, color=colors[i],
                 alpha=0.4, label=name, log=True, density=True)
axes[2].set_xlabel("Nombre de reviews par livre")
axes[2].set_ylabel("Densité (log)")
axes[2].set_title("Distribution des reviews par livre")
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.show()


fig, axes = plt.subplots(1, 2, figsize=(18, 7))
colors = sns.color_palette("husl", len(stats_df))
labels = stats_df["label"]  # use the combined label

# Left: Sparsity bar chart
axes[0].barh(labels, stats_df["sparsity"] * 100, color=colors)
axes[0].set_xlabel("Sparsité ρ (%)")
axes[0].set_title("Sparsité de la matrice user-item par échantillon")
axes[0].set_xlim(
    max(stats_df["sparsity"].min() * 100 - 0.01, 0),
    100.0
)
for i, v in enumerate(stats_df["sparsity"]):
    axes[0].text(v * 100 + 0.001, i, f"{v*100:.4f}%", va="center", fontsize=9)

# Right: Density bar chart (log scale) — more informative than scatter
density = (1 - stats_df["sparsity"]) * 100  # in percent
axes[1].barh(labels, density, color=colors)
axes[1].set_xscale("log")
axes[1].set_xlabel("Densité (1 − ρ) en % (échelle log)")
axes[1].set_title("Densité de la matrice user-item")
for i, v in enumerate(density):
    axes[1].text(v * 1.05, i, f"{v:.4f}%", va="center", fontsize=9)

plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10, 8))

matrix_size = stats_df["n_users"] * stats_df["n_books"]
n_ratings = stats_df["n_reviews"]

ax.scatter(matrix_size, n_ratings, c=colors, s=120, edgecolors="black", zorder=5)

for i, row in stats_df.iterrows():
    ax.annotate(
        row["label"], (matrix_size[i], n_ratings[i]),
        textcoords="offset points", xytext=(8, 4),
        fontsize=7, ha="left"
    )

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("|U| × |I| (taille théorique)")
ax.set_ylabel("|R| (ratings observés)")
ax.set_title("|R| vs |U|×|I|")

# Reference diagonals computed from actual data range
x_min, x_max = matrix_size.min() * 0.5, matrix_size.max() * 2
x_range = np.logspace(np.log10(x_min), np.log10(x_max), 100)
for d in [1e-3, 1e-4, 1e-5]:
    ax.plot(x_range, x_range * d, "--", alpha=0.4, label=f"densité = {d:.0e}")

ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

comparison = stats_df[[
    "label", "n_reviews", "n_users", "n_books",
    "avg_rating", "avg_reviews_per_user", "avg_reviews_per_book"
]].copy()
comparison.columns = [
    "Échantillon", "Reviews", "Utilisateurs", "Livres",
    "Note moy.", "Rev/User", "Rev/Livre"
]
comparison["Note moy."] = comparison["Note moy."].round(2)
comparison["Rev/User"] = comparison["Rev/User"].round(1)
comparison["Rev/Livre"] = comparison["Rev/Livre"].round(1)

print(comparison.to_string(index=False))

### 2) Taux de Sparsité

Why this matters
For recommendation systems on Amazon Books data, you should expect extremely high sparsity (> 99.99%).

This is typical for real-world user-item matrices, and it's an important metric because:
- It justifies using sparse matrix representations (e.g., scipy.sparse)
- It highlights the cold-start problem — most user-item pairs have no observation
- It helps compare your sampling strategies: the "active users" sample should be denser than the random temporal sample, since you filtered for users with 20+ reviews

In [ ]:
sns.set_theme(style="whitegrid")

SAMPLE_PATHS = sorted(glob.glob("sample-*/*.parquet", recursive=False))

# Collect summary stats + per-sample distributions
summary = []
distributions = {}

for path in SAMPLE_PATHS:
    if os.path.getsize(path) < 1024:  # skip files smaller than 1KB
        print(f"  Skipping {path} (only {os.path.getsize(path)} bytes, likely corrupt)")
        continue
    df = pd.read_parquet(path)
    print(f"  {path}: {len(df)} rows, columns: {list(df.columns)}")    
    label = Path(path)  # e.g. "sample-cudf-claude"
    if len(df) == 0:
        continue

    reviews_per_user = df.groupby("user_id").size()
    reviews_per_book = df.groupby("parent_asin").size()

    summary.append({
        "sample": label,
        "file": Path(path).stem,
        "n_reviews": len(df),
        "n_users": df["user_id"].nunique(),
        "n_books": df["parent_asin"].nunique(),
        "avg_rating": df["rating"].mean(),
        "median_rating": df["rating"].median(),
        "avg_reviews_per_user": reviews_per_user.mean(),
        "median_reviews_per_user": reviews_per_user.median(),
        "avg_reviews_per_book": reviews_per_book.mean(),
        "median_reviews_per_book": reviews_per_book.median(),
        "rating_dist": df["rating"].value_counts().sort_index(),
        "sparsity": 1 - len(df) / (df["user_id"].nunique() * df["parent_asin"].nunique()),
    })

    distributions[f"{label}/{Path(path).stem}"] = {
        "reviews_per_user": reviews_per_user,
        "reviews_per_book": reviews_per_book,
        "ratings": df["rating"],
    }
    print(f"Loaded {label}/{Path(path).stem}: {len(df):,} reviews")

stats_df = pd.DataFrame(summary)
print(f"\n{len(stats_df)} samples loaded.")
stats_df["label"] = stats_df["sample"].astype(str)
stats_df[["sample", "file", "n_reviews", "n_users", "n_books", "avg_rating"]].to_string(index=False)
stats_df["sparsity"] = 1 - stats_df["n_reviews"] / (stats_df["n_users"] * stats_df["n_books"])

sparsity_df = stats_df[["label", "n_reviews", "n_users", "n_books", "sparsity"]].copy()
sparsity_df["density (%)"] = (1 - sparsity_df["sparsity"]) * 100
sparsity_df["sparsity (%)"] = sparsity_df["sparsity"] * 100
sparsity_df.columns = ["Échantillon", "|R|", "|U|", "|I|", "ρ", "Densité (%)", "Sparsité (%)"]

print(sparsity_df.to_string(index=False, float_format="%.6f"))

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
colors = sns.color_palette("husl", len(stats_df))
labels = stats_df["label"]  # use the combined label

# Left: Sparsity bar chart
axes[0].barh(labels, stats_df["sparsity"] * 100, color=colors)
axes[0].set_xlabel("Sparsité ρ (%)")
axes[0].set_title("Sparsité de la matrice user-item par échantillon")
axes[0].set_xlim(
    max(stats_df["sparsity"].min() * 100 - 0.01, 0),
    100.0
)
for i, v in enumerate(stats_df["sparsity"]):
    axes[0].text(v * 100 + 0.001, i, f"{v*100:.4f}%", va="center", fontsize=9)

# Right: Density bar chart (log scale) — more informative than scatter
density = (1 - stats_df["sparsity"]) * 100  # in percent
axes[1].barh(labels, density, color=colors)
axes[1].set_xscale("log")
axes[1].set_xlabel("Densité (1 − ρ) en % (échelle log)")
axes[1].set_title("Densité de la matrice user-item")
for i, v in enumerate(density):
    axes[1].text(v * 1.05, i, f"{v:.4f}%", va="center", fontsize=9)

plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10, 8))

matrix_size = stats_df["n_users"] * stats_df["n_books"]
n_ratings = stats_df["n_reviews"]

ax.scatter(matrix_size, n_ratings, c=colors, s=120, edgecolors="black", zorder=5)

for i, row in stats_df.iterrows():
    ax.annotate(
        row["label"], (matrix_size[i], n_ratings[i]),
        textcoords="offset points", xytext=(8, 4),
        fontsize=7, ha="left"
    )

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("|U| × |I| (taille théorique)")
ax.set_ylabel("|R| (ratings observés)")
ax.set_title("|R| vs |U|×|I|")

# Reference diagonals computed from actual data range
x_min, x_max = matrix_size.min() * 0.5, matrix_size.max() * 2
x_range = np.logspace(np.log10(x_min), np.log10(x_max), 100)
for d in [1e-3, 1e-4, 1e-5]:
    ax.plot(x_range, x_range * d, "--", alpha=0.4, label=f"densité = {d:.0e}")

ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

### 3) Analyse de la distribution des données

In [ ]:
SAMPLE_PATHS = sorted(glob.glob("sample-*/*.parquet", recursive=False))

for path in SAMPLE_PATHS:
    if os.path.getsize(path) < 1024:  # skip files smaller than 1KB
        print(f"  Skipping {path} (only {os.path.getsize(path)} bytes, likely corrupt)")
        continue
    df = pd.read_parquet(path)
    if len(df) == 0:
        continue

    print(f"\n{'=' * 60}")
    print(f"  Distribution Analysis — {path}")
    print(f"{'=' * 60}")

    fig, axes = plt.subplots(2, 2, figsize=(18, 12))

    # ── 1. Popularité des livres (longue traîne) ──────────────────
    reviews_per_book = (
        df.groupby("parent_asin").size()
        .sort_values(ascending=False)
        .reset_index(drop=True)
    )
    rank = np.arange(1, len(reviews_per_book) + 1)
    axes[0, 0].loglog(rank, reviews_per_book.values, ".", markersize=1.5, alpha=0.4, color="steelblue")
    axes[0, 0].set_xlabel("Rang du livre (log)")
    axes[0, 0].set_ylabel("Nombre de reviews (log)")
    axes[0, 0].set_title("Popularité des livres — Longue traîne")
    axes[0, 0].axhline(reviews_per_book.mean(), color="red", ls="--", lw=0.8,
                        label=f"Moyenne: {reviews_per_book.mean():.1f}")
    axes[0, 0].legend()

    # ── 2. Distribution temporelle des évaluations ────────────────
    df["date"] = pd.to_datetime(df["timestamp"], unit="ms")
    reviews_by_month = df.set_index("date").resample("ME").size()
    axes[0, 1].plot(reviews_by_month.index, reviews_by_month.values,
                    color="teal", linewidth=1)
    axes[0, 1].fill_between(reviews_by_month.index, reviews_by_month.values,
                            alpha=0.15, color="teal")
    axes[0, 1].set_xlabel("Date")
    axes[0, 1].set_ylabel("Nombre de reviews / mois")
    axes[0, 1].set_title("Distribution temporelle des évaluations")
    axes[0, 1].tick_params(axis="x", rotation=45)

    # ── 3. Distribution des votes d'utilité ───────────────────────
    helpful = df["helpful_vote"]
    n_zero = (helpful == 0).sum()
    n_nonzero = (helpful > 0).sum()
    helpful_nonzero = helpful[helpful > 0]

    axes[1, 0].hist(helpful_nonzero, bins=100, color="orchid", edgecolor="white", log=True)
    axes[1, 0].set_xlabel("Nombre de votes utiles")
    axes[1, 0].set_ylabel("Nombre de reviews (log)")
    axes[1, 0].set_title(f"Votes d'utilité — {n_nonzero:,} non-nuls / {len(helpful):,} total "
                         f"({n_nonzero / len(helpful) * 100:.1f}%)")
    axes[1, 0].axvline(helpful_nonzero.median(), color="red", ls="--", lw=0.8,
                       label=f"Médiane (non-nuls): {helpful_nonzero.median():.0f}")
    axes[1, 0].legend()

    # ── 4. Proportion d'achats vérifiés ───────────────────────────
    vp_counts = df["verified_purchase"].value_counts()
    labels = [f"Vérifié\n({vp_counts.get(True, 0):,})",
              f"Non vérifié\n({vp_counts.get(False, 0):,})"]
    colors = ["#2ecc71", "#e74c3c"]
    axes[1, 1].pie(vp_counts.values, labels=labels, colors=colors,
                   autopct="%1.1f%%", startangle=90, textprops={"fontsize": 11})
    axes[1, 1].set_title("Proportion d'achats vérifiés")

    plt.suptitle(path, fontsize=13, fontweight="bold", y=1.01)
    plt.tight_layout()
    plt.show()

    df.drop(columns=["date"], inplace=True)
    del df
    gc.collect()

### 4) Visualisation

#### A) Premier set de visualisations

###### Panel 1 (2x2 grid):
- Rating by Verified Purchase (violin plot) — Do verified buyers rate differently? This matters because unverified reviews may introduce bias.
- Review Text Length Distribution (histogram) — Distribution of len(text). Important if you plan NLP-based features. Long-tailed distributions may need log-transform.
- Average Rating Over Time (line chart) — Monthly mean rating to detect rating inflation/deflation trends. Complements your existing volume over time chart.
- Review Length vs. Rating (box plot) — Do users write more for low or high ratings? Useful to understand if text length is a predictive signal.
###### Panel 2 (1x2 grid):
- Helpful Votes by Rating Level (box plot, log scale) — Are certain ratings more likely to receive helpful votes? (e.g., critical 1-star reviews tend to get more votes)
- User Activity Lorenz Curve — Cumulative share of reviews vs. cumulative share of users (sorted by activity). Visualizes inequality/concentration — a Gini coefficient for your user base. Directly relevant to cold-start analysis.

In [ ]:
sns.set_theme(style="whitegrid")

SAMPLE_PATHS = sorted(glob.glob("sample-*/*.parquet", recursive=False))

for path in SAMPLE_PATHS:
    if os.path.getsize(path) < 1024:
        print(f"  Skipping {path} (only {os.path.getsize(path)} bytes, likely corrupt)")
        continue
    df = pd.read_parquet(path)
    if len(df) == 0:
        continue

    print(f"\n{'=' * 60}")
    print(f"  Data Understanding — {path}")
    print(f"{'=' * 60}")

    df["text_len"] = df["text"].fillna("").str.len()
    df["date"] = pd.to_datetime(df["timestamp"], unit="ms")

    # ── Panel 1: 2×2 ──────────────────────────────────────────────
    fig, axes = plt.subplots(2, 2, figsize=(18, 12))

    # 1. Rating by Verified Purchase (violin)
    sns.violinplot(
        data=df, x="rating", y="verified_purchase",
        orient="h", ax=axes[0, 0],
        palette={"True": "#2ecc71", "False": "#e74c3c"},
        inner="quartile", cut=0,
    )
    vp_mean = df.groupby("verified_purchase")["rating"].mean()
    axes[0, 0].set_title("Distribution des notes — Vérifié vs Non vérifié")
    axes[0, 0].set_xlabel("Note (rating)")
    axes[0, 0].set_ylabel("")
    axes[0, 0].set_yticklabels(
        [f"Non vérifié (μ={vp_mean.get(False, 0):.2f})",
         f"Vérifié (μ={vp_mean.get(True, 0):.2f})"]
    )

    # 2. Review text length distribution
    axes[0, 1].hist(
        df["text_len"].clip(upper=df["text_len"].quantile(0.99)),
        bins=80, color="mediumpurple", edgecolor="white", log=True,
    )
    axes[0, 1].axvline(
        df["text_len"].median(), color="red", ls="--", lw=1,
        label=f"Médiane: {df['text_len'].median():.0f} car.",
    )
    axes[0, 1].set_xlabel("Longueur du texte (caractères)")
    axes[0, 1].set_ylabel("Nombre de reviews (log)")
    axes[0, 1].set_title("Distribution de la longueur des reviews")
    axes[0, 1].legend()

    # 3. Average rating over time
    monthly_mean = df.set_index("date")["rating"].resample("ME").mean()
    axes[1, 0].plot(monthly_mean.index, monthly_mean.values, color="teal", lw=1.2)
    axes[1, 0].fill_between(
        monthly_mean.index, monthly_mean.values, alpha=0.12, color="teal"
    )
    axes[1, 0].axhline(
        df["rating"].mean(), color="red", ls="--", lw=0.8,
        label=f"Globale: {df['rating'].mean():.2f}",
    )
    axes[1, 0].set_xlabel("Date")
    axes[1, 0].set_ylabel("Note moyenne")
    axes[1, 0].set_title("Évolution de la note moyenne dans le temps")
    axes[1, 0].tick_params(axis="x", rotation=45)
    axes[1, 0].legend()

    # 4. Review length vs rating (box plot)
    sns.boxplot(
        data=df, x="rating", y="text_len",
        ax=axes[1, 1], color="sandybrown",
        showfliers=False,
    )
    axes[1, 1].set_xlabel("Note (rating)")
    axes[1, 1].set_ylabel("Longueur du texte (car.)")
    axes[1, 1].set_title("Longueur de la review selon la note")

    plt.suptitle(path, fontsize=13, fontweight="bold", y=1.01)
    plt.tight_layout()
    plt.show()

    # ── Panel 2: 1×2 ──────────────────────────────────────────────
    fig2, axes2 = plt.subplots(1, 2, figsize=(18, 6))

    # 5. Helpful votes by rating level
    df_with_votes = df[df["helpful_vote"] > 0]
    if len(df_with_votes) > 0:
        sns.boxplot(
            data=df_with_votes, x="rating", y="helpful_vote",
            ax=axes2[0], color="orchid", showfliers=False,
        )
        axes2[0].set_yscale("log")
    axes2[0].set_xlabel("Note (rating)")
    axes2[0].set_ylabel("Votes utiles (log)")
    axes2[0].set_title("Votes d'utilité par niveau de note (reviews avec ≥1 vote)")

    # 6. User activity Lorenz curve
    reviews_per_user = df.groupby("user_id").size().sort_values().values
    cum_users = np.arange(1, len(reviews_per_user) + 1) / len(reviews_per_user)
    cum_reviews = np.cumsum(reviews_per_user) / reviews_per_user.sum()

    axes2[1].plot(cum_users, cum_reviews, color="steelblue", lw=1.5, label="Lorenz")
    axes2[1].plot([0, 1], [0, 1], "k--", lw=0.8, alpha=0.5, label="Égalité parfaite")
    axes2[1].fill_between(cum_users, cum_reviews, cum_users, alpha=0.12, color="steelblue")

    gini = 1 - 2 * np.trapezoid(cum_reviews, cum_users)
    axes2[1].set_xlabel("Part cumulée des utilisateurs")
    axes2[1].set_ylabel("Part cumulée des reviews")
    axes2[1].set_title(f"Courbe de Lorenz — Concentration utilisateurs (Gini = {gini:.3f})")
    axes2[1].legend()

    plt.suptitle(path, fontsize=13, fontweight="bold", y=1.01)
    plt.tight_layout()
    plt.show()

    del df
    gc.collect()

#### B) Second set de visualisations

*** Utilisation de PyTorch pour les calculs à la place de cuML ***


##### Avec T-SNE

In [ ]:
sns.set_theme(style="whitegrid")

SAMPLE_PATHS = sorted(glob.glob("sample-*/*.parquet", recursive=False))

# ── Backend detection ─────────────────────────────────────────
USE_GPU = torch.cuda.is_available()
if USE_GPU:
    GPU_DEV = torch.device("cuda")
    print(f"Backend: PyTorch GPU ({torch.cuda.get_device_name()})")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    GPU_DEV = torch.device("cpu")
    print("Backend: PyTorch CPU")

# ── Tunables ──────────────────────────────────────────────────
N_SVD_COMPONENTS   = 50
N_USERS_TSNE       = 50_000 if USE_GPU else 5_000
N_USERS_HEATMAP    = 1_000    if USE_GPU else 150
N_USERS_SILHOUETTE = 50_000 if USE_GPU else 8_000
N_SILHOUETTE_DETAIL = 20_000   # subsample for the detailed diagram only
K_RANGE            = range(2, 16) if USE_GPU else range(2, 11)

TARGET_SAMPLES = [
    "sample-cudf-claude/sample_gpu_active_users.parquet",
    "sample-cudf-tempor/sample_gpu_temporal.parquet",
]


# ── Helper: GPU KMeans via PyTorch ────────────────────────────
def torch_kmeans(X_np, n_clusters, max_iter=200, seed=42):
    torch.manual_seed(seed)
    X = torch.tensor(X_np, dtype=torch.float32, device=GPU_DEV)
    n = X.shape[0]
    idx = torch.randperm(n, device=GPU_DEV)[:n_clusters]
    centroids = X[idx].clone()

    for _ in range(max_iter):
        dists = torch.cdist(X, centroids)
        labels = dists.argmin(dim=1)
        new_centroids = torch.zeros_like(centroids)
        for j in range(n_clusters):
            mask = labels == j
            if mask.any():
                new_centroids[j] = X[mask].mean(dim=0)
            else:
                new_centroids[j] = X[torch.randint(n, (1,), device=GPU_DEV)].squeeze()
        if torch.allclose(centroids, new_centroids, atol=1e-6):
            break
        centroids = new_centroids

    return labels.cpu().numpy()


# ── Helper: sparse scipy → sparse torch on GPU ───────────────
def scipy_sparse_to_torch(sp_matrix, device):
    coo = coo_matrix(sp_matrix)
    indices = torch.tensor(
        np.array([coo.row, coo.col]), dtype=torch.long, device=device
    )
    values = torch.tensor(coo.data, dtype=torch.float32, device=device)
    return torch.sparse_coo_tensor(indices, values, size=coo.shape)


# ── Main loop ─────────────────────────────────────────────────
for path in SAMPLE_PATHS:
    # if TARGET_SAMPLES and path not in TARGET_SAMPLES:
    #     continue
    if os.path.getsize(path) < 1024:
        continue
    df = pd.read_parquet(path)
    if len(df) == 0:
        continue

    print(f"\n{'=' * 60}")
    print(f"  Advanced Analysis — {path}")
    print(f"{'=' * 60}")
    t0 = time.time()

    # ━━━ Sparse user-item matrix ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    user_enc = LabelEncoder()
    item_enc = LabelEncoder()
    uid = user_enc.fit_transform(df["user_id"])
    iid = item_enc.fit_transform(df["parent_asin"])

    R = csr_matrix(
        (df["rating"].values.astype(np.float32), (uid, iid)),
        shape=(len(user_enc.classes_), len(item_enc.classes_)),
    )
    n_users, n_items = R.shape
    print(f"  Matrix: {n_users:,} × {n_items:,}  "
          f"(nnz={R.nnz:,}, density={R.nnz / (n_users * n_items) * 100:.4f}%)")

    # ━━━ TruncatedSVD via torch.svd_lowrank ━━━━━━━━━━━━━━━━━━
    n_comp = min(N_SVD_COMPONENTS, n_users - 1, n_items - 1)

    if USE_GPU:
        R_torch = scipy_sparse_to_torch(R, GPU_DEV)
        U_t, S_t, V_t = torch.svd_lowrank(R_torch, q=n_comp, niter=5)

        U_svd = (U_t * S_t).cpu().numpy()

        total_var = float((R.data ** 2).sum())
        explained_var = float((S_t ** 2).sum())
        var_explained = explained_var / total_var
        var_ratios = (S_t ** 2 / total_var).cpu().numpy()

        del R_torch
        torch.cuda.empty_cache()
    else:
        from sklearn.decomposition import TruncatedSVD
        svd_cpu = TruncatedSVD(n_components=n_comp, random_state=42)
        U_svd = svd_cpu.fit_transform(R)
        var_explained = svd_cpu.explained_variance_ratio_.sum()
        var_ratios = svd_cpu.explained_variance_ratio_

    print(f"  SVD: {n_comp} comp, var={var_explained:.2%} ({time.time()-t0:.1f}s)")

    rng = np.random.default_rng(42)

    interactions = np.array((R > 0).sum(axis=1)).flatten()
    rating_sums  = np.array(R.sum(axis=1)).flatten()
    user_mean_rating = np.divide(
        rating_sums, interactions,
        out=np.zeros_like(rating_sums), where=interactions > 0,
    )

    # ━━━ VIZ 1 — 2D Projection (SVD + t-SNE) ━━━━━━━━━━━━━━━━
    n_tsne = min(N_USERS_TSNE, n_users)
    idx_tsne = rng.choice(n_users, size=n_tsne, replace=False)
    U_sub = U_svd[idx_tsne]
    colors = user_mean_rating[idx_tsne]

    fig1, axes1 = plt.subplots(1, 2, figsize=(18, 7))

    sc1 = axes1[0].scatter(
        U_sub[:, 0], U_sub[:, 1],
        c=colors, cmap="RdYlGn", s=2, alpha=0.3,
    )
    axes1[0].set_xlabel(f"SVD 1 ({var_ratios[0]:.1%})")
    axes1[0].set_ylabel(f"SVD 2 ({var_ratios[1]:.1%})")
    axes1[0].set_title(f"Projection SVD 2D — {n_tsne:,} utilisateurs")
    plt.colorbar(sc1, ax=axes1[0], label="Note moyenne")

    print(f"  t-SNE on {n_tsne:,} users (CPU sklearn)...")
    tsne = TSNE(
        n_components=2, perplexity=30, random_state=42,
        max_iter=1000, init="pca", learning_rate="auto",
    )
    U_tsne = tsne.fit_transform(U_sub)
    print(f"  t-SNE done ({time.time()-t0:.1f}s)")

    sc2 = axes1[1].scatter(
        U_tsne[:, 0], U_tsne[:, 1],
        c=colors, cmap="RdYlGn", s=2, alpha=0.3,
    )
    axes1[1].set_xlabel("t-SNE 1")
    axes1[1].set_ylabel("t-SNE 2")
    axes1[1].set_title(f"Projection t-SNE — {n_tsne:,} utilisateurs")
    plt.colorbar(sc2, ax=axes1[1], label="Note moyenne")

    plt.suptitle(f"Projection 2D — {path}", fontsize=13, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.show()

    # ━━━ VIZ 2 — Similarity Heatmap ━━━━━━━━━━━━━━━━━━━━━━━━━
    n_hm = min(N_USERS_HEATMAP, n_users)
    idx_hm = rng.choice(n_users, size=n_hm, replace=False)

    if USE_GPU:
        hm_gpu = torch.tensor(U_svd[idx_hm], dtype=torch.float32, device=GPU_DEV)
        hm_norm = hm_gpu / hm_gpu.norm(dim=1, keepdim=True).clamp(min=1e-8)
        sim = (hm_norm @ hm_norm.T).cpu().numpy()
        del hm_gpu, hm_norm
        torch.cuda.empty_cache()
    else:
        sim = cosine_similarity(U_svd[idx_hm])

    g = sns.clustermap(
        sim, cmap="YlOrRd", figsize=(11, 10),
        xticklabels=False, yticklabels=False,
        method="ward", linewidths=0,
    )
    g.fig.suptitle(
        f"Similarité cosinus ({n_hm} utilisateurs) — {path}",
        fontsize=13, fontweight="bold", y=1.02,
    )
    plt.show()
    print(f"  Heatmap done ({time.time()-t0:.1f}s)")

    # ━━━ VIZ 3 — Power-law degree distribution ━━━━━━━━━━━━━━━
    deg_items = np.array((R > 0).sum(axis=0)).flatten()
    deg_users = np.array((R > 0).sum(axis=1)).flatten()

    fig3, axes3 = plt.subplots(1, 2, figsize=(18, 6))
    for ax, deg, label, color in [
        (axes3[0], deg_items[deg_items > 0], "Livres", "steelblue"),
        (axes3[1], deg_users[deg_users > 0], "Utilisateurs", "darkorange"),
    ]:
        vals, counts = np.unique(deg, return_counts=True)
        ax.loglog(vals, counts, ".", markersize=4, alpha=0.5, color=color)
        slope, intercept = np.polyfit(np.log10(vals), np.log10(counts), 1)
        x_fit = np.logspace(np.log10(vals.min()), np.log10(vals.max()), 100)
        ax.loglog(x_fit, 10**intercept * x_fit**slope,
                  "r--", lw=1, label=f"α = {slope:.2f}")
        ax.set_xlabel("Degré (nombre de reviews)")
        ax.set_ylabel("Fréquence")
        ax.set_title(f"Distribution des degrés — {label}")
        ax.legend()

    plt.suptitle(f"Loi de puissance — {path}", fontsize=13, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.show()

    # ━━━ VIZ 4 — Silhouette analysis ━━━━━━━━━━━━━━━━━━━━━━━━
    n_sil = min(N_USERS_SILHOUETTE, n_users)
    idx_sil = rng.choice(n_users, size=n_sil, replace=False)
    U_sil = U_svd[idx_sil]

    print(f"  Silhouette k={K_RANGE.start}..{K_RANGE.stop-1} on {n_sil:,} users...")
    sil_avgs = []
    for k in K_RANGE:
        if USE_GPU:
            labels = torch_kmeans(U_sil, n_clusters=k, seed=42)
        else:
            from sklearn.cluster import MiniBatchKMeans
            km = MiniBatchKMeans(n_clusters=k, random_state=42, batch_size=2048)
            labels = km.fit_predict(U_sil)
        s = silhouette_score(U_sil, labels, sample_size=min(10_000, n_sil))
        sil_avgs.append(s)
        print(f"    k={k:2d}  silhouette={s:.4f}")

    best_k = list(K_RANGE)[int(np.argmax(sil_avgs))]

    fig4, axes4 = plt.subplots(1, 2, figsize=(18, 7))
    axes4[0].plot(list(K_RANGE), sil_avgs, "o-", color="steelblue", lw=1.5)
    axes4[0].axvline(best_k, color="red", ls="--", lw=0.8, label=f"Meilleur k = {best_k}")
    axes4[0].set_xlabel("Nombre de clusters (k)")
    axes4[0].set_ylabel("Score silhouette moyen")
    axes4[0].set_title("Silhouette moyenne vs k")
    axes4[0].legend()

    if USE_GPU:
        labels_best = torch_kmeans(U_sil, n_clusters=best_k, seed=42)
    else:
        km_best = MiniBatchKMeans(n_clusters=best_k, random_state=42, batch_size=2048)
        labels_best = km_best.fit_predict(U_sil)

    sample_sil = silhouette_samples(U_sil, labels_best)

    y_lower = 0
    for i in range(best_k):
        cluster_vals = np.sort(sample_sil[labels_best == i])
        y_upper = y_lower + len(cluster_vals)
        axes4[1].fill_betweenx(
            np.arange(y_lower, y_upper), 0, cluster_vals,
            facecolor=cm.tab10(i / best_k), alpha=0.7, label=f"Cluster {i}",
        )
        y_lower = y_upper + 10

    axes4[1].axvline(sil_avgs[best_k - K_RANGE.start], color="red", ls="--", lw=0.8)
    axes4[1].set_xlabel("Coefficient de silhouette")
    axes4[1].set_ylabel("Utilisateurs (triés par cluster)")
    axes4[1].set_title(f"Diagramme de silhouette — k={best_k}")
    axes4[1].legend(loc="lower right", fontsize=8)

    plt.suptitle(f"Clustering — {path}", fontsize=13, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.show()
    print(f"  Clustering done ({time.time()-t0:.1f}s)")

    # ━━━ VIZ 5 — Rating ↔ Helpful Votes ━━━━━━━━━━━━━━━━━━━━━
    fig5, axes5 = plt.subplots(1, 2, figsize=(18, 6))

    agg = (df.groupby("rating")["helpful_vote"]
             .agg(["mean", "median", "std"]).reset_index())

    axes5[0].bar(agg["rating"], agg["mean"], color="orchid", edgecolor="white")
    axes5[0].errorbar(agg["rating"], agg["mean"], yerr=agg["std"],
                      fmt="none", color="black", capsize=4)
    for _, row in agg.iterrows():
        axes5[0].text(row["rating"], row["mean"] + row["std"] + 0.1,
                      f'μ={row["mean"]:.1f}', ha="center", fontsize=9)
    axes5[0].set_xlabel("Note (rating)")
    axes5[0].set_ylabel("Votes utiles (moyenne ± σ)")
    axes5[0].set_title("Votes d'utilité moyens par note")
    axes5[0].set_xticks([1, 2, 3, 4, 5])

    df_nz = df[df["helpful_vote"] > 0]
    if len(df_nz) > 0:
        hb = axes5[1].hexbin(
            df_nz["rating"], np.log1p(df_nz["helpful_vote"]),
            gridsize=25, cmap="YlOrRd", mincnt=1,
        )
        plt.colorbar(hb, ax=axes5[1], label="Nb de reviews")
    axes5[1].set_xlabel("Note (rating)")
    axes5[1].set_ylabel("log(1 + helpful_vote)")
    axes5[1].set_title(f"Densité: Note vs Votes ({len(df_nz):,} reviews avec votes)")

    plt.suptitle(f"Corrélation Note ↔ Votes — {path}",
                 fontsize=13, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.show()

    print(f"\n  Total: {time.time()-t0:.1f}s")
    del df, R, U_svd, U_sil
    torch.cuda.empty_cache()
    gc.collect()

##### Avec UMAP

In [ ]:
sns.set_theme(style="whitegrid")

SAMPLE_PATHS = sorted(glob.glob("sample-*/*.parquet", recursive=False))

# ── Backend detection ─────────────────────────────────────────
USE_GPU = torch.cuda.is_available()
if USE_GPU:
    GPU_DEV = torch.device("cuda")
    print(f"Backend: PyTorch GPU ({torch.cuda.get_device_name()})")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    GPU_DEV = torch.device("cpu")
    print("Backend: PyTorch CPU")

# ── Tunables ──────────────────────────────────────────────────
N_SVD_COMPONENTS    = 50
N_USERS_PROJ        = 50_000 if USE_GPU else 5_000
N_USERS_HEATMAP     = 1_000  if USE_GPU else 150
N_USERS_SILHOUETTE  = 50_000 if USE_GPU else 8_000
N_SILHOUETTE_DETAIL = 20_000
K_RANGE             = range(2, 16) if USE_GPU else range(2, 11)

TARGET_SAMPLES = [
    "sample-cudf-claude/sample_gpu_active_users.parquet",
    "sample-cudf-tempor/sample_gpu_temporal.parquet",
]

# Try to import UMAP (much faster than t-SNE); fall back to t-SNE
try:
    from umap import UMAP
    HAS_UMAP = True
    print("  Projection: UMAP (fast)")
except ImportError:
    HAS_UMAP = False
    print("  Projection: t-SNE (slower, `pip install umap-learn` for speed)")


def torch_kmeans(X_np, n_clusters, max_iter=200, seed=42):
    """KMeans on GPU via torch.cdist for distance computation."""
    torch.manual_seed(seed)
    X = torch.tensor(X_np, dtype=torch.float32, device=GPU_DEV)
    n = X.shape[0]
    idx = torch.randperm(n, device=GPU_DEV)[:n_clusters]
    centroids = X[idx].clone()

    for _ in range(max_iter):
        dists = torch.cdist(X, centroids)
        labels = dists.argmin(dim=1)
        new_centroids = torch.zeros_like(centroids)
        for j in range(n_clusters):
            mask = labels == j
            if mask.any():
                new_centroids[j] = X[mask].mean(dim=0)
            else:
                new_centroids[j] = X[torch.randint(n, (1,), device=GPU_DEV)].squeeze()
        if torch.allclose(centroids, new_centroids, atol=1e-6):
            break
        centroids = new_centroids

    return labels.cpu().numpy()


def scipy_sparse_to_torch(sp_matrix, device):
    """Zero-copy conversion: scipy COO sparse -> torch sparse tensor on GPU."""
    coo = coo_matrix(sp_matrix)
    indices = torch.tensor(
        np.array([coo.row, coo.col]), dtype=torch.long, device=device
    )
    values = torch.tensor(coo.data, dtype=torch.float32, device=device)
    return torch.sparse_coo_tensor(indices, values, size=coo.shape)


for path in SAMPLE_PATHS:
    # if TARGET_SAMPLES and path not in TARGET_SAMPLES:
    #     continue
    if os.path.getsize(path) < 1024:
        continue
    df = pd.read_parquet(path)
    if len(df) == 0:
        continue

    print(f"\n{'=' * 70}")
    print(f"  ADVANCED ANALYSIS — {path}")
    print(f"{'=' * 70}")
    t0 = time.time()

    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    #  STEP 0 — Build sparse user-item rating matrix
    #  Rows = users, Columns = books, Values = ratings (1-5).
    #  Most entries are zero (unobserved), hence sparse representation.
    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    user_enc = LabelEncoder()
    item_enc = LabelEncoder()
    uid = user_enc.fit_transform(df["user_id"])
    iid = item_enc.fit_transform(df["parent_asin"])

    R = csr_matrix(
        (df["rating"].values.astype(np.float32), (uid, iid)),
        shape=(len(user_enc.classes_), len(item_enc.classes_)),
    )
    n_users, n_items = R.shape

    print(f"\n  [MATRIX] User-item rating matrix built:")
    print(f"    {n_users:,} users × {n_items:,} books")
    print(f"    {R.nnz:,} observed ratings (non-zero entries)")
    print(f"    Density: {R.nnz / (n_users * n_items) * 100:.4f}%")
    print(f"    → {100 - R.nnz / (n_users * n_items) * 100:.4f}% of user-book "
          f"pairs have NO rating (cold-start territory)")

    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    #  STEP 1 — Dimensionality reduction: TruncatedSVD
    #  Reduce each user from a ~1M-dimensional sparse vector to a
    #  dense 50-dimensional embedding. This captures the dominant
    #  latent factors (reading preferences).
    #  Uses torch.svd_lowrank on GPU for speed on sparse tensors.
    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    n_comp = min(N_SVD_COMPONENTS, n_users - 1, n_items - 1)

    if USE_GPU:
        R_torch = scipy_sparse_to_torch(R, GPU_DEV)
        U_t, S_t, V_t = torch.svd_lowrank(R_torch, q=n_comp, niter=5)
        U_svd = (U_t * S_t).cpu().numpy()
        total_var = float((R.data ** 2).sum())
        explained_var = float((S_t ** 2).sum())
        var_explained = explained_var / total_var
        var_ratios = (S_t ** 2).cpu().numpy() / total_var
        del R_torch, U_t, S_t, V_t
        torch.cuda.empty_cache()
    else:
        from sklearn.decomposition import TruncatedSVD
        svd_cpu = TruncatedSVD(n_components=n_comp, random_state=42)
        U_svd = svd_cpu.fit_transform(R)
        var_explained = svd_cpu.explained_variance_ratio_.sum()
        var_ratios = svd_cpu.explained_variance_ratio_

    print(f"\n  [SVD] Dimensionality reduction (GPU: {USE_GPU}):")
    print(f"    {n_items:,}-dim sparse → {n_comp}-dim dense per user")
    print(f"    Variance explained: {var_explained:.2%}")
    print(f"    → A low % is normal for extremely sparse data. The top-{n_comp}")
    print(f"      latent factors still capture the main structure.")
    print(f"    Time: {time.time()-t0:.1f}s")

    rng = np.random.default_rng(42)

    # Per-user mean rating (for coloring projections)
    interactions = np.array((R > 0).sum(axis=1)).flatten()
    rating_sums  = np.array(R.sum(axis=1)).flatten()
    user_mean_rating = np.divide(
        rating_sums, interactions,
        out=np.zeros_like(rating_sums), where=interactions > 0,
    )

    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    #  VIZ 1 — 2D User Projection (SVD first 2 dims + UMAP/t-SNE)
    #
    #  PURPOSE: Identify visual clusters of readers. If users group
    #  into "bubbles", it validates that collaborative filtering or
    #  clustering-based recommendations can find meaningful segments.
    #  Left panel: raw SVD components (linear projection).
    #  Right panel: UMAP or t-SNE (non-linear, preserves local
    #  neighborhoods — better at revealing cluster structure).
    #  Color = mean rating: green=generous raters, red=harsh raters.
    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    n_proj = min(N_USERS_PROJ, n_users)
    idx_proj = rng.choice(n_users, size=n_proj, replace=False)
    U_sub = U_svd[idx_proj]
    colors = user_mean_rating[idx_proj]

    print(f"\n  [VIZ 1] 2D Projection of {n_proj:,} users")
    print(f"    Left: SVD (linear) — first 2 of {n_comp} latent dimensions")
    print(f"    Right: {'UMAP' if HAS_UMAP else 't-SNE'} (non-linear) — "
          f"reveals local cluster structure")
    print(f"    Color encodes each user's mean rating (red=low, green=high)")
    print(f"    INTERPRETATION: Distinct clusters → users have different taste")
    print(f"    profiles → collaborative filtering is viable.")

    fig1, axes1 = plt.subplots(1, 2, figsize=(18, 7))

    sc1 = axes1[0].scatter(
        U_sub[:, 0], U_sub[:, 1],
        c=colors, cmap="RdYlGn", s=2, alpha=0.3,
    )
    axes1[0].set_xlabel(f"SVD 1 ({var_ratios[0]:.1%} var)")
    axes1[0].set_ylabel(f"SVD 2 ({var_ratios[1]:.1%} var)")
    axes1[0].set_title(f"Projection SVD 2D — {n_proj:,} utilisateurs")
    plt.colorbar(sc1, ax=axes1[0], label="Note moyenne")

    t_proj = time.time()
    if HAS_UMAP:
        reducer = UMAP(n_components=2, n_neighbors=30, min_dist=0.3,
                       random_state=42, n_jobs=-1)
        U_2d = reducer.fit_transform(U_sub)
        proj_label = "UMAP"
    else:
        tsne = TSNE(n_components=2, perplexity=30, random_state=42,
                    max_iter=1000, init="pca", learning_rate="auto")
        U_2d = tsne.fit_transform(U_sub)
        proj_label = "t-SNE"
    print(f"    {proj_label} computed in {time.time()-t_proj:.1f}s (CPU)")

    sc2 = axes1[1].scatter(
        U_2d[:, 0], U_2d[:, 1],
        c=colors, cmap="RdYlGn", s=2, alpha=0.3,
    )
    axes1[1].set_xlabel(f"{proj_label} 1")
    axes1[1].set_ylabel(f"{proj_label} 2")
    axes1[1].set_title(f"Projection {proj_label} — {n_proj:,} utilisateurs")
    plt.colorbar(sc2, ax=axes1[1], label="Note moyenne")

    plt.suptitle(f"Projection 2D des utilisateurs — {path}",
                 fontsize=13, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.show()

    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    #  VIZ 2 — User Similarity Heatmap (cosine similarity)
    #
    #  PURPOSE: Detect neighborhood structure. Rows and columns are
    #  users, reordered by hierarchical clustering (Ward linkage).
    #  Dense yellow/red blocks on the diagonal = groups of users
    #  with highly correlated tastes → "neighborhoods" that k-NN
    #  recommendation can exploit.
    #  Cosine similarity computed on GPU via normalized matmul.
    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    n_hm = min(N_USERS_HEATMAP, n_users)
    idx_hm = rng.choice(n_users, size=n_hm, replace=False)

    print(f"\n  [VIZ 2] Cosine similarity heatmap — {n_hm} users")
    print(f"    Computed on {n_comp}-dim SVD embeddings (GPU: {USE_GPU})")
    print(f"    Ward hierarchical clustering reorders rows/columns")
    print(f"    INTERPRETATION: Bright blocks along diagonal = user")
    print(f"    neighborhoods with similar reading preferences.")
    print(f"    More/larger blocks → richer structure for k-NN recommenders.")

    if USE_GPU:
        hm_gpu = torch.tensor(U_svd[idx_hm], dtype=torch.float32, device=GPU_DEV)
        hm_norm = hm_gpu / hm_gpu.norm(dim=1, keepdim=True).clamp(min=1e-8)
        sim = (hm_norm @ hm_norm.T).cpu().numpy()
        del hm_gpu, hm_norm
        torch.cuda.empty_cache()
    else:
        sim = cosine_similarity(U_svd[idx_hm])

    g = sns.clustermap(
        sim, cmap="YlOrRd", figsize=(11, 10),
        xticklabels=False, yticklabels=False,
        method="ward", linewidths=0,
    )
    g.fig.suptitle(
        f"Similarité cosinus ({n_hm} utilisateurs, Ward) — {path}",
        fontsize=13, fontweight="bold", y=1.02,
    )
    plt.show()
    print(f"    Heatmap done ({time.time()-t0:.1f}s)")

    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    #  VIZ 3 — Power-law degree distribution (log-log)
    #
    #  PURPOSE: Visualize the "long tail" phenomenon precisely.
    #  X-axis: degree (# reviews a book received, or # reviews a
    #  user wrote). Y-axis: how many books/users have that degree.
    #  A straight line on log-log scale = power law.
    #  Slope α characterizes the tail: more negative = heavier tail.
    #  Typical real-world: α between -1.5 and -3.
    #  This matters because most books have very few reviews
    #  (cold start), while a few blockbusters dominate.
    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    deg_items = np.array((R > 0).sum(axis=0)).flatten()
    deg_users = np.array((R > 0).sum(axis=1)).flatten()

    print(f"\n  [VIZ 3] Power-law degree distribution")
    print(f"    Left: books — how many books have k reviews?")
    print(f"    Right: users — how many users wrote k reviews?")
    print(f"    A linear fit on log-log gives the power-law exponent α.")
    print(f"    INTERPRETATION: More negative α = heavier long tail =")
    print(f"    more items/users with very few interactions (cold start).")

    fig3, axes3 = plt.subplots(1, 2, figsize=(18, 6))
    for ax, deg, label, color in [
        (axes3[0], deg_items[deg_items > 0], "Livres", "steelblue"),
        (axes3[1], deg_users[deg_users > 0], "Utilisateurs", "darkorange"),
    ]:
        vals, counts = np.unique(deg, return_counts=True)
        ax.loglog(vals, counts, ".", markersize=4, alpha=0.5, color=color)
        slope, intercept = np.polyfit(np.log10(vals), np.log10(counts), 1)
        x_fit = np.logspace(np.log10(vals.min()), np.log10(vals.max()), 100)
        ax.loglog(x_fit, 10**intercept * x_fit**slope,
                  "r--", lw=1, label=f"α = {slope:.2f}")
        ax.set_xlabel("Degré (nombre de reviews)")
        ax.set_ylabel("Fréquence")
        ax.set_title(f"Distribution des degrés — {label}")
        ax.legend()

    plt.suptitle(f"Loi de puissance — {path}", fontsize=13, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.show()

    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    #  VIZ 4 — Silhouette analysis for user clustering
    #
    #  PURPOSE: Determine the optimal number of user profiles (K).
    #  KMeans groups users into K clusters in the SVD embedding space.
    #  The silhouette score [-1, 1] measures cluster cohesion vs
    #  separation: higher = tighter, more distinct clusters.
    #  Left panel: avg silhouette vs K → pick the peak.
    #  Right panel: per-user silhouette at optimal K → wide positive
    #  bars = well-assigned users; negative = misclassified.
    #  KMeans runs on GPU (torch_kmeans), silhouette on CPU (sklearn).
    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    n_sil = min(N_USERS_SILHOUETTE, n_users)
    idx_sil = rng.choice(n_users, size=n_sil, replace=False)
    U_sil = U_svd[idx_sil]

    print(f"\n  [VIZ 4] Silhouette analysis — k={K_RANGE.start}..{K_RANGE.stop-1}")
    print(f"    Clustering {n_sil:,} users in {n_comp}-dim SVD space")
    print(f"    KMeans on GPU, silhouette scoring on CPU")
    print(f"    INTERPRETATION: Peak silhouette score → optimal K.")
    print(f"    High silhouette (>0.5) → well-separated user profiles.")
    print(f"    This K can drive cluster-specific recommendation strategies.")

    sil_avgs = []
    for k in K_RANGE:
        if USE_GPU:
            labels = torch_kmeans(U_sil, n_clusters=k, seed=42)
        else:
            from sklearn.cluster import MiniBatchKMeans
            km = MiniBatchKMeans(n_clusters=k, random_state=42, batch_size=2048)
            labels = km.fit_predict(U_sil)
        s = silhouette_score(U_sil, labels, sample_size=min(10_000, n_sil))
        sil_avgs.append(s)
        print(f"      k={k:2d}  silhouette={s:.4f}")

    best_k = list(K_RANGE)[int(np.argmax(sil_avgs))]
    print(f"    → Best k = {best_k} (silhouette = {max(sil_avgs):.4f})")

    fig4, axes4 = plt.subplots(1, 2, figsize=(18, 7))
    axes4[0].plot(list(K_RANGE), sil_avgs, "o-", color="steelblue", lw=1.5)
    axes4[0].axvline(best_k, color="red", ls="--", lw=0.8, label=f"Meilleur k = {best_k}")
    axes4[0].set_xlabel("Nombre de clusters (k)")
    axes4[0].set_ylabel("Score silhouette moyen")
    axes4[0].set_title("Silhouette moyenne vs k")
    axes4[0].legend()

    # Detailed silhouette diagram on a subsample (O(n²) cost)
    n_detail = min(N_SILHOUETTE_DETAIL, n_sil)
    idx_detail = rng.choice(n_sil, size=n_detail, replace=False)
    U_detail = U_sil[idx_detail]

    if USE_GPU:
        labels_detail = torch_kmeans(U_detail, n_clusters=best_k, seed=42)
    else:
        km_best = MiniBatchKMeans(n_clusters=best_k, random_state=42, batch_size=2048)
        labels_detail = km_best.fit_predict(U_detail)

    sample_sil = silhouette_samples(U_detail, labels_detail)

    y_lower = 0
    for i in range(best_k):
        cluster_vals = np.sort(sample_sil[labels_detail == i])
        y_upper = y_lower + len(cluster_vals)
        axes4[1].fill_betweenx(
            np.arange(y_lower, y_upper), 0, cluster_vals,
            facecolor=cm.tab10(i / best_k), alpha=0.7, label=f"Cluster {i}",
        )
        y_lower = y_upper + 10

    axes4[1].axvline(sil_avgs[best_k - K_RANGE.start], color="red", ls="--", lw=0.8)
    axes4[1].set_xlabel("Coefficient de silhouette")
    axes4[1].set_ylabel("Utilisateurs (triés par cluster)")
    axes4[1].set_title(f"Diagramme de silhouette — k={best_k} ({n_detail:,} users)")
    axes4[1].legend(loc="lower right", fontsize=8)

    plt.suptitle(f"Clustering utilisateurs — {path}",
                 fontsize=13, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.show()
    print(f"    Clustering done ({time.time()-t0:.1f}s)")

    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    #  VIZ 5 — Correlation: Rating ↔ Helpful Votes
    #
    #  PURPOSE: Investigate whether extreme ratings (1 or 5 stars)
    #  attract more "helpful" votes from the community.
    #  Left panel: mean helpful votes per rating level (with std bars).
    #  Right panel: 2D density (hexbin) of rating vs log(helpful_vote)
    #  for reviews that received at least one vote.
    #  If low-star reviews get more votes, they carry a stronger
    #  quality signal and could be weighted higher in training.
    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    print(f"\n  [VIZ 5] Rating vs Helpful Votes correlation")
    print(f"    Left: average helpful votes per rating level (1-5 stars)")
    print(f"    Right: density plot for reviews with ≥1 vote")
    print(f"    INTERPRETATION: If extreme ratings (1★, 5★) get more votes,")
    print(f"    the community finds polarized opinions more useful — this")
    print(f"    signal could weight reviews in a recommendation model.")

    fig5, axes5 = plt.subplots(1, 2, figsize=(18, 6))

    agg = (df.groupby("rating")["helpful_vote"]
             .agg(["mean", "median", "std"]).reset_index())

    axes5[0].bar(agg["rating"], agg["mean"], color="orchid", edgecolor="white")
    axes5[0].errorbar(agg["rating"], agg["mean"], yerr=agg["std"],
                      fmt="none", color="black", capsize=4)
    for _, row in agg.iterrows():
        axes5[0].text(row["rating"], row["mean"] + row["std"] + 0.1,
                      f'μ={row["mean"]:.1f}', ha="center", fontsize=9)
    axes5[0].set_xlabel("Note (rating)")
    axes5[0].set_ylabel("Votes utiles (moyenne ± σ)")
    axes5[0].set_title("Votes d'utilité moyens par note")
    axes5[0].set_xticks([1, 2, 3, 4, 5])

    df_nz = df[df["helpful_vote"] > 0]
    if len(df_nz) > 0:
        hb = axes5[1].hexbin(
            df_nz["rating"], np.log1p(df_nz["helpful_vote"]),
            gridsize=25, cmap="YlOrRd", mincnt=1,
        )
        plt.colorbar(hb, ax=axes5[1], label="Nb de reviews")
    axes5[1].set_xlabel("Note (rating)")
    axes5[1].set_ylabel("log(1 + helpful_vote)")
    axes5[1].set_title(f"Densité: Note vs Votes ({len(df_nz):,} reviews avec votes)")

    plt.suptitle(f"Corrélation Note ↔ Votes d'utilité — {path}",
                 fontsize=13, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.show()

    print(f"\n  TOTAL: {time.time()-t0:.1f}s")
    print(f"{'=' * 70}\n")

    del df, R, U_svd, U_sil
    torch.cuda.empty_cache()
    gc.collect()

## 3.1.3 Prétraitement


- Nettoyage des ratings
- Filtrage utilisateurs/items
- Construction matrice utilisateur-item (CSR)
- Split train/test (80/20 stratifié)

### 1) Nettoyage des Données

 - Remove reviews without rating 
 - Handle invalid timestamps
 - Convert rating to float

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# NETTOYAGE DES DONNÉES (Data Preprocessing)
#
# Ce script parcourt tous les fichiers Parquet d'échantillons et
# applique trois étapes de nettoyage :
#   1. Suppression des reviews sans note (rating manquant ou invalide)
#   2. Filtrage des timestamps invalides (hors plage temporelle)
#   3. Conversion du rating en float pour cohérence numérique
#
# À la fin, chaque fichier est sauvegardé en version nettoyée.
# ══════════════════════════════════════════════════════════════════════

SAMPLE_PATHS = sorted(glob.glob("sample-*/*.parquet"))

# Amazon a été fondé en juillet 1995 — aucune review ne peut être antérieure.
# On borne aussi par le futur pour exclure les timestamps manifestement erronés.
MIN_DATE = pd.Timestamp("1995-07-01")
MAX_DATE = pd.Timestamp("2025-12-31")

for path in SAMPLE_PATHS:
    # ── Garde : fichiers trop petits (probablement corrompus) ──────
    if os.path.getsize(path) < 1024:
        print(f"  ⚠ Fichier ignoré : {path} ({os.path.getsize(path)} octets — probablement corrompu)")
        continue

    print(f"\n{'═' * 60}")
    print(f"  Fichier : {path}")
    print(f"{'═' * 60}")

    df = pd.read_parquet(path)
    n_original = len(df)

    if n_original == 0:
        print("  ⚠ Fichier vide, passage au suivant.")
        continue

    print(f"  Nombre initial de reviews : {n_original:,}")

    # ──────────────────────────────────────────────────────────────────
    # ÉTAPE 1 : Suppression des reviews sans note valide
    # ──────────────────────────────────────────────────────────────────
    # Deux cas possibles dans le jeu de données Amazon :
    #   - Le champ 'rating' est absent du JSON original → NaN en Pandas
    #   - Le champ 'rating' vaut 0, ce qui est hors de l'échelle 1–5
    # Dans les deux cas, la review est inutilisable pour un système de
    # recommandation, on la supprime.
    # ──────────────────────────────────────────────────────────────────

    n_nan_rating = df["rating"].isna().sum()
    n_zero_rating = (df["rating"] == 0).sum()

    df = df.dropna(subset=["rating"])
    df = df[df["rating"] > 0]
    n_after_rating = len(df)
    n_dropped_rating = n_original - n_after_rating

    print(f"\n  ── Étape 1 : Nettoyage des notes (rating) ──")
    print(f"     Reviews avec rating NaN   : {n_nan_rating:,}")
    print(f"     Reviews avec rating == 0  : {n_zero_rating:,}")
    print(f"     Total supprimées          : {n_dropped_rating:,}")
    print(f"     Reviews restantes         : {n_after_rating:,}")
    if n_dropped_rating == 0:
        print(f"     ✓ Aucune review sans note — le jeu de données est propre sur ce critère.")
    else:
        print(f"     → {n_dropped_rating / n_original * 100:.2f}% des reviews n'avaient pas de note valide.")

    # ──────────────────────────────────────────────────────────────────
    # ÉTAPE 2 : Filtrage des timestamps invalides
    # ──────────────────────────────────────────────────────────────────
    # Les timestamps sont stockés en millisecondes Unix (ms depuis
    # le 1er janvier 1970). On les convertit en datetime pour vérifier
    # qu'ils tombent dans une plage plausible :
    #   - Borne inférieure : 1er juillet 1995 (fondation d'Amazon)
    #   - Borne supérieure : 31 décembre 2025
    #
    # errors="coerce" transforme les valeurs non convertibles en NaT
    # (Not a Time), qui sont ensuite exclues par le filtre between().
    #
    # Cas détectés :
    #   - Timestamps nuls ou manquants → NaT après conversion
    #   - Timestamps négatifs → dates avant 1970, hors plage
    #   - Timestamps en secondes au lieu de millisecondes → dates
    #     autour de 1970, également hors plage
    #   - Timestamps dans le futur → données erronées
    # ──────────────────────────────────────────────────────────────────

    n_null_ts = df["timestamp"].isna().sum()
    df["_date"] = pd.to_datetime(df["timestamp"], unit="ms", errors="coerce")
    n_nat = df["_date"].isna().sum()
    n_before_min = (df["_date"] < MIN_DATE).sum()
    n_after_max = (df["_date"] > MAX_DATE).sum()

    df = df[df["_date"].between(MIN_DATE, MAX_DATE)]
    df.drop(columns=["_date"], inplace=True)
    n_after_ts = len(df)
    n_dropped_ts = n_after_rating - n_after_ts

    print(f"\n  ── Étape 2 : Nettoyage des timestamps ──")
    print(f"     Timestamps nuls/manquants          : {n_null_ts:,}")
    print(f"     Non convertibles (→ NaT)           : {n_nat:,}")
    print(f"     Avant {MIN_DATE.date()} (pré-Amazon) : {n_before_min:,}")
    print(f"     Après {MAX_DATE.date()} (futur)      : {n_after_max:,}")
    print(f"     Total supprimées                   : {n_dropped_ts:,}")
    print(f"     Reviews restantes                  : {n_after_ts:,}")
    if n_dropped_ts == 0:
        print(f"     ✓ Tous les timestamps sont valides — aucune suppression nécessaire.")
    else:
        print(f"     → {n_dropped_ts / n_after_rating * 100:.2f}% des reviews avaient un timestamp invalide.")

    # ──────────────────────────────────────────────────────────────────
    # ÉTAPE 3 : Conversion du rating en float
    # ──────────────────────────────────────────────────────────────────
    # Selon la méthode d'écriture du Parquet, le rating peut être stocké
    # en int64, int32, ou déjà en float. On force float64 pour :
    #   - Garantir la cohérence entre tous les échantillons
    #   - Éviter les erreurs de division entière dans les calculs de
    #     moyennes et corrélations en aval
    # ──────────────────────────────────────────────────────────────────

    dtype_before = df["rating"].dtype
    df["rating"] = df["rating"].astype(float)
    dtype_after = df["rating"].dtype

    print(f"\n  ── Étape 3 : Conversion du type de rating ──")
    print(f"     Type avant conversion : {dtype_before}")
    print(f"     Type après conversion : {dtype_after}")
    if dtype_before == dtype_after:
        print(f"     ✓ Le rating était déjà en {dtype_after} — aucune conversion nécessaire.")
    else:
        print(f"     → Converti de {dtype_before} vers {dtype_after}.")

    # ══════════════════════════════════════════════════════════════════
    # RÉSUMÉ FINAL
    # ══════════════════════════════════════════════════════════════════

    n_final = len(df)
    n_total_dropped = n_original - n_final

    print(f"\n  {'─' * 56}")
    print(f"  RÉSUMÉ — {path}")
    print(f"  {'─' * 56}")
    print(f"     Reviews initiales     : {n_original:,}")
    print(f"     Supprimées (rating)   : {n_dropped_rating:,}")
    print(f"     Supprimées (timestamp): {n_dropped_ts:,}")
    print(f"     Reviews finales       : {n_final:,}")
    print(f"     Taux de rétention     : {n_final / n_original * 100:.2f}%")
    print(f"     Utilisateurs restants : {df['user_id'].nunique():,}")
    print(f"     Livres restants       : {df['parent_asin'].nunique():,}")
    print(f"  {'─' * 56}")

    # ── Sauvegarde (décommenter quand prêt) ────────────────────────
    df.to_parquet(path, index=False)
    print(f"  ✓ Fichier nettoyé sauvegardé : {path}")

### 2) Filtrage

- Minimum number of ratings per user: 10–20
- Minimum number of ratings per book: 5–10
- Recalculate the sparsity rate after filtering

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# FILTRAGE PAR SEUILS D'ACTIVITÉ (Threshold Filtering)
#
# Objectif : éliminer les utilisateurs et les livres ayant trop peu
# d'interactions. En filtrage collaboratif, un utilisateur avec 1–2 notes
# ne fournit aucun signal exploitable (problème du « cold start »), et un
# livre noté par un seul lecteur ne peut pas être recommandé par similarité.
#
# ── Choix des seuils ──────────────────────────────────────────────────────
#
# MIN_RATINGS_USER = 20
#   Notre échantillon « cudf-claude » a été construit en ne gardant que
#   les utilisateurs ayant ≥ 20 reviews dans le jeu COMPLET (~27M).
#   Toutefois, après le nettoyage précédent (rating/timestamps) et après
#   le filtrage des livres ci-dessous, certains utilisateurs pourraient
#   passer sous ce seuil. On réapplique donc 20 pour maintenir la
#   cohérence avec le critère d'échantillonnage initial.
#   ► Justification : 20 est le seuil standard dans la littérature
#     sur les systèmes de recommandation (cf. Koren 2008, He et al. 2017).
#     Il garantit un profil utilisateur assez riche pour que les algorithmes
#     de voisinage (kNN) ou de factorisation (SVD/ALS) convergent.
#
# MIN_RATINGS_BOOK = 5
#   Un livre noté par < 5 lecteurs est statistiquement « invisible » :
#   sa moyenne est instable et la co-occurrence avec d'autres livres est
#   trop faible pour contribuer aux vecteurs de similarité.
#   ► Justification : le seuil de 5 est couramment utilisé dans les
#     benchmarks Amazon (McAuley et al. 2015, Ni et al. 2019).
#     On privilégie 5 plutôt que 10 pour conserver un catalogue plus
#     large et limiter la perte de diversité (long tail).
#
# ── Filtrage itératif ─────────────────────────────────────────────────────
#
# IMPORTANT : le filtrage n'est PAS une opération unique.
# Supprimer des livres rares peut faire descendre certains utilisateurs
# sous le seuil, et inversement. On boucle donc jusqu'à convergence
# (plus aucune suppression à effectuer). En pratique, 2–4 itérations
# suffisent.
# ══════════════════════════════════════════════════════════════════════════

SAMPLE_PATHS = sorted(glob.glob("sample-*/*.parquet"))

MIN_RATINGS_USER = 20
MIN_RATINGS_BOOK = 5

for path in SAMPLE_PATHS:
    if os.path.getsize(path) < 1024:
        print(f"  ⚠ Fichier ignoré : {path} (trop petit)")
        continue

    print(f"\n{'═' * 70}")
    print(f"  Fichier : {path}")
    print(f"{'═' * 70}")

    df = pd.read_parquet(path)
    n_before_filter = len(df)
    u_before = df["user_id"].nunique()
    b_before = df["parent_asin"].nunique()

    if n_before_filter == 0:
        print("  ⚠ Fichier vide, passage au suivant.")
        continue

    # ──────────────────────────────────────────────────────────────────
    # Calcul de la sparsité AVANT filtrage
    # ──────────────────────────────────────────────────────────────────
    # La sparsité mesure le pourcentage de cases vides dans la matrice
    # utilisateur × livre. Formule :
    #   sparsité = 1 − |R| / (|U| × |I|)
    # où |R| = nombre de ratings, |U| = nb d'utilisateurs, |I| = nb de livres.
    #
    # Une sparsité de 99.99% signifie que seule 0.01% de la matrice est
    # remplie — c'est typique des grands jeux de données de recommandation.
    # ──────────────────────────────────────────────────────────────────

    sparsity_before = 1 - n_before_filter / (u_before * b_before)

    print(f"\n  ── État AVANT filtrage ──")
    print(f"     Reviews       : {n_before_filter:,}")
    print(f"     Utilisateurs  : {u_before:,}")
    print(f"     Livres        : {b_before:,}")
    print(f"     Sparsité      : {sparsity_before * 100:.4f}%")
    print(f"     Densité       : {(1 - sparsity_before) * 100:.6f}%")

    # ──────────────────────────────────────────────────────────────────
    # Diagnostic pré-filtrage : distribution de l'activité
    # ──────────────────────────────────────────────────────────────────
    # On affiche combien d'utilisateurs/livres seraient éliminés pour
    # donner une idée de l'impact avant de filtrer.
    # ──────────────────────────────────────────────────────────────────

    ratings_per_user = df.groupby("user_id").size()
    ratings_per_book = df.groupby("parent_asin").size()

    n_users_below = (ratings_per_user < MIN_RATINGS_USER).sum()
    n_books_below = (ratings_per_book < MIN_RATINGS_BOOK).sum()

    print(f"\n  ── Diagnostic pré-filtrage ──")
    print(f"     Seuil utilisateur : ≥ {MIN_RATINGS_USER} ratings")
    print(f"     Seuil livre       : ≥ {MIN_RATINGS_BOOK} ratings")
    print(f"     Utilisateurs sous le seuil : {n_users_below:,} / {u_before:,}"
          f" ({n_users_below / u_before * 100:.1f}%)")
    print(f"     Livres sous le seuil       : {n_books_below:,} / {b_before:,}"
          f" ({n_books_below / b_before * 100:.1f}%)")

    # ──────────────────────────────────────────────────────────────────
    # Filtrage itératif jusqu'à convergence
    # ──────────────────────────────────────────────────────────────────
    # À chaque itération :
    #   1. On supprime les livres ayant < MIN_RATINGS_BOOK notes
    #   2. On supprime les utilisateurs ayant < MIN_RATINGS_USER notes
    #   3. Si rien n'a changé → on a convergé, on arrête
    #
    # On limite à 20 itérations par sécurité (jamais atteint en pratique).
    # ──────────────────────────────────────────────────────────────────

    print(f"\n  ── Filtrage itératif ──")
    MAX_ITER = 20

    for iteration in range(1, MAX_ITER + 1):
        n_start = len(df)

        # Filtrer les livres avec trop peu de notes
        book_counts = df.groupby("parent_asin").size()
        books_ok = book_counts[book_counts >= MIN_RATINGS_BOOK].index
        df = df[df["parent_asin"].isin(books_ok)]
        n_after_books = len(df)
        dropped_books_reviews = n_start - n_after_books

        # Filtrer les utilisateurs avec trop peu de notes
        user_counts = df.groupby("user_id").size()
        users_ok = user_counts[user_counts >= MIN_RATINGS_USER].index
        df = df[df["user_id"].isin(users_ok)]
        n_after_users = len(df)
        dropped_users_reviews = n_after_books - n_after_users

        total_dropped = n_start - n_after_users

        print(f"     Itération {iteration:>2d} : "
              f"−{dropped_books_reviews:,} (livres) "
              f"−{dropped_users_reviews:,} (users) "
              f"→ {n_after_users:,} reviews restantes")

        if total_dropped == 0:
            print(f"     ✓ Convergence atteinte à l'itération {iteration}.")
            break
    else:
        print(f"     ⚠ Limite de {MAX_ITER} itérations atteinte sans convergence.")

    # ──────────────────────────────────────────────────────────────────
    # Calcul de la sparsité APRÈS filtrage
    # ──────────────────────────────────────────────────────────────────

    n_after_filter = len(df)
    u_after = df["user_id"].nunique()
    b_after = df["parent_asin"].nunique()

    if u_after > 0 and b_after > 0:
        sparsity_after = 1 - n_after_filter / (u_after * b_after)
    else:
        sparsity_after = 1.0

    # ──────────────────────────────────────────────────────────────────
    # Résumé comparatif AVANT / APRÈS
    # ──────────────────────────────────────────────────────────────────

    print(f"\n  {'─' * 66}")
    print(f"  RÉSUMÉ DU FILTRAGE — {path}")
    print(f"  {'─' * 66}")
    print(f"  {'':30s} {'AVANT':>14s}   {'APRÈS':>14s}   {'Δ':>10s}")
    print(f"  {'─' * 66}")

    delta_r = n_after_filter - n_before_filter
    delta_u = u_after - u_before
    delta_b = b_after - b_before

    print(f"  {'Reviews':30s} {n_before_filter:>14,}   {n_after_filter:>14,}   {delta_r:>+10,}")
    print(f"  {'Utilisateurs':30s} {u_before:>14,}   {u_after:>14,}   {delta_u:>+10,}")
    print(f"  {'Livres':30s} {b_before:>14,}   {b_after:>14,}   {delta_b:>+10,}")
    print(f"  {'Sparsité (%)':30s} {sparsity_before * 100:>13.4f}%   {sparsity_after * 100:>13.4f}%")
    print(f"  {'Densité (%)':30s} {(1 - sparsity_before) * 100:>13.6f}%   {(1 - sparsity_after) * 100:>13.6f}%")
    print(f"  {'─' * 66}")

    # ── Interprétation automatique des résultats ──────────────────────

    pct_reviews_kept = n_after_filter / n_before_filter * 100 if n_before_filter > 0 else 0
    pct_users_kept = u_after / u_before * 100 if u_before > 0 else 0
    pct_books_kept = b_after / b_before * 100 if b_before > 0 else 0
    density_gain = ((1 - sparsity_after) / (1 - sparsity_before) - 1) * 100 if sparsity_before < 1 else 0

    print(f"\n  ── Interprétation ──")
    print(f"     Taux de rétention des reviews       : {pct_reviews_kept:.1f}%")
    print(f"     Taux de rétention des utilisateurs   : {pct_users_kept:.1f}%")
    print(f"     Taux de rétention des livres         : {pct_books_kept:.1f}%")
    print(f"     Gain de densité                      : ×{density_gain / 100 + 1:.1f} ({density_gain:+.1f}%)")
    print()
    print(f"     La matrice U×I est passée de {u_before:,}×{b_before:,} = {u_before * b_before:,} cases")
    print(f"     à {u_after:,}×{b_after:,} = {u_after * b_after:,} cases.")
    print(f"     En éliminant les livres rares (< {MIN_RATINGS_BOOK} notes) et les utilisateurs")
    print(f"     peu actifs (< {MIN_RATINGS_USER} notes), on concentre le signal utile")
    print(f"     sur un sous-ensemble plus dense, ce qui améliore directement")
    print(f"     la qualité des recommandations par filtrage collaboratif.")

    # ── Sauvegarde (décommenter quand prêt) ────────────────────────
    df.to_parquet(path, index=False)
    print(f"\n  ✓ Fichier filtré sauvegardé : {path}")

### 3) Matrice utilisateur-item

- Build the matrix R ∈ R^(|U|×|I|) where r_(u,i) represents the rating of user u for book i

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# CONSTRUCTION DE LA MATRICE UTILISATEUR-LIVRE (User-Item Matrix)
#
# On construit la matrice R ∈ ℝ^(|U| × |I|) où r(u,i) = note de
# l'utilisateur u pour le livre i, et 0 si l'utilisateur n'a pas noté
# ce livre.
#
# ── Pourquoi une matrice creuse (sparse) ? ────────────────────────────────
#
# Même après filtrage, la matrice est extrêmement creuse :
#   - Si |U| = 10 000 et |I| = 20 000, la matrice dense ferait
#     10 000 × 20 000 = 200 000 000 entrées × 8 octets = ~1.5 Go
#   - Or seules ~500 000 cases sont remplies (< 0.25% de la matrice)
#   - En format CSR, on ne stocke QUE les valeurs non nulles + leurs
#     indices → ~500 000 × 12 octets ≈ 6 Mo (250× moins de mémoire)
#
# ── Format CSR (Compressed Sparse Row) ────────────────────────────────────
#
# scipy.sparse.csr_matrix stocke la matrice via 3 tableaux :
#   - data[]    : les valeurs non nulles (les ratings)
#   - indices[] : l'indice de colonne de chaque valeur
#   - indptr[]  : pour chaque ligne i, data[indptr[i]:indptr[i+1]]
#                 contient les valeurs de la ligne i
#
# Avantages du CSR :
#   - Accès rapide par ligne (O(1) pour récupérer tous les ratings d'un user)
#   - Multiplication matrice-vecteur efficace (cœur de SVD/ALS)
#   - Compatible avec scikit-learn, surprise, implicit, etc.
#
# ── Stratégie d'encodage des identifiants ─────────────────────────────────
#
# Les user_id et parent_asin sont des chaînes de caractères (ex:
# "AHJKFDS83KD", "B00005N5PF"). Il faut les convertir en indices
# entiers 0...|U|-1 et 0...|I|-1 pour indexer la matrice.
#
# Approche classique (dict Python) :
#   user_map = {uid: i for i, uid in enumerate(df["user_id"].unique())}
#   → Lent pour > 100k identifiants (boucle Python pure)
#
# Approche optimisée (pd.factorize) :
#   codes, uniques = pd.factorize(df["user_id"])
#   → Implémenté en C dans Pandas, ~10-50× plus rapide
#   → Produit directement les indices entiers et la table de correspondance
#
# Les deux approches produisent une matrice CSR identique.
# ══════════════════════════════════════════════════════════════════════════

SAMPLE_PATHS = sorted(glob.glob("sample-*/*.parquet"))

# On stocke les matrices et mappings dans un dictionnaire pour usage ultérieur
matrices = {}

for path in SAMPLE_PATHS:
    if os.path.getsize(path) < 1024:
        continue

    print(f"\n{'═' * 70}")
    print(f"  Matrice utilisateur-livre : {path}")
    print(f"{'═' * 70}")

    df = pd.read_parquet(path)

    if len(df) == 0:
        print("  ⚠ Fichier vide, passage au suivant.")
        continue

    # ──────────────────────────────────────────────────────────────────
    # ÉTAPE 1 : Encodage des identifiants avec pd.factorize
    # ──────────────────────────────────────────────────────────────────
    # pd.factorize attribue un entier consécutif (0, 1, 2, ...) à chaque
    # valeur unique, dans l'ordre de première apparition.
    # Retourne :
    #   - codes : array d'entiers (même longueur que la colonne)
    #   - uniques : array des valeurs originales (table de correspondance)
    #
    # Ainsi : uniques[codes[i]] == df["user_id"].iloc[i]  (bijectif)
    # ──────────────────────────────────────────────────────────────────

    t0 = time.perf_counter()

    user_codes, user_ids = pd.factorize(df["user_id"], sort=False)
    item_codes, item_ids = pd.factorize(df["parent_asin"], sort=False)
    ratings = df["rating"].values.astype(np.float32)

    n_users = len(user_ids)
    n_items = len(item_ids)
    n_ratings = len(ratings)

    t_encode = time.perf_counter() - t0

    print(f"\n  ── Étape 1 : Encodage des identifiants (pd.factorize) ──")
    print(f"     Utilisateurs encodés : {n_users:,}  (indices 0 à {n_users - 1:,})")
    print(f"     Livres encodés       : {n_items:,}  (indices 0 à {n_items - 1:,})")
    print(f"     Ratings à insérer    : {n_ratings:,}")
    print(f"     Temps d'encodage     : {t_encode * 1000:.1f} ms")
    print(f"     → pd.factorize est implémenté en C dans Pandas ; un dict")
    print(f"       Python serait ~10-50× plus lent sur {n_ratings:,} entrées.")

    # ──────────────────────────────────────────────────────────────────
    # ÉTAPE 2 : Construction de la matrice CSR
    # ──────────────────────────────────────────────────────────────────
    # On passe par le constructeur COO (Coordinate format) de scipy :
    #   csr_matrix((data, (row, col)), shape=(n_rows, n_cols))
    #
    # En interne, scipy convertit automatiquement en CSR.
    # Si un même couple (u, i) apparaît plusieurs fois (doublons),
    # scipy ADDITIONNE les valeurs. On vérifie et gère ce cas.
    #
    # Note : on utilise float32 au lieu de float64 pour les ratings.
    # Les notes étant des entiers 1–5, float32 (7 décimales) est
    # largement suffisant et divise l'empreinte mémoire par 2.
    # ──────────────────────────────────────────────────────────────────

    t1 = time.perf_counter()

    # Vérification des doublons (un utilisateur ayant noté 2 fois le même livre)
    n_duplicates = df.duplicated(subset=["user_id", "parent_asin"]).sum()

    if n_duplicates > 0:
        # En cas de doublons, on garde la note la plus récente (dernière)
        print(f"\n  ⚠ {n_duplicates:,} doublons détectés (même utilisateur + même livre)")
        print(f"    → On conserve la note la plus récente (dernière occurrence).")
        df = df.drop_duplicates(subset=["user_id", "parent_asin"], keep="last")
        user_codes, user_ids = pd.factorize(df["user_id"], sort=False)
        item_codes, item_ids = pd.factorize(df["parent_asin"], sort=False)
        ratings = df["rating"].values.astype(np.float32)
        n_users = len(user_ids)
        n_items = len(item_ids)
        n_ratings = len(ratings)

    R = csr_matrix(
        (ratings, (user_codes, item_codes)),
        shape=(n_users, n_items),
        dtype=np.float32,
    )

    t_build = time.perf_counter() - t1

    print(f"\n  ── Étape 2 : Construction de la matrice CSR ──")
    print(f"     Dimensions           : {R.shape[0]:,} × {R.shape[1]:,}")
    print(f"     Entrées non nulles   : {R.nnz:,}")
    print(f"     Doublons (u, i)      : {n_duplicates:,}")
    print(f"     Temps de construction: {t_build * 1000:.1f} ms")

    # ──────────────────────────────────────────────────────────────────
    # ÉTAPE 3 : Analyse mémoire et validation
    # ──────────────────────────────────────────────────────────────────
    # On compare l'empreinte mémoire de la matrice creuse à celle
    # qu'aurait une matrice dense de mêmes dimensions.
    # ──────────────────────────────────────────────────────────────────

    mem_data = R.data.nbytes
    mem_indices = R.indices.nbytes
    mem_indptr = R.indptr.nbytes
    mem_sparse_total = mem_data + mem_indices + mem_indptr

    mem_dense = n_users * n_items * np.dtype(np.float32).itemsize

    sparsity = 1 - R.nnz / (n_users * n_items)

    print(f"\n  ── Étape 3 : Analyse mémoire ──")
    print(f"     Matrice creuse (CSR) :")
    print(f"       data[]    ({R.data.dtype})  : {mem_data / 1024**2:>8.2f} Mo  ({R.nnz:,} valeurs)")
    print(f"       indices[] ({R.indices.dtype}) : {mem_indices / 1024**2:>8.2f} Mo  (indices de colonnes)")
    print(f"       indptr[]  ({R.indptr.dtype}) : {mem_indptr / 1024**2:>8.2f} Mo  ({n_users + 1:,} pointeurs de lignes)")
    print(f"       TOTAL CSR            : {mem_sparse_total / 1024**2:>8.2f} Mo")
    print(f"     Matrice dense équivalente :")
    print(f"       {n_users:,} × {n_items:,} × 4 octets : {mem_dense / 1024**2:>8.2f} Mo")
    print(f"     Ratio de compression       : {mem_dense / mem_sparse_total:>8.1f}×")
    print(f"     → La représentation CSR est {mem_dense / mem_sparse_total:.0f} fois plus compacte")
    print(f"       que la matrice dense pour une sparsité de {sparsity * 100:.2f}%.")

    # ──────────────────────────────────────────────────────────────────
    # Validation de la matrice
    # ──────────────────────────────────────────────────────────────────
    # On vérifie que les ratings dans la matrice correspondent bien
    # aux ratings originaux, et que les bornes sont respectées.
    # ──────────────────────────────────────────────────────────────────

    min_val = R.data.min()
    max_val = R.data.max()
    mean_val = R.data.mean()

    print(f"\n  ── Validation ──")
    print(f"     Rating min dans R  : {min_val:.1f}  (attendu : 1.0)")
    print(f"     Rating max dans R  : {max_val:.1f}  (attendu : 5.0)")
    print(f"     Rating moyen       : {mean_val:.2f}")
    print(f"     Nb ratings / user  : min={np.diff(R.indptr).min()}, "
          f"max={np.diff(R.indptr).max()}, "
          f"moy={np.diff(R.indptr).mean():.1f}")

    if 1.0 <= min_val and max_val <= 5.0:
        print(f"     ✓ Tous les ratings sont dans l'intervalle [1, 5] — matrice valide.")
    else:
        print(f"     ⚠ ATTENTION : des ratings hors de [1, 5] détectés !")

    # ──────────────────────────────────────────────────────────────────
    # Stockage pour usage ultérieur
    # ──────────────────────────────────────────────────────────────────
    # On conserve la matrice R, ainsi que les tables de correspondance
    # (user_ids, item_ids) qui permettent de retrouver les identifiants
    # originaux à partir des indices de la matrice.
    #
    # Exemple d'utilisation :
    #   user_ids[42]  → "AHJKFDS83KD"   (ID Amazon de l'utilisateur 42)
    #   item_ids[7]   → "B00005N5PF"     (ASIN du livre en colonne 7)
    #   R[42, 7]      → 4.0             (note donnée par cet utilisateur)
    # ──────────────────────────────────────────────────────────────────

    matrices[path] = {
        "R": R,
        "user_ids": user_ids,
        "item_ids": item_ids,
    }

    print(f"\n  {'─' * 66}")
    print(f"  RÉSUMÉ — {path}")
    print(f"  {'─' * 66}")
    print(f"     Matrice R       : {R.shape[0]:,} utilisateurs × {R.shape[1]:,} livres")
    print(f"     Ratings stockés : {R.nnz:,}")
    print(f"     Sparsité        : {sparsity * 100:.2f}%")
    print(f"     Mémoire CSR     : {mem_sparse_total / 1024**2:.2f} Mo")
    print(f"     Mémoire dense   : {mem_dense / 1024**2:.2f} Mo (économie : {mem_dense / mem_sparse_total:.0f}×)")
    print(f"     Temps total     : {(t_encode + t_build) * 1000:.1f} ms")
    print(f"  {'─' * 66}")

print(f"\n✓ {len(matrices)} matrice(s) construite(s) et stockée(s) dans `matrices`.")

### 4) Division en ensembles d'entraînement et de test

#### A) Division des données

Split the data into:
- Training set: 80% of ratings
- Test set: 20% of ratings
- Stratify by user: each user must have at least one rating in each set

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# SÉPARATION TRAIN / TEST STRATIFIÉE PAR UTILISATEUR
#
# Objectif : diviser les ratings en 80% entraînement / 20% test,
# en garantissant que CHAQUE utilisateur possède au moins 1 rating
# dans chaque ensemble.
#
# ── Pourquoi stratifier par utilisateur ? ─────────────────────────────────
#
# En recommandation, on évalue la capacité du modèle à prédire les
# goûts d'un utilisateur CONNU à partir de son historique partiel.
# Si un utilisateur n'a aucun rating dans le train, le modèle ne peut
# rien apprendre de lui (cold start total). Si un utilisateur n'a
# aucun rating dans le test, on ne peut pas mesurer la qualité des
# prédictions le concernant.
#
# Un split aléatoire global (sans stratification) risquerait de
# concentrer tous les ratings d'un utilisateur rare dans un seul
# ensemble — d'où la nécessité de stratifier.
#
# ── Contrainte pour les utilisateurs avec peu de ratings ──────────────────
#
# Pour un utilisateur avec n ratings et un ratio test de 20% :
#   n_test  = max(1, floor(n × 0.2))   — au moins 1 dans le test
#   n_test  = min(n_test, n − 1)        — au moins 1 dans le train
#
# Cas limites :
#   n = 2 → n_test = 1, n_train = 1  (split 50/50, inévitable)
#   n = 5 → n_test = 1, n_train = 4  (split 80/20)
#   n = 10 → n_test = 2, n_train = 8  (split exact 80/20)
#   n = 50 → n_test = 10, n_train = 40  (split exact 80/20)
#
# Les utilisateurs avec n = 2 auront un split plus généreux pour le
# test (50/50 au lieu de 80/20), ce qui est le prix à payer pour
# respecter la contrainte « au moins 1 dans chaque ensemble ».
#
# ── Approche vectorisée (sans boucle Python sur les utilisateurs) ─────────
#
# L'approche naïve serait de boucler sur chaque utilisateur et de
# faire un random.sample par utilisateur → O(|U|) itérations Python,
# lent pour |U| > 10 000.
#
# Notre approche vectorisée :
#   1. Attribuer un nombre aléatoire à chaque rating
#   2. Trier par (user_id, aléatoire) pour mélanger intra-utilisateur
#   3. Calculer la position cumulative (rank) de chaque rating au sein
#      de son utilisateur via groupby().cumcount()
#   4. Calculer n_test pour chaque utilisateur de manière vectorisée
#   5. Les derniers n_test ratings de chaque utilisateur vont dans le
#      test, le reste dans le train
#
# Complexité : O(|R| log |R|) pour le tri, sans aucune boucle Python.
# Sur ~500 000 ratings, cela prend < 500 ms.
# ══════════════════════════════════════════════════════════════════════════

SAMPLE_PATHS = sorted(glob.glob("sample-*/*.parquet"))

TRAIN_RATIO = 0.80
TEST_RATIO = 1 - TRAIN_RATIO
SEED = 42

splits = {}

for path in SAMPLE_PATHS:
    if os.path.getsize(path) < 1024:
        continue

    print(f"\n{'═' * 70}")
    print(f"  Split Train/Test : {path}")
    print(f"{'═' * 70}")

    df = pd.read_parquet(path)

    if len(df) == 0:
        print("  ⚠ Fichier vide, passage au suivant.")
        continue

    # Dédoublonner (même logique que la cellule matrice)
    n_dup = df.duplicated(subset=["user_id", "parent_asin"]).sum()
    if n_dup > 0:
        df = df.drop_duplicates(subset=["user_id", "parent_asin"], keep="last")
        print(f"  {n_dup:,} doublons supprimés (même logique que la matrice CSR)")

    n_total = len(df)
    n_users = df["user_id"].nunique()
    n_items = df["parent_asin"].nunique()

    print(f"  Ratings : {n_total:,}  |  Utilisateurs : {n_users:,}  |  Livres : {n_items:,}")

    # ──────────────────────────────────────────────────────────────────
    # ÉTAPE 1 : Mélange aléatoire intra-utilisateur (vectorisé)
    # ──────────────────────────────────────────────────────────────────
    # On attribue un nombre aléatoire à chaque ligne, puis on trie
    # par (user_id, aléatoire). Cela revient à faire un shuffle
    # indépendant des ratings de chaque utilisateur, mais en une
    # seule opération de tri sur tout le DataFrame.
    # ──────────────────────────────────────────────────────────────────

    t0 = time.perf_counter()

    rng = np.random.RandomState(SEED)
    df["_rand"] = rng.random(len(df))
    df = df.sort_values(["user_id", "_rand"]).reset_index(drop=True)

    # ──────────────────────────────────────────────────────────────────
    # ÉTAPE 2 : Calcul vectorisé de la position et du seuil de split
    # ──────────────────────────────────────────────────────────────────
    # Pour chaque utilisateur :
    #   - cumcount() : position 0, 1, 2, ... au sein de ses ratings
    #   - transform("count") : nombre total de ratings de cet utilisateur
    #   - n_test = max(1, floor(total × 0.2)), borné à total − 1
    #
    # Un rating est dans le TEST si sa position ≥ (total − n_test),
    # c'est-à-dire s'il fait partie des derniers n_test ratings
    # (après mélange aléatoire).
    # ──────────────────────────────────────────────────────────────────

    df["_pos"] = df.groupby("user_id").cumcount()
    df["_total"] = df.groupby("user_id")["_pos"].transform("count")

    n_test_per_user = np.floor(df["_total"].values * TEST_RATIO).astype(int)
    n_test_per_user = np.maximum(n_test_per_user, 1)          # au moins 1 en test
    n_test_per_user = np.minimum(n_test_per_user, df["_total"].values - 1)  # au moins 1 en train

    df["_n_test"] = n_test_per_user
    df["is_test"] = df["_pos"] >= (df["_total"] - df["_n_test"])

    train_df = df[~df["is_test"]].drop(columns=["_rand", "_pos", "_total", "_n_test", "is_test"])
    test_df = df[df["is_test"]].drop(columns=["_rand", "_pos", "_total", "_n_test", "is_test"])

    t_split = time.perf_counter() - t0

    # ──────────────────────────────────────────────────────────────────
    # ÉTAPE 3 : Validation de la contrainte de stratification
    # ──────────────────────────────────────────────────────────────────
    # On vérifie que CHAQUE utilisateur a au moins 1 rating dans le
    # train ET dans le test. C'est la contrainte fondamentale du split.
    # ──────────────────────────────────────────────────────────────────

    users_in_train = set(train_df["user_id"].unique())
    users_in_test = set(test_df["user_id"].unique())
    users_only_train = users_in_train - users_in_test
    users_only_test = users_in_test - users_in_train
    users_in_both = users_in_train & users_in_test

    print(f"\n  ── Étape 1-2 : Split vectorisé (seed={SEED}) ──")
    print(f"     Temps de calcul : {t_split * 1000:.1f} ms")
    print(f"     Ratio demandé   : {TRAIN_RATIO:.0%} train / {TEST_RATIO:.0%} test")

    actual_train_ratio = len(train_df) / n_total
    actual_test_ratio = len(test_df) / n_total

    print(f"     Ratio effectif  : {actual_train_ratio:.2%} train / {actual_test_ratio:.2%} test")
    if abs(actual_train_ratio - TRAIN_RATIO) > 0.02:
        print(f"     → Écart de {abs(actual_train_ratio - TRAIN_RATIO):.1%} par rapport au ratio cible.")
        print(f"       Cela est dû aux utilisateurs avec peu de ratings (n=2 ou 3)")
        print(f"       pour lesquels on est forcé de donner 1 rating au test,")
        print(f"       ce qui « sur-représente » légèrement le test.")
    else:
        print(f"     → Ratio respecté à ±2% près.")

    print(f"\n  ── Étape 3 : Validation de la stratification ──")
    print(f"     Utilisateurs dans train ET test  : {len(users_in_both):,}")
    print(f"     Utilisateurs SEULEMENT dans train: {len(users_only_train):,}")
    print(f"     Utilisateurs SEULEMENT dans test : {len(users_only_test):,}")

    if len(users_only_train) == 0 and len(users_only_test) == 0:
        print(f"     ✓ Contrainte respectée : chaque utilisateur a au moins 1 rating")
        print(f"       dans chaque ensemble.")
    else:
        print(f"     ⚠ VIOLATION : {len(users_only_train) + len(users_only_test)} "
              f"utilisateur(s) absent(s) d'un ensemble !")

    # ──────────────────────────────────────────────────────────────────
    # ÉTAPE 4 : Construction des matrices CSR train et test
    # ──────────────────────────────────────────────────────────────────
    # On réutilise pd.factorize sur l'ensemble COMPLET (train + test)
    # pour garantir que les indices utilisateur/livre sont cohérents
    # entre les deux matrices. Sinon, l'utilisateur 42 dans R_train
    # pourrait correspondre à un utilisateur différent dans R_test.
    # ──────────────────────────────────────────────────────────────────

    t1 = time.perf_counter()

    all_users = pd.concat([train_df["user_id"], test_df["user_id"]])
    all_items = pd.concat([train_df["parent_asin"], test_df["parent_asin"]])
    user_codes_all, user_ids = pd.factorize(all_users, sort=False)
    item_codes_all, item_ids = pd.factorize(all_items, sort=False)

    n_u = len(user_ids)
    n_i = len(item_ids)
    n_train = len(train_df)
    n_test = len(test_df)

    # Les n_train premiers codes correspondent au train, le reste au test
    train_user_codes = user_codes_all[:n_train]
    train_item_codes = item_codes_all[:n_train]
    test_user_codes = user_codes_all[n_train:]
    test_item_codes = item_codes_all[n_train:]

    R_train = csr_matrix(
        (train_df["rating"].values.astype(np.float32),
         (train_user_codes, train_item_codes)),
        shape=(n_u, n_i),
        dtype=np.float32,
    )

    R_test = csr_matrix(
        (test_df["rating"].values.astype(np.float32),
         (test_user_codes, test_item_codes)),
        shape=(n_u, n_i),
        dtype=np.float32,
    )

    t_build = time.perf_counter() - t1

    print(f"\n  ── Étape 4 : Matrices CSR train/test ──")
    print(f"     Dimensions communes : {n_u:,} × {n_i:,}")
    print(f"     R_train : {R_train.nnz:,} entrées  ({R_train.nnz / (n_u * n_i) * 100:.4f}% dense)")
    print(f"     R_test  : {R_test.nnz:,} entrées  ({R_test.nnz / (n_u * n_i) * 100:.4f}% dense)")
    print(f"     R_train + R_test = {R_train.nnz + R_test.nnz:,} "
          f"(total original : {n_total:,}, "
          f"après dédup : {n_train + n_test:,})")
    print(f"     Temps de construction : {t_build * 1000:.1f} ms")

    mem_train = R_train.data.nbytes + R_train.indices.nbytes + R_train.indptr.nbytes
    mem_test = R_test.data.nbytes + R_test.indices.nbytes + R_test.indptr.nbytes

    print(f"     Mémoire R_train : {mem_train / 1024**2:.2f} Mo")
    print(f"     Mémoire R_test  : {mem_test / 1024**2:.2f} Mo")

    # ──────────────────────────────────────────────────────────────────
    # ÉTAPE 5 : Distribution du nombre de ratings test par utilisateur
    # ──────────────────────────────────────────────────────────────────
    # On affiche la distribution pour vérifier que le split est
    # raisonnablement équilibré et que les cas extrêmes (n=2) sont
    # bien gérés.
    # ──────────────────────────────────────────────────────────────────

    test_per_user = test_df.groupby("user_id").size()
    train_per_user = train_df.groupby("user_id").size()

    print(f"\n  ── Étape 5 : Distribution du split par utilisateur ──")
    print(f"     Ratings TRAIN par utilisateur :")
    print(f"       min = {train_per_user.min()}  |  "
          f"médiane = {train_per_user.median():.0f}  |  "
          f"moyenne = {train_per_user.mean():.1f}  |  "
          f"max = {train_per_user.max()}")
    print(f"     Ratings TEST par utilisateur :")
    print(f"       min = {test_per_user.min()}  |  "
          f"médiane = {test_per_user.median():.0f}  |  "
          f"moyenne = {test_per_user.mean():.1f}  |  "
          f"max = {test_per_user.max()}")

    n_users_1_test = (test_per_user == 1).sum()
    print(f"     Utilisateurs avec exactement 1 rating test : {n_users_1_test:,}"
          f" ({n_users_1_test / n_users * 100:.1f}%)")
    print(f"     → Ce sont les utilisateurs avec ≤ 5 ratings originaux,")
    print(f"       pour lesquels floor(n × 0.2) ≤ 1.")

    # ──────────────────────────────────────────────────────────────────
    # Résumé final
    # ──────────────────────────────────────────────────────────────────

    print(f"\n  {'─' * 66}")
    print(f"  RÉSUMÉ DU SPLIT — {path}")
    print(f"  {'─' * 66}")
    print(f"  {'':30s} {'TRAIN':>14s}   {'TEST':>14s}   {'TOTAL':>10s}")
    print(f"  {'─' * 66}")
    print(f"  {'Ratings':30s} {n_train:>14,}   {n_test:>14,}   {n_train + n_test:>10,}")
    print(f"  {'Proportion':30s} {actual_train_ratio:>13.2%}   {actual_test_ratio:>13.2%}   {'100.00%':>10s}")
    print(f"  {'Utilisateurs présents':30s} {train_df['user_id'].nunique():>14,}   "
          f"{test_df['user_id'].nunique():>14,}   {n_users:>10,}")
    print(f"  {'Livres présents':30s} {train_df['parent_asin'].nunique():>14,}   "
          f"{test_df['parent_asin'].nunique():>14,}   {n_items:>10,}")
    print(f"  {'Mémoire CSR':30s} {mem_train / 1024**2:>13.2f}Mo   "
          f"{mem_test / 1024**2:>13.2f}Mo")
    print(f"  {'─' * 66}")

    # Stockage pour cellules suivantes
    splits[path] = {
        "R_train": R_train,
        "R_test": R_test,
        "train_df": train_df,
        "test_df": test_df,
        "user_ids": user_ids,
        "item_ids": item_ids,
    }

print(f"\n✓ {len(splits)} split(s) train/test construit(s) et stocké(s) dans `splits`.")

#### B) Stockage des ensembles d'entraînement et de test

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# SAUVEGARDE PERSISTANTE DES SPLITS TRAIN / TEST
#
# On écrit sur disque tous les artefacts nécessaires pour reprendre
# le travail sans avoir à ré-exécuter les cellules précédentes :
#
#   1. train.parquet / test.parquet
#      → Les DataFrames bruts (toutes les colonnes : user_id, parent_asin,
#        rating, timestamp, text, etc.). Format Parquet pour la portabilité
#        et la vitesse de lecture.
#
#   2. R_train.npz / R_test.npz
#      → Les matrices CSR au format natif scipy. Rechargement instantané
#        avec scipy.sparse.load_npz(), sans recalcul de pd.factorize
#        ni reconstruction de la matrice.
#
#   3. user_ids.npy / item_ids.npy
#      → Les tables de correspondance indice ↔ identifiant original.
#        Indispensables pour interpréter les résultats des modèles :
#          user_ids[42] → "AHJKFDS83KD" (ID Amazon)
#          item_ids[7]  → "B00005N5PF"  (ASIN du livre)
#
#   4. metadata.json
#      → Paramètres du split et statistiques clés. Permet de vérifier
#        la provenance et la cohérence des données sans les recharger.
#
# ── Structure sur disque ──────────────────────────────────────────────────
#
# Pour chaque échantillon (ex: sample-cudf-claude/), on crée un
# sous-dossier « splits/ » :
#
#   sample-cudf-claude/
#   ├── sample_gpu_active_users.parquet   ← fichier source (déjà existant)
#   └── splits/
#       ├── train.parquet                 ← 80% des ratings
#       ├── test.parquet                  ← 20% des ratings
#       ├── R_train.npz                   ← matrice CSR d'entraînement
#       ├── R_test.npz                    ← matrice CSR de test
#       ├── user_ids.npy                  ← mapping indice → user_id
#       ├── item_ids.npy                  ← mapping indice → parent_asin
#       └── metadata.json                ← paramètres et statistiques
#
# ── Pourquoi ce choix de formats ? ────────────────────────────────────────
#
# Parquet (DataFrames) :
#   - Compression columnar → 3-5× plus compact que CSV
#   - Préserve les types (float, int, string) sans ambiguïté
#   - Lisible par Pandas, Polars, DuckDB, Spark, etc.
#
# NPZ (matrices CSR) :
#   - Format natif de scipy.sparse → load_npz() reconstitue la CSR
#     en une seule instruction, sans re-factoriser les identifiants
#   - Stocke data[], indices[], indptr[] + shape en un seul fichier
#
# NPY (mappings) :
#   - Format natif NumPy, le plus rapide pour charger un array 1D
#   - allow_pickle=True nécessaire pour les arrays de strings
#
# JSON (métadonnées) :
#   - Lisible par un humain, facilement parsable
#   - Documente la provenance exacte des splits
# ══════════════════════════════════════════════════════════════════════════

for path, data in splits.items():
    sample_dir = Path(path).parent
    split_dir = sample_dir / "splits"
    split_dir.mkdir(exist_ok=True)

    print(f"\n{'═' * 70}")
    print(f"  Sauvegarde : {split_dir}/")
    print(f"{'═' * 70}")

    t0 = time.perf_counter()

    R_train = data["R_train"]
    R_test = data["R_test"]
    train_df = data["train_df"]
    test_df = data["test_df"]
    user_ids = data["user_ids"]
    item_ids = data["item_ids"]

    # ── 1. DataFrames Parquet ─────────────────────────────────────────

    train_path = split_dir / "train.parquet"
    test_path = split_dir / "test.parquet"

    train_df.to_parquet(train_path, index=False)
    test_df.to_parquet(test_path, index=False)

    size_train_pq = os.path.getsize(train_path)
    size_test_pq = os.path.getsize(test_path)

    print(f"\n  ── DataFrames Parquet ──")
    print(f"     {train_path.name:20s} : {len(train_df):>10,} lignes  |  {size_train_pq / 1024**2:>7.2f} Mo")
    print(f"     {test_path.name:20s} : {len(test_df):>10,} lignes  |  {size_test_pq / 1024**2:>7.2f} Mo")

    # ── 2. Matrices CSR (scipy NPZ) ──────────────────────────────────

    r_train_path = split_dir / "R_train.npz"
    r_test_path = split_dir / "R_test.npz"

    save_npz(r_train_path, R_train)
    save_npz(r_test_path, R_test)

    size_train_npz = os.path.getsize(r_train_path)
    size_test_npz = os.path.getsize(r_test_path)

    print(f"\n  ── Matrices CSR (NPZ) ──")
    print(f"     {r_train_path.name:20s} : {R_train.shape[0]:,}×{R_train.shape[1]:,}  "
          f"|  nnz={R_train.nnz:>10,}  |  {size_train_npz / 1024**2:>7.2f} Mo")
    print(f"     {r_test_path.name:20s} : {R_test.shape[0]:,}×{R_test.shape[1]:,}  "
          f"|  nnz={R_test.nnz:>10,}  |  {size_test_npz / 1024**2:>7.2f} Mo")

    # ── 3. Mappings (NumPy NPY) ──────────────────────────────────────

    user_path = split_dir / "user_ids.npy"
    item_path = split_dir / "item_ids.npy"

    np.save(user_path, user_ids)
    np.save(item_path, item_ids)

    size_user = os.path.getsize(user_path)
    size_item = os.path.getsize(item_path)

    print(f"\n  ── Mappings (NPY) ──")
    print(f"     {user_path.name:20s} : {len(user_ids):>10,} entrées  |  {size_user / 1024**2:>7.2f} Mo")
    print(f"     {item_path.name:20s} : {len(item_ids):>10,} entrées  |  {size_item / 1024**2:>7.2f} Mo")

    # ── 4. Métadonnées JSON ──────────────────────────────────────────

    metadata = {
        "source_file": str(path),
        "split_seed": SEED,
        "train_ratio": TRAIN_RATIO,
        "test_ratio": TEST_RATIO,
        "n_users": int(len(user_ids)),
        "n_items": int(len(item_ids)),
        "train": {
            "n_ratings": int(R_train.nnz),
            "n_users": int(train_df["user_id"].nunique()),
            "n_items": int(train_df["parent_asin"].nunique()),
            "sparsity": float(1 - R_train.nnz / (len(user_ids) * len(item_ids))),
            "file_parquet": "train.parquet",
            "file_csr": "R_train.npz",
        },
        "test": {
            "n_ratings": int(R_test.nnz),
            "n_users": int(test_df["user_id"].nunique()),
            "n_items": int(test_df["parent_asin"].nunique()),
            "sparsity": float(1 - R_test.nnz / (len(user_ids) * len(item_ids))),
            "file_parquet": "test.parquet",
            "file_csr": "R_test.npz",
        },
        "mappings": {
            "file_user_ids": "user_ids.npy",
            "file_item_ids": "item_ids.npy",
        },
    }

    meta_path = split_dir / "metadata.json"
    with open(meta_path, "w") as f:
        json.dump(metadata, f, indent=2, ensure_ascii=False)

    print(f"\n  ── Métadonnées ──")
    print(f"     {meta_path.name:20s} : paramètres du split + statistiques")

    # ── Résumé ────────────────────────────────────────────────────────

    t_save = time.perf_counter() - t0
    total_size = size_train_pq + size_test_pq + size_train_npz + size_test_npz + size_user + size_item

    print(f"\n  {'─' * 66}")
    print(f"  RÉSUMÉ DE LA SAUVEGARDE — {split_dir}/")
    print(f"  {'─' * 66}")
    print(f"     Fichiers écrits  : 7")
    print(f"     Taille totale    : {total_size / 1024**2:.2f} Mo")
    print(f"     Temps d'écriture : {t_save * 1000:.1f} ms")
    print(f"  {'─' * 66}")
    print(f"     Contenu du dossier :")
    for f in sorted(split_dir.iterdir()):
        print(f"       {f.name:25s}  {os.path.getsize(f) / 1024**2:>7.2f} Mo")
    print(f"  {'─' * 66}")

print(f"\n✓ Tous les splits sont sauvegardés sur disque.")
print(f"\n  Pour recharger dans un autre notebook :")
print(f"  ┌─────────────────────────────────────────────────────────────────┐")
print(f"  │  from scipy.sparse import load_npz                            │")
print(f"  │  import numpy as np, pandas as pd, json                       │")
print(f"  │                                                               │")
print(f"  │  SPLIT = 'sample-cudf-claude/splits'                          │")
print(f"  │                                                               │")
print(f"  │  train_df = pd.read_parquet(f'{{SPLIT}}/train.parquet')        │")
print(f"  │  test_df  = pd.read_parquet(f'{{SPLIT}}/test.parquet')         │")
print(f"  │  R_train  = load_npz(f'{{SPLIT}}/R_train.npz')                │")
print(f"  │  R_test   = load_npz(f'{{SPLIT}}/R_test.npz')                 │")
print(f"  │  user_ids = np.load(f'{{SPLIT}}/user_ids.npy',                │")
print(f"  │                     allow_pickle=True)                        │")
print(f"  │  item_ids = np.load(f'{{SPLIT}}/item_ids.npy',                │")
print(f"  │                     allow_pickle=True)                        │")
print(f"  │  meta     = json.load(open(f'{{SPLIT}}/metadata.json'))        │")
print(f"  └─────────────────────────────────────────────────────────────────┘")

# 3.2 Tâche 1 - Mesures de similarité

## 3.2.1 Implémentation des similarités

- Cosinus
- Pearson
- Jaccard

##### Version optimisée pour reduire les temps de calculs

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# TÂCHE 1 : MESURES DE SIMILARITÉ ENTRE UTILISATEURS
# ══════════════════════════════════════════════════════════════════════════
#
# On implémente trois mesures classiques de similarité pour le filtrage
# collaboratif basé sur les utilisateurs (user-based CF) :
#
#   1. Similarité cosinus  — sim_cos(u,v)
#   2. Corrélation de Pearson — sim_pear(u,v)
#   3. Similarité de Jaccard — sim_jac(u,v)
#
# ── Pourquoi calculer la similarité entre utilisateurs ? ──────────────────
#
# Dans un système de recommandation par filtrage collaboratif, on prédit
# la note qu'un utilisateur u donnerait à un livre i en s'appuyant sur
# les notes d'utilisateurs SIMILAIRES à u qui ont déjà noté ce livre.
# La qualité des prédictions dépend directement de la qualité de la
# mesure de similarité choisie.
#
# ── Stratégie d'optimisation (conformément aux consignes) ─────────────────
#
# 1. NE PAS calculer la matrice complète |U|×|U| en dense :
#    Pour |U| = 10 714, cela ferait 10 714² ≈ 115M entrées × 4 octets
#    = 460 Mo. La plupart des valeurs seraient proches de 0 et inutiles.
#
# 2. Utiliser sklearn.metrics.pairwise.cosine_similarity sur matrices CSR :
#    sklearn exploite des routines BLAS optimisées (C/Fortran) pour le
#    produit matriciel creux, ce qui est ~100× plus rapide qu'une boucle
#    Python.
#
# 3. Calcul par BLOCS (batches) :
#    On découpe les utilisateurs en blocs de BATCH_SIZE. Pour chaque bloc,
#    on calcule la similarité avec TOUS les utilisateurs, on filtre
#    immédiatement par seuil, et on ne garde que les entrées significatives.
#    → La mémoire pic est BATCH_SIZE × |U| × 4 octets au lieu de |U|² × 4.
#
# 4. Stockage SPARSE (CSR) :
#    On ne conserve que les paires dont la similarité dépasse SIM_THRESHOLD.
#    Cela réduit la mémoire de stockage de ~100× par rapport à une matrice
#    dense, et accélère les recherches de voisins en aval.
#
# 5. Triangle supérieur seulement :
#    Toutes les mesures sont symétriques : sim(u,v) = sim(v,u).
#    On ne stocke que les paires (u,v) avec u < v pour diviser par 2
#    l'espace de stockage et le temps de filtrage.
#
# ── Pourquoi R_train et non R ? ──────────────────────────────────────────
#
# Les similarités sont calculées sur les données d'ENTRAÎNEMENT uniquement.
# Utiliser R (train + test) serait du « data leakage » : on intégrerait
# des informations de test dans le modèle, biaisant l'évaluation.
# ══════════════════════════════════════════════════════════════════════════

# ── Paramètres ────────────────────────────────────────────────────────────

# SIM_THRESHOLD : seuil minimal de similarité pour conserver une paire.
# Les paires avec sim < SIM_THRESHOLD sont considérées comme trop faibles
# pour contribuer utilement aux prédictions. Valeur typique : 0.1 à 0.3.
SIM_THRESHOLD = 0.1

# BATCH_SIZE : nombre d'utilisateurs traités simultanément.
# Un batch de 512 users sur ~44 000 items crée une sous-matrice dense de
# 512 × 10 714 ≈ 21 Mo — confortable en RAM.
BATCH_SIZE = 512

# MIN_COMMON_PEARSON : nombre minimal d'items co-notés pour que la
# corrélation de Pearson soit considérée comme fiable. Avec < 3 items
# en commun, la corrélation est statistiquement instable.
MIN_COMMON_PEARSON = 3

SPLIT_DIRS = sorted(glob.glob("sample-*/splits"))

print("╔══════════════════════════════════════════════════════════════════════╗")
print("║    TÂCHE 1 : MESURES DE SIMILARITÉ ENTRE UTILISATEURS              ║")
print("║    Cosine · Pearson · Jaccard                                       ║")
print("╚══════════════════════════════════════════════════════════════════════╝")
print(f"\n  Paramètres :")
print(f"     Seuil de similarité (threshold) : {SIM_THRESHOLD}")
print(f"     Taille de batch                 : {BATCH_SIZE} utilisateurs")
print(f"     Items communs min (Pearson)     : {MIN_COMMON_PEARSON}")
print(f"     Splits détectés                 : {len(SPLIT_DIRS)}")

similarities = {}

for split_dir in SPLIT_DIRS:
    split_path = Path(split_dir)
    sample_name = split_path.parent.name

    print(f"\n{'═' * 70}")
    print(f"  Échantillon : {sample_name}")
    print(f"{'═' * 70}")

    # ──────────────────────────────────────────────────────────────────
    # CHARGEMENT DES DONNÉES D'ENTRAÎNEMENT
    # ──────────────────────────────────────────────────────────────────
    # On recharge depuis le disque pour être indépendant des cellules
    # précédentes (le notebook peut être relancé partiellement).
    # ──────────────────────────────────────────────────────────────────

    R_train = load_npz(split_path / "R_train.npz")
    user_ids = np.load(split_path / "user_ids.npy", allow_pickle=True)
    item_ids = np.load(split_path / "item_ids.npy", allow_pickle=True)

    n_users, n_items = R_train.shape
    total_possible_pairs = n_users * (n_users - 1) // 2

    print(f"\n  ── Données chargées ──")
    print(f"     R_train              : {n_users:,} utilisateurs × {n_items:,} livres")
    print(f"     Ratings (nnz)        : {R_train.nnz:,}")
    print(f"     Sparsité             : {(1 - R_train.nnz / (n_users * n_items)) * 100:.4f}%")
    print(f"     Paires (u,v) totales : {total_possible_pairs:,}  (triangle supérieur)")
    mem_rtrain = (R_train.data.nbytes + R_train.indices.nbytes + R_train.indptr.nbytes)
    print(f"     Mémoire R_train      : {mem_rtrain / 1024**2:.2f} Mo")

    n_batches = (n_users + BATCH_SIZE - 1) // BATCH_SIZE
    print(f"     Nombre de batches    : {n_batches}  ({BATCH_SIZE} users/batch)")

    # ==================================================================
    # PRÉPARATIONS COMMUNES AUX TROIS MESURES
    # ==================================================================
    #
    # Plusieurs matrices dérivées de R_train sont réutilisées par les
    # trois mesures. On les calcule une seule fois pour éviter la
    # redondance.
    # ==================================================================

    # ── Matrice binaire : 1 si l'utilisateur a noté le livre, 0 sinon ──
    # Sert au calcul de Jaccard (|I_u ∩ I_v|) et Pearson (masque des
    # items en commun).
    R_binary = R_train.copy()
    R_binary.data = np.ones_like(R_binary.data, dtype=np.float32)

    # ── Nombre d'items notés par chaque utilisateur : |I_u| ──
    items_per_user = np.array(R_binary.sum(axis=1)).ravel()

    # ── Moyennes par utilisateur (pour Pearson) ──
    # r̄_u = Σ r_{u,i} / |I_u|, calculé uniquement sur les items notés.
    user_sums = np.array(R_train.sum(axis=1)).ravel()
    user_means = np.zeros(n_users, dtype=np.float64)
    active_mask = items_per_user > 0
    user_means[active_mask] = user_sums[active_mask] / items_per_user[active_mask]

    # ── Matrice centrée par la moyenne (pour Pearson) ──
    # On soustrait r̄_u de chaque entrée non nulle de la ligne u.
    # Les zéros (livres non notés) restent à 0 — ils ne participent
    # pas au calcul de similarité.
    #
    # JUSTIFICATION DE L'APPROXIMATION « Adjusted Cosine » :
    # La formule exacte de Pearson exige que les normes au dénominateur
    # soient calculées sur les items en commun I_{u,v} uniquement, ce
    # qui varie pour chaque paire et empêche toute vectorisation efficace.
    #
    # L'approximation « adjusted cosine » (Sarwar et al., 2001) calcule
    # les normes sur TOUS les items notés par chaque utilisateur :
    #   sim_pear(u,v) ≈ cosine(r_u − r̄_u, r_v − r̄_v)
    #
    # Cette approximation est :
    #   - Standard dans les bibliothèques (Surprise, LensKit, Spark ALS)
    #   - Quasi-identique à Pearson exact quand les profils sont denses
    #   - Compatible avec sklearn.cosine_similarity → calcul en secondes
    #   - Le calcul par batch respecte la consigne « batch computation
    #     for Pearson correlation »
    R_centered = R_train.copy().astype(np.float64)
    for i in range(n_users):
        s, e = R_centered.indptr[i], R_centered.indptr[i + 1]
        R_centered.data[s:e] -= user_means[i]
    R_centered = R_centered.astype(np.float32)

    print(f"\n  ── Préparations communes ──")
    print(f"     Matrice binaire R_binary    : construite ({R_binary.nnz:,} entrées)")
    print(f"     Items par utilisateur       : min={items_per_user.min():.0f}, "
          f"moy={items_per_user.mean():.1f}, max={items_per_user.max():.0f}")
    print(f"     Moyennes r̄_u               : min={user_means[active_mask].min():.2f}, "
          f"moy={user_means[active_mask].mean():.2f}, max={user_means[active_mask].max():.2f}")
    print(f"     Matrice centrée R_centered  : construite (pour Pearson approché)")
    print(f"     Mémoire totale préparations : "
          f"{(R_binary.data.nbytes + R_centered.data.nbytes) / 1024**2:.2f} Mo")

    # ── Fonction utilitaire : extraction sparse par batch ─────────────
    # Cette fonction est appelée par chacune des trois mesures.
    # Elle prend une sous-matrice dense (batch × n_users), met à zéro
    # la diagonale et le triangle inférieur, applique le seuil, et
    # retourne les triplets (row, col, val) pour le stockage CSR.

    def extract_sparse_upper(sim_block, batch_start, threshold):
        """
        Extrait les paires (u, v) avec u < v et sim > threshold
        à partir d'un bloc dense de similarités.

        Paramètres :
            sim_block  : array (batch_size, n_users) — similarités du batch
            batch_start: indice global du premier utilisateur du batch
            threshold  : seuil minimal de similarité

        Retourne :
            rows, cols, vals : listes de triplets pour construction CSR
        """
        batch_size_actual = sim_block.shape[0]
        rows, cols, vals = [], [], []

        for local_i in range(batch_size_actual):
            global_i = batch_start + local_i
            row = sim_block[local_i]

            # Ne garder que j > global_i (triangle supérieur) et sim > seuil
            candidates = np.where(
                (np.arange(n_users) > global_i) & (row > threshold)
            )[0]

            if len(candidates) > 0:
                rows.append(np.full(len(candidates), global_i, dtype=np.int32))
                cols.append(candidates.astype(np.int32))
                vals.append(row[candidates].astype(np.float32))

        return rows, cols, vals

    # ==================================================================
    # 1. SIMILARITÉ COSINUS
    # ==================================================================
    #
    #              r_u · r_v              Σ_{i ∈ I_{u,v}} r_{u,i} · r_{v,i}
    # sim_cos = ─────────── = ─────────────────────────────────────────────
    #            |r_u| |r_v|   √(Σ_{i ∈ I_u} r²_{u,i}) · √(Σ_{i ∈ I_v} r²_{v,i})
    #
    # Propriétés :
    #   - Valeurs dans [0, 1] (car tous les ratings sont ≥ 0)
    #   - 1 = vecteurs colinéaires (mêmes proportions de notes)
    #   - 0 = aucun livre en commun ou profils orthogonaux
    #   - Insensible à l'échelle : si u note systématiquement 2× plus
    #     haut que v, sim_cos = 1 quand même
    #
    # Avantage  : très rapide via sklearn (BLAS, matrices CSR natives)
    # Inconvénient : ne corrige pas le biais de notation (un utilisateur
    #   « généreux » qui note tout 5/5 paraît similaire à tout le monde)
    #
    # Implémentation :
    #   sklearn.metrics.pairwise.cosine_similarity(A, B) calcule :
    #     (A / |A|_rows) × (B / |B|_rows)ᵀ
    #   en une seule opération BLAS sur matrices CSR.
    #   On l'appelle par blocs : cosine_similarity(R[batch], R_train)
    #   produit une matrice dense (batch_size × n_users) qu'on filtre
    #   immédiatement.
    # ==================================================================

    print(f"\n  ══════════════════════════════════════════════════════════════")
    print(f"  1. SIMILARITÉ COSINUS")
    print(f"  ══════════════════════════════════════════════════════════════")
    print(f"     Formule : sim_cos(u,v) = (r_u · r_v) / (|r_u| × |r_v|)")
    print(f"     Méthode : sklearn.cosine_similarity sur matrice CSR")
    print(f"     Calcul par blocs de {BATCH_SIZE} utilisateurs")
    print(f"     Seuil : ne conserver que sim > {SIM_THRESHOLD}\n")

    t0 = time.perf_counter()
    cos_rows, cos_cols, cos_vals = [], [], []
    n_pairs_cos = 0

    for batch_idx in range(n_batches):
        start = batch_idx * BATCH_SIZE
        end = min(start + BATCH_SIZE, n_users)

        # sklearn gère nativement les matrices CSR : le produit matriciel
        # et la normalisation L2 sont effectués en C via BLAS, sans
        # conversion en dense.
        sim_block = cosine_similarity(R_train[start:end], R_train)

        # Mettre la diagonale à 0 (sim(u,u) = 1 triviale, inutile)
        for k in range(end - start):
            sim_block[k, start + k] = 0.0

        r, c, v = extract_sparse_upper(sim_block, start, SIM_THRESHOLD)
        cos_rows.extend(r)
        cos_cols.extend(c)
        cos_vals.extend(v)
        n_pairs_cos += sum(len(x) for x in r)

        if (batch_idx + 1) % max(1, n_batches // 5) == 0 or batch_idx == n_batches - 1:
            print(f"     Batch {batch_idx + 1:>4d}/{n_batches} "
                  f"(users {start:>6,}–{end - 1:>6,}) "
                  f"| paires : {n_pairs_cos:>10,} "
                  f"| {time.perf_counter() - t0:.1f}s")

    # Construction de la matrice CSR sparse (triangle supérieur)
    if cos_rows:
        cos_sim = csr_matrix(
            (np.concatenate(cos_vals), (np.concatenate(cos_rows), np.concatenate(cos_cols))),
            shape=(n_users, n_users), dtype=np.float32)
    else:
        cos_sim = csr_matrix((n_users, n_users), dtype=np.float32)

    t_cos = time.perf_counter() - t0
    pct_cos = n_pairs_cos / total_possible_pairs * 100 if total_possible_pairs > 0 else 0
    mem_cos = cos_sim.data.nbytes + cos_sim.indices.nbytes + cos_sim.indptr.nbytes

    cos_data = cos_sim.data
    print(f"\n  ── Résultat Cosine Similarity ──")
    print(f"     Paires totales possibles    : {total_possible_pairs:,}")
    print(f"     Paires retenues (sim > {SIM_THRESHOLD}) : {n_pairs_cos:,} ({pct_cos:.2f}%)")
    if len(cos_data) > 0:
        print(f"     Similarité moyenne          : {cos_data.mean():.4f}")
        print(f"     Similarité médiane          : {np.median(cos_data):.4f}")
        print(f"     Similarité min (retenue)    : {cos_data.min():.4f}")
        print(f"     Similarité max              : {cos_data.max():.4f}")
    print(f"     Mémoire (CSR, tri. sup.)    : {mem_cos / 1024**2:.2f} Mo")
    print(f"     Temps de calcul             : {t_cos:.1f}s")
    print(f"     → {pct_cos:.2f}% des paires retenues = stockage {100/max(pct_cos,0.01):.0f}× plus")
    print(f"       compact qu'une matrice dense.")

    # ==================================================================
    # 2. CORRÉLATION DE PEARSON (via Adjusted Cosine)
    # ==================================================================
    #
    #          Σ_{i ∈ I_{u,v}} (r_{u,i} − r̄_u)(r_{v,i} − r̄_v)
    # sim_pear = ──────────────────────────────────────────────────────
    #            √[Σ (r_{u,i}−r̄_u)²] × √[Σ (r_{v,i}−r̄_v)²]
    #
    # où r̄_u = moyenne des notes de u (sur I_u uniquement)
    #
    # DIFFÉRENCE CLÉ AVEC LE COSINUS :
    #   Le cosinus compare les vecteurs BRUTS de ratings.
    #   Pearson compare les vecteurs CENTRÉS (rating − moyenne utilisateur).
    #   Pearson mesure si u et v DÉVIENT de la même manière par rapport
    #   à leurs propres moyennes respectives.
    #
    # Exemple illustratif :
    #   u note : [5, 4, 5, 3] → r̄_u = 4.25 → centrés : [+0.75, −0.25, +0.75, −1.25]
    #   v note : [3, 2, 3, 1] → r̄_v = 2.25 → centrés : [+0.75, −0.25, +0.75, −1.25]
    #   → Pearson = 1.0 (profils parfaitement corrélés)
    #   → Cosinus < 1.0 (les vecteurs bruts ne sont pas colinéaires)
    #   → Pearson corrige le biais : un « sévère » et un « généreux »
    #     peuvent être très corrélés s'ils aiment les mêmes livres.
    #
    # Propriétés :
    #   - Valeurs dans [-1, 1]
    #   - +1 = corrélation parfaite positive (mêmes préférences relatives)
    #   - -1 = corrélation parfaite négative (goûts opposés)
    #   -  0 = aucune corrélation linéaire
    #
    # ── Implémentation : Adjusted Cosine (Sarwar et al., 2001) ────────
    #
    # La formule exacte de Pearson exige que les normes au dénominateur
    # soient calculées sur I_{u,v} (items co-notés), qui varie pour
    # chaque paire → impossible à vectoriser efficacement.
    #
    # L'Adjusted Cosine est l'approximation standard dans la littérature :
    #   sim_pear(u,v) ≈ cosine(r_u − r̄_u,  r_v − r̄_v)
    #
    # On centre chaque ligne de R_train par la moyenne de l'utilisateur,
    # puis on applique cosine_similarity sur la matrice centrée.
    # La seule différence : les normes sont calculées sur I_u et I_v
    # (tous les items notés) plutôt que sur I_{u,v} seul.
    # En pratique, cette différence est négligeable et cette approche
    # est utilisée par Surprise, LensKit, et Spark ALS.
    #
    # Avantage : même vitesse que le cosinus (sklearn BLAS par batch).
    # On applique un seuil sur |sim| > threshold (les corrélations
    # négatives fortes sont aussi informatives).
    # ==================================================================

    print(f"\n  ══════════════════════════════════════════════════════════════")
    print(f"  2. CORRÉLATION DE PEARSON (Adjusted Cosine)")
    print(f"  ══════════════════════════════════════════════════════════════")
    print(f"     Formule : sim_pear(u,v) ≈ cosine(r_u − r̄_u, r_v − r̄_v)")
    print(f"     Méthode : centrage par r̄_u + sklearn.cosine_similarity")
    print(f"     Calcul par blocs de {BATCH_SIZE} utilisateurs")
    print(f"     Seuil : ne conserver que |sim| > {SIM_THRESHOLD}")
    print(f"     Filtre additionnel : ≥ {MIN_COMMON_PEARSON} items en commun\n")

    t0 = time.perf_counter()
    pear_rows, pear_cols, pear_vals = [], [], []
    n_pairs_pear = 0

    for batch_idx in range(n_batches):
        start = batch_idx * BATCH_SIZE
        end = min(start + BATCH_SIZE, n_users)

        # cosine_similarity sur la matrice CENTRÉE = adjusted cosine ≈ Pearson
        sim_block = cosine_similarity(R_centered[start:end], R_centered)

        # ── Filtre par nombre d'items en commun ──
        # On calcule |I_u ∩ I_v| par produit de matrices binaires.
        # Les paires avec < MIN_COMMON_PEARSON items en commun sont
        # mises à 0 : la corrélation n'est pas fiable statistiquement.
        common_block = (R_binary[start:end] @ R_binary.T).toarray()
        sim_block[common_block < MIN_COMMON_PEARSON] = 0.0

        # Diagonale à 0
        for k in range(end - start):
            sim_block[k, start + k] = 0.0

        # Pour Pearson, on retient les valeurs dont |sim| > seuil
        # car les corrélations négatives fortes sont aussi significatives
        # (elles indiquent des goûts opposés — utiles pour anti-recommandations).
        batch_actual = end - start
        for local_i in range(batch_actual):
            global_i = start + local_i
            row = sim_block[local_i]
            candidates = np.where(
                (np.arange(n_users) > global_i) & (np.abs(row) > SIM_THRESHOLD)
            )[0]
            if len(candidates) > 0:
                pear_rows.append(np.full(len(candidates), global_i, dtype=np.int32))
                pear_cols.append(candidates.astype(np.int32))
                pear_vals.append(row[candidates].astype(np.float32))
                n_pairs_pear += len(candidates)

        if (batch_idx + 1) % max(1, n_batches // 5) == 0 or batch_idx == n_batches - 1:
            print(f"     Batch {batch_idx + 1:>4d}/{n_batches} "
                  f"(users {start:>6,}–{end - 1:>6,}) "
                  f"| paires : {n_pairs_pear:>10,} "
                  f"| {time.perf_counter() - t0:.1f}s")

    if pear_rows:
        pear_sim = csr_matrix(
            (np.concatenate(pear_vals), (np.concatenate(pear_rows), np.concatenate(pear_cols))),
            shape=(n_users, n_users), dtype=np.float32)
    else:
        pear_sim = csr_matrix((n_users, n_users), dtype=np.float32)

    t_pear = time.perf_counter() - t0
    pct_pear = n_pairs_pear / total_possible_pairs * 100 if total_possible_pairs > 0 else 0
    mem_pear = pear_sim.data.nbytes + pear_sim.indices.nbytes + pear_sim.indptr.nbytes

    pear_data = pear_sim.data
    n_negative = (pear_data < 0).sum() if len(pear_data) > 0 else 0

    print(f"\n  ── Résultat Pearson Correlation ──")
    print(f"     Paires retenues (|sim| > {SIM_THRESHOLD}): {n_pairs_pear:,} ({pct_pear:.2f}%)")
    if len(pear_data) > 0:
        print(f"     Corrélation moyenne          : {pear_data.mean():.4f}")
        print(f"     Corrélation médiane          : {np.median(pear_data):.4f}")
        print(f"     Corrélation min (retenue)    : {pear_data.min():.4f}")
        print(f"     Corrélation max              : {pear_data.max():.4f}")
        print(f"     Paires à corrélation < 0     : {n_negative:,} "
              f"({n_negative / len(pear_data) * 100:.1f}%)")
    print(f"     Mémoire (CSR, tri. sup.)     : {mem_pear / 1024**2:.2f} Mo")
    print(f"     Temps de calcul              : {t_pear:.1f}s")
    print(f"     → Pearson corrige le biais de notation : deux utilisateurs")
    print(f"       « sévères » ou « généreux » peuvent être très corrélés")
    print(f"       s'ils aiment les mêmes livres, même avec des notes")
    print(f"       absolues très différentes.")

    # ==================================================================
    # 3. SIMILARITÉ DE JACCARD
    # ==================================================================
    #
    #               |I_u ∩ I_v|
    # sim_jac = ─────────────────
    #            |I_u ∪ I_v|
    #
    # où I_u = ensemble des livres notés par u
    #    I_v = ensemble des livres notés par v
    #
    # DIFFÉRENCE FONDAMENTALE avec les deux mesures précédentes :
    #   - Cosinus et Pearson utilisent les VALEURS des ratings
    #   - Jaccard utilise uniquement la PRÉSENCE/ABSENCE de rating
    #   - Jaccard ignore complètement si u a noté 5/5 et v a noté 1/5
    #
    # Propriétés :
    #   - Valeurs dans [0, 1]
    #   - 1 = u et v ont noté exactement les mêmes livres
    #   - 0 = aucun livre en commun
    #   - Mesure la « proximité des centres d'intérêt » sans tenir
    #     compte de l'appréciation
    #
    # Utilité dans un système de recommandation :
    #   - Filtre préalable : si Jaccard(u,v) ≈ 0, inutile de calculer
    #     Pearson ou Cosinus (pas de signal exploitable)
    #   - Pondération hybride : sim_final = α·Pearson + (1−α)·Jaccard
    #   - Détection de communautés (utilisateurs lisant le même « genre »)
    #
    # Implémentation vectorisée :
    #   |I_u ∩ I_v| = R_binary[u] · R_binary[v]  (produit scalaire 0/1)
    #   |I_u ∪ I_v| = |I_u| + |I_v| − |I_u ∩ I_v|
    #
    # Le produit R_binary × R_binary.T donne directement la matrice
    # des intersections pour tout un batch. On calcule ensuite l'union
    # par broadcasting numpy — aucune boucle Python sur les paires.
    # ==================================================================

    print(f"\n  ══════════════════════════════════════════════════════════════")
    print(f"  3. SIMILARITÉ DE JACCARD")
    print(f"  ══════════════════════════════════════════════════════════════")
    print(f"     Formule : sim_jac(u,v) = |I_u ∩ I_v| / |I_u ∪ I_v|")
    print(f"     Méthode : produit R_binary × R_binary.T (intersection)")
    print(f"               union = |I_u| + |I_v| − intersection")
    print(f"     Calcul par blocs de {BATCH_SIZE} utilisateurs")
    print(f"     Seuil : ne conserver que sim > {SIM_THRESHOLD}\n")

    t0 = time.perf_counter()
    jac_rows, jac_cols, jac_vals = [], [], []
    n_pairs_jac = 0

    for batch_idx in range(n_batches):
        start = batch_idx * BATCH_SIZE
        end = min(start + BATCH_SIZE, n_users)
        batch_actual = end - start

        # |I_u ∩ I_v| : produit de matrices binaires creuses
        # R_binary est CSR, le produit CSR × CSC est optimisé par scipy.
        intersection = (R_binary[start:end] @ R_binary.T).toarray().astype(np.float64)

        # |I_u ∪ I_v| = |I_u| + |I_v| − |I_u ∩ I_v|
        # Broadcasting : items_batch est (batch,1), items_all est (1,n_users)
        items_batch = items_per_user[start:end].reshape(-1, 1)  # (batch, 1)
        items_all = items_per_user.reshape(1, -1)                # (1, n_users)
        union = items_batch + items_all - intersection

        # Jaccard = intersection / union (0 si union = 0)
        jaccard_block = np.divide(
            intersection, union,
            where=(union > 0),
            out=np.zeros_like(intersection)
        ).astype(np.float32)

        # Diagonale à 0
        for k in range(batch_actual):
            jaccard_block[k, start + k] = 0.0

        # Extraction sparse (triangle supérieur, seuil)
        r, c, v = extract_sparse_upper(jaccard_block, start, SIM_THRESHOLD)
        jac_rows.extend(r)
        jac_cols.extend(c)
        jac_vals.extend(v)
        n_pairs_jac += sum(len(x) for x in r)

        if (batch_idx + 1) % max(1, n_batches // 5) == 0 or batch_idx == n_batches - 1:
            print(f"     Batch {batch_idx + 1:>4d}/{n_batches} "
                  f"(users {start:>6,}–{end - 1:>6,}) "
                  f"| paires : {n_pairs_jac:>10,} "
                  f"| {time.perf_counter() - t0:.1f}s")

    if jac_rows:
        jac_sim = csr_matrix(
            (np.concatenate(jac_vals), (np.concatenate(jac_rows), np.concatenate(jac_cols))),
            shape=(n_users, n_users), dtype=np.float32)
    else:
        jac_sim = csr_matrix((n_users, n_users), dtype=np.float32)

    t_jac = time.perf_counter() - t0
    pct_jac = n_pairs_jac / total_possible_pairs * 100 if total_possible_pairs > 0 else 0
    mem_jac = jac_sim.data.nbytes + jac_sim.indices.nbytes + jac_sim.indptr.nbytes

    jac_data = jac_sim.data
    print(f"\n  ── Résultat Jaccard Similarity ──")
    print(f"     Paires retenues (sim > {SIM_THRESHOLD}) : {n_pairs_jac:,} ({pct_jac:.2f}%)")
    if len(jac_data) > 0:
        print(f"     Similarité moyenne           : {jac_data.mean():.4f}")
        print(f"     Similarité médiane           : {jac_data.median() if hasattr(jac_data, 'median') else np.median(jac_data):.4f}")
        print(f"     Similarité min (retenue)     : {jac_data.min():.4f}")
        print(f"     Similarité max               : {jac_data.max():.4f}")
    print(f"     Mémoire (CSR, tri. sup.)     : {mem_jac / 1024**2:.2f} Mo")
    print(f"     Temps de calcul              : {t_jac:.1f}s")
    print(f"     → Jaccard est une mesure purement binaire : elle capture le")
    print(f"       recouvrement des « bibliothèques » de deux utilisateurs,")
    print(f"       indépendamment de leurs appréciations.")

    # ==================================================================
    # RÉSUMÉ COMPARATIF DES TROIS MESURES
    # ==================================================================

    print(f"\n  {'═' * 68}")
    print(f"  RÉSUMÉ COMPARATIF — {sample_name}")
    print(f"  {'═' * 68}")
    print(f"  {'Mesure':20s} {'Paires':>12s} {'%':>8s} {'Moy':>8s} "
          f"{'Méd':>8s} {'Max':>8s} {'Temps':>8s} {'Mém':>8s}")
    print(f"  {'─' * 68}")

    for label, n_p, pct, data, t_sec, mem in [
        ("Cosinus", n_pairs_cos, pct_cos, cos_data, t_cos, mem_cos),
        ("Pearson", n_pairs_pear, pct_pear, pear_data, t_pear, mem_pear),
        ("Jaccard", n_pairs_jac, pct_jac, jac_data, t_jac, mem_jac),
    ]:
        if len(data) > 0:
            print(f"  {label:20s} {n_p:>12,} {pct:>7.2f}% "
                  f"{data.mean():>8.4f} {np.median(data):>8.4f} "
                  f"{data.max():>8.4f} {t_sec:>7.1f}s {mem / 1024**2:>7.2f}M")
        else:
            print(f"  {label:20s} {n_p:>12,} {pct:>7.2f}% "
                  f"{'—':>8s} {'—':>8s} {'—':>8s} {t_sec:>7.1f}s {mem / 1024**2:>7.2f}M")

    print(f"  {'─' * 68}")
    t_total = t_cos + t_pear + t_jac
    mem_total = mem_cos + mem_pear + mem_jac
    print(f"  {'TOTAL':20s} {'':>12s} {'':>8s} {'':>8s} {'':>8s} "
          f"{'':>8s} {t_total:>7.1f}s {mem_total / 1024**2:>7.2f}M")
    print(f"  {'─' * 68}")

    print(f"\n  ── Interprétation ──")
    print(f"     • Cosinus : rapide grâce à sklearn/BLAS sur CSR. Ne corrige pas")
    print(f"       le biais de notation — un utilisateur « généreux » (tout à 5★)")
    print(f"       semble similaire à tous les autres.")
    print(f"     • Pearson : même vitesse (adjusted cosine sur matrice centrée),")
    print(f"       mais corrige le biais en soustrayant r̄_u à chaque note.")
    if n_negative > 0:
        print(f"       {n_negative:,} paires ont une corrélation négative (goûts opposés).")
    print(f"     • Jaccard : mesure purement binaire (présence/absence de note).")
    print(f"       Utile comme filtre ou en combinaison avec Pearson/Cosinus.")
    print(f"     → Pour la prédiction, Pearson est généralement le meilleur choix")
    print(f"       car il capture les préférences RELATIVES des utilisateurs.")

    # ── Sauvegarde des matrices de similarité ─────────────────────────
    # On écrit chaque matrice au format scipy NPZ (natif CSR) et un
    # fichier metadata.json avec les paramètres et statistiques.
    # Structure :
    #   sample-xxx/splits/similarities/
    #   ├── sim_cosine.npz
    #   ├── sim_pearson.npz
    #   ├── sim_jaccard.npz
    #   └── metadata.json

    sim_dir = split_path / "similarities"
    sim_dir.mkdir(exist_ok=True)

    save_npz(sim_dir / "sim_cosine.npz", cos_sim)
    save_npz(sim_dir / "sim_pearson.npz", pear_sim)
    save_npz(sim_dir / "sim_jaccard.npz", jac_sim)

    sim_meta = {
        "sample": sample_name,
        "n_users": int(n_users),
        "n_items": int(n_items),
        "threshold": SIM_THRESHOLD,
        "batch_size": BATCH_SIZE,
        "min_common_pearson": MIN_COMMON_PEARSON,
        "storage": "upper_triangular_csr",
        "cosine": {
            "n_pairs": int(n_pairs_cos),
            "pct_retained": round(pct_cos, 4),
            "mean": round(float(cos_data.mean()), 4) if len(cos_data) > 0 else None,
            "max": round(float(cos_data.max()), 4) if len(cos_data) > 0 else None,
            "time_seconds": round(t_cos, 1),
        },
        "pearson": {
            "n_pairs": int(n_pairs_pear),
            "pct_retained": round(pct_pear, 4),
            "mean": round(float(pear_data.mean()), 4) if len(pear_data) > 0 else None,
            "max": round(float(pear_data.max()), 4) if len(pear_data) > 0 else None,
            "n_negative": int(n_negative),
            "time_seconds": round(t_pear, 1),
        },
        "jaccard": {
            "n_pairs": int(n_pairs_jac),
            "pct_retained": round(pct_jac, 4),
            "mean": round(float(jac_data.mean()), 4) if len(jac_data) > 0 else None,
            "max": round(float(jac_data.max()), 4) if len(jac_data) > 0 else None,
            "time_seconds": round(t_jac, 1),
        },
    }

    with open(sim_dir / "metadata.json", "w") as f:
        json.dump(sim_meta, f, indent=2, ensure_ascii=False)

    print(f"\n  ✓ Matrices de similarité sauvegardées dans {sim_dir}/")
    for fname in ["sim_cosine.npz", "sim_pearson.npz", "sim_jaccard.npz", "metadata.json"]:
        fpath = sim_dir / fname
        if fpath.exists():
            print(f"     {fname:25s} : {os.path.getsize(fpath) / 1024**2:>7.2f} Mo")

    # Stockage en mémoire pour les cellules suivantes
    similarities[sample_name] = {
        "cosine": cos_sim,
        "pearson": pear_sim,
        "jaccard": jac_sim,
        "user_ids": user_ids,
        "item_ids": item_ids,
        "R_train": R_train,
    }

print(f"\n╔══════════════════════════════════════════════════════════════════════╗")
print(f"║  ✓ Toutes les similarités ont été calculées et sauvegardées.        ║")
print(f"╚══════════════════════════════════════════════════════════════════════╝")
print(f"\n  Pour recharger dans un autre notebook :")
print(f"  ┌─────────────────────────────────────────────────────────────────┐")
print(f"  │  from scipy.sparse import load_npz                            │")
print(f"  │  SIM = 'sample-cudf-claude/splits/similarities'               │")
print(f"  │  cos_sim  = load_npz(f'{{SIM}}/sim_cosine.npz')               │")
print(f"  │  pear_sim = load_npz(f'{{SIM}}/sim_pearson.npz')              │")
print(f"  │  jac_sim  = load_npz(f'{{SIM}}/sim_jaccard.npz')              │")
print(f"  └─────────────────────────────────────────────────────────────────┘")

## 3.2.2 Analyse comparative

### 1) Echantillon d'utilisateurs

Select 5 users with different activity profiles:
- 1 very active user (> 100 reviews)
- 2 moderately active users (30–50 reviews)
- 2 low-activity users (10–20 reviews)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# 1) ÉCHANTILLONNAGE D'UTILISATEURS (User Sampling)
#
# Pour analyser le comportement des mesures de similarité, on sélectionne
# 5 utilisateurs représentatifs de différents profils d'activité :
#
#   - 1 très actif    : > 100 reviews (power user, profil riche)
#   - 2 modérément actifs : 30–50 reviews (profil typique)
#   - 2 peu actifs    : 10–20 reviews (profil sparse, cas difficile)
#
# Ce panel permet d'évaluer comment chaque mesure de similarité se
# comporte selon la densité du profil utilisateur. Un utilisateur avec
# 100+ reviews partage potentiellement beaucoup de livres avec d'autres,
# tandis qu'un utilisateur avec 10 reviews aura peu de co-notations,
# rendant les similarités moins fiables.
#
# REPRODUCTIBILITÉ : on fixe un seed pour que la sélection soit
# identique à chaque exécution. Les IDs sont sauvegardés sur disque.
# ══════════════════════════════════════════════════════════════════════════

SEED = 42
SPLIT_DIRS = sorted(glob.glob("sample-*/splits"))

sampled_users = {}

for split_dir in SPLIT_DIRS:
    split_path = Path(split_dir)
    sample_name = split_path.parent.name

    print(f"\n{'═' * 70}")
    print(f"  Échantillonnage d'utilisateurs : {sample_name}")
    print(f"{'═' * 70}")

    # Chargement de la matrice d'entraînement
    R_train = load_npz(split_path / "R_train.npz")
    user_ids = np.load(split_path / "user_ids.npy", allow_pickle=True)
    n_users, n_items = R_train.shape

    # ──────────────────────────────────────────────────────────────────
    # Étape 1 : Calcul du nombre de reviews par utilisateur
    # ──────────────────────────────────────────────────────────────────
    # np.diff(R_train.indptr) donne le nombre d'entrées non nulles par
    # ligne dans la matrice CSR — c'est exactement le nombre de livres
    # notés par chaque utilisateur dans le jeu d'entraînement.
    # ──────────────────────────────────────────────────────────────────

    reviews_per_user = np.diff(R_train.indptr)

    print(f"\n  ── Étape 1 : Distribution de l'activité ──")
    print(f"     Utilisateurs totaux : {n_users:,}")
    print(f"     Reviews par user    : min={reviews_per_user.min()}, "
          f"médiane={np.median(reviews_per_user):.0f}, "
          f"moy={reviews_per_user.mean():.1f}, "
          f"max={reviews_per_user.max()}")

    # Distribution par tranche d'activité
    n_high = (reviews_per_user > 100).sum()
    n_moderate = ((reviews_per_user >= 30) & (reviews_per_user <= 50)).sum()
    n_low = ((reviews_per_user >= 10) & (reviews_per_user <= 20)).sum()

    print(f"     > 100 reviews (très actifs)  : {n_high:,} utilisateurs")
    print(f"     30–50 reviews (modérés)      : {n_moderate:,} utilisateurs")
    print(f"     10–20 reviews (peu actifs)   : {n_low:,} utilisateurs")

    # ──────────────────────────────────────────────────────────────────
    # Étape 2 : Sélection des 5 utilisateurs
    # ──────────────────────────────────────────────────────────────────
    # On utilise np.random.RandomState(SEED) pour la reproductibilité.
    # Pour chaque catégorie, on identifie tous les candidats, puis on
    # en sélectionne le nombre requis aléatoirement.
    # ──────────────────────────────────────────────────────────────────

    rng = np.random.RandomState(SEED)

    # 1 très actif (> 100 reviews)
    idx_high = np.where(reviews_per_user > 100)[0]
    selected_high = rng.choice(idx_high, size=1, replace=False)

    # 2 modérément actifs (30–50 reviews)
    idx_moderate = np.where((reviews_per_user >= 30) & (reviews_per_user <= 50))[0]
    selected_moderate = rng.choice(idx_moderate, size=2, replace=False)

    # 2 peu actifs (10–20 reviews)
    idx_low = np.where((reviews_per_user >= 10) & (reviews_per_user <= 20))[0]
    selected_low = rng.choice(idx_low, size=2, replace=False)

    selected_indices = np.concatenate([selected_high, selected_moderate, selected_low])
    selected_original_ids = user_ids[selected_indices]

    print(f"\n  ── Étape 2 : Utilisateurs sélectionnés (seed={SEED}) ──")
    print(f"     {'Catégorie':20s} {'Indice':>8s} {'Reviews':>8s} {'User ID':>30s}")
    print(f"     {'─' * 66}")

    categories = (["Très actif (>100)"] * 1 +
                  ["Modéré (30–50)"] * 2 +
                  ["Peu actif (10–20)"] * 2)

    for cat, idx in zip(categories, selected_indices):
        n_rev = reviews_per_user[idx]
        uid = user_ids[idx]
        print(f"     {cat:20s} {idx:>8d} {n_rev:>8d} {str(uid):>30s}")

    # ──────────────────────────────────────────────────────────────────
    # Étape 3 : Sauvegarde pour reproductibilité
    # ──────────────────────────────────────────────────────────────────

    sampling_info = {
        "seed": SEED,
        "sample": sample_name,
        "users": []
    }

    for cat, idx in zip(categories, selected_indices):
        sampling_info["users"].append({
            "category": cat,
            "matrix_index": int(idx),
            "n_reviews": int(reviews_per_user[idx]),
            "user_id": str(user_ids[idx]),
        })

    sim_dir = split_path / "similarities"
    sim_dir.mkdir(exist_ok=True)
    with open(sim_dir / "sampled_users.json", "w") as f:
        json.dump(sampling_info, f, indent=2, ensure_ascii=False)

    print(f"\n  ✓ Sélection sauvegardée : {sim_dir / 'sampled_users.json'}")

    sampled_users[sample_name] = {
        "indices": selected_indices,
        "ids": selected_original_ids,
        "categories": categories,
        "reviews": reviews_per_user[selected_indices],
        "R_train": R_train,
        "user_ids": user_ids,
    }

print(f"\n✓ Utilisateurs échantillonnés pour {len(sampled_users)} échantillon(s).")

### 2) Identification des voisins

For each sampled user, identify their 10 nearest neighbors under each similarity measure.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# 2) IDENTIFICATION DES VOISINS (Neighbor Identification)
#
# Pour chaque utilisateur échantillonné, on identifie ses 10 plus proches
# voisins selon chacune des trois mesures de similarité.
#
# ── Rappel sur le stockage ────────────────────────────────────────────────
#
# Les matrices de similarité sont stockées en TRIANGLE SUPÉRIEUR :
#   sim(u, v) est stocké à la position (u, v) si u < v.
# Pour récupérer tous les voisins d'un utilisateur u, il faut lire :
#   - La LIGNE u de la matrice  : donne sim(u, v) pour v > u
#   - La COLONNE u de la matrice : donne sim(v, u) pour v < u
# En CSR, lire une colonne nécessite de passer en CSC (transposée).
#
# ── Gestion des cas limites ───────────────────────────────────────────────
#
# - Si un utilisateur a moins de 10 voisins au-dessus du seuil, on
#   retourne tous ceux disponibles (sans remplissage artificiel).
# - Les NaN/Inf sont filtrés avant le tri.
# - Pour Pearson, on trie par valeur absolue (les anti-corrélations
#   fortes sont aussi des voisins « informatifs »), mais on affiche
#   le signe.
# ══════════════════════════════════════════════════════════════════════════

TOP_K = 10

neighbors_results = {}

for sample_name, sdata in sampled_users.items():
    split_path = Path(f"{sample_name}/splits")
    sim_dir = split_path / "similarities"

    print(f"\n{'═' * 70}")
    print(f"  Identification des voisins : {sample_name}")
    print(f"{'═' * 70}")

    # Chargement des matrices de similarité
    sim_matrices = {
        "Cosinus": load_npz(sim_dir / "sim_cosine.npz"),
        "Pearson": load_npz(sim_dir / "sim_pearson.npz"),
        "Jaccard": load_npz(sim_dir / "sim_jaccard.npz"),
    }

    R_train = sdata["R_train"]
    user_ids = sdata["user_ids"]
    n_users = R_train.shape[0]
    reviews_per_user = np.diff(R_train.indptr)

    # Pré-calculer les transposées CSC pour accéder aux colonnes
    sim_matrices_T = {name: M.T.tocsr() for name, M in sim_matrices.items()}

    sample_neighbors = {}

    for user_rank, (cat, u_idx) in enumerate(zip(sdata["categories"], sdata["indices"])):
        uid = user_ids[u_idx]
        n_rev = reviews_per_user[u_idx]

        print(f"\n  ── Utilisateur {user_rank + 1}/5 : indice={u_idx}, "
              f"reviews={n_rev}, catégorie=\"{cat}\" ──")
        print(f"     User ID : {uid}")

        user_neighbors = {}

        for measure_name, sim_mat in sim_matrices.items():
            sim_mat_T = sim_matrices_T[measure_name]

            # ──────────────────────────────────────────────────────
            # Récupération de toutes les similarités de l'utilisateur u_idx
            # ──────────────────────────────────────────────────────
            # Ligne u_idx : voisins v > u_idx (triangle supérieur)
            row = sim_mat.getrow(u_idx).toarray().ravel()
            # Colonne u_idx : voisins v < u_idx (triangle inférieur = transposée)
            col = sim_mat_T.getrow(u_idx).toarray().ravel()

            # Fusion : pour chaque voisin v, la similarité est dans row OU col
            all_sims = row + col  # Les deux ne se chevauchent jamais (tri. sup.)

            # Filtrer les NaN et Inf
            valid_mask = np.isfinite(all_sims) & (all_sims != 0)
            valid_indices = np.where(valid_mask)[0]
            valid_values = all_sims[valid_indices]

            # Pour Pearson, trier par valeur absolue (anti-corrélations comptent)
            if measure_name == "Pearson":
                sort_key = np.abs(valid_values)
            else:
                sort_key = valid_values

            # Top-K : indices des K plus grandes similarités
            if len(valid_values) >= TOP_K:
                top_k_local = np.argpartition(sort_key, -TOP_K)[-TOP_K:]
                top_k_local = top_k_local[np.argsort(sort_key[top_k_local])[::-1]]
            else:
                top_k_local = np.argsort(sort_key)[::-1]

            top_k_global = valid_indices[top_k_local]
            top_k_sims = all_sims[top_k_global]
            top_k_reviews = reviews_per_user[top_k_global]

            user_neighbors[measure_name] = {
                "indices": top_k_global,
                "similarities": top_k_sims,
                "reviews": top_k_reviews,
                "total_neighbors": len(valid_values),
            }

            print(f"\n     {measure_name} — Top-{min(TOP_K, len(top_k_global))} voisins "
                  f"(sur {len(valid_values):,} non nuls) :")
            print(f"     {'Rang':>5s} {'Indice':>8s} {'Sim':>10s} {'Reviews':>8s} {'User ID':>25s}")
            print(f"     {'─' * 56}")

            for rank, (v_idx, sim_val, n_rev_v) in enumerate(
                    zip(top_k_global, top_k_sims, top_k_reviews), 1):
                print(f"     {rank:>5d} {v_idx:>8d} {sim_val:>+10.4f} "
                      f"{n_rev_v:>8d} {str(user_ids[v_idx]):>25s}")

        sample_neighbors[u_idx] = user_neighbors

    neighbors_results[sample_name] = sample_neighbors

print(f"\n✓ Voisins identifiés pour tous les utilisateurs échantillonnés.")

### 3) Comparaison

- Build a comparison table for each target user
- Compute Jaccard overlap between neighbor sets (top-10 vs top-10) for each pair of measures
- Analyze differences

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# 3) COMPARAISON DES MESURES (Comparison Across Measures)
#
# Pour chaque utilisateur échantillonné, on compare les ensembles de
# 10 plus proches voisins obtenus par chaque mesure.
#
# ── Coefficient de Jaccard entre ensembles de voisins ─────────────────────
#
# Pour quantifier la concordance entre deux mesures A et B, on calcule :
#   J(N_A, N_B) = |N_A ∩ N_B| / |N_A ∪ N_B|
#
# où N_A et N_B sont les ensembles de top-10 voisins.
#
# Interprétation :
#   J = 1.0 → les deux mesures identifient exactement les mêmes voisins
#   J = 0.5 → moitié des voisins en commun
#   J = 0.0 → aucun voisin en commun (les mesures capturent des aspects
#              complètement différents)
#
# ── Pourquoi les mesures peuvent diverger ? ───────────────────────────────
#
# 1. Biais de notation (Cosinus vs Pearson) :
#    Le cosinus favorise les utilisateurs avec des ratings similaires en
#    VALEUR ABSOLUE. Pearson corrige ce biais → peut trouver des voisins
#    différents (un « sévère » et un « généreux » sont corrélés en Pearson
#    mais pas en cosinus).
#
# 2. Volume vs goût (Jaccard vs les autres) :
#    Jaccard mesure le RECOUVREMENT des catalogues (combien de livres en
#    commun), indépendamment des notes. Deux utilisateurs qui notent les
#    mêmes 50 livres auront un Jaccard élevé même si leurs notes sont
#    opposées.
#
# 3. Sparsité (peu d'items en commun) :
#    Avec peu d'items co-notés, Cosinus et Pearson deviennent instables
#    (une seule note en commun peut donner sim = 1.0). Jaccard est plus
#    robuste dans ce cas car il pénalise les faibles recouvrements.
# ══════════════════════════════════════════════════════════════════════════

measure_names = ["Cosinus", "Pearson", "Jaccard"]
measure_pairs = list(combinations(measure_names, 2))

for sample_name, sample_nb in neighbors_results.items():
    sdata = sampled_users[sample_name]

    print(f"\n{'═' * 70}")
    print(f"  Comparaison des mesures : {sample_name}")
    print(f"{'═' * 70}")

    # ──────────────────────────────────────────────────────────────────
    # Tableau comparatif : pour chaque utilisateur, les top-10 de chaque mesure
    # ──────────────────────────────────────────────────────────────────

    for user_rank, (cat, u_idx) in enumerate(zip(sdata["categories"], sdata["indices"])):
        n_rev = sdata["reviews"][user_rank]
        uid = sdata["ids"][user_rank]

        print(f"\n  ── Utilisateur {user_rank + 1}/5 : indice={u_idx}, "
              f"reviews={n_rev}, \"{cat}\" ──")

        nb_data = sample_nb[u_idx]

        # Tableau côte à côte : rang | Cosinus | Pearson | Jaccard
        print(f"\n  {'Rang':>5s} │ {'Cosinus':^20s} │ {'Pearson':^20s} │ {'Jaccard':^20s}")
        print(f"  {'─' * 5}─┼─{'─' * 20}─┼─{'─' * 20}─┼─{'─' * 20}")

        max_rows = max(len(nb_data[m]["indices"]) for m in measure_names)
        for rank in range(min(10, max_rows)):
            row_parts = [f"  {rank + 1:>5d} │"]
            for m in measure_names:
                indices = nb_data[m]["indices"]
                sims = nb_data[m]["similarities"]
                if rank < len(indices):
                    row_parts.append(f" {indices[rank]:>6d} ({sims[rank]:+.3f}) │")
                else:
                    row_parts.append(f" {'—':^20s} │")
            print("".join(row_parts))

        # ──────────────────────────────────────────────────────────────
        # Jaccard overlap entre ensembles de voisins
        # ──────────────────────────────────────────────────────────────

        print(f"\n     Overlap (Jaccard) entre ensembles de top-{TOP_K} voisins :")
        print(f"     {'Paire':^25s} │ {'|A∩B|':>6s} │ {'|A∪B|':>6s} │ {'J(A,B)':>8s}")
        print(f"     {'─' * 25}─┼─{'─' * 6}─┼─{'─' * 6}─┼─{'─' * 8}")

        for m_a, m_b in measure_pairs:
            set_a = set(nb_data[m_a]["indices"].tolist())
            set_b = set(nb_data[m_b]["indices"].tolist())

            inter = len(set_a & set_b)
            union = len(set_a | set_b)
            jaccard_overlap = inter / union if union > 0 else 0.0

            print(f"     {m_a + ' vs ' + m_b:^25s} │ {inter:>6d} │ {union:>6d} │ {jaccard_overlap:>8.3f}")

    # ──────────────────────────────────────────────────────────────────
    # Synthèse : overlap moyen par paire de mesures, toutes catégories
    # ──────────────────────────────────────────────────────────────────

    print(f"\n  ── Synthèse : overlap moyen sur les 5 utilisateurs ──")
    print(f"     {'Paire':^25s} │ {'Overlap moyen':>14s}")
    print(f"     {'─' * 25}─┼─{'─' * 14}")

    for m_a, m_b in measure_pairs:
        overlaps = []
        for u_idx in sample_nb:
            set_a = set(sample_nb[u_idx][m_a]["indices"].tolist())
            set_b = set(sample_nb[u_idx][m_b]["indices"].tolist())
            union = len(set_a | set_b)
            overlaps.append(len(set_a & set_b) / union if union > 0 else 0.0)
        print(f"     {m_a + ' vs ' + m_b:^25s} │ {np.mean(overlaps):>14.3f}")

    print(f"\n  ── Analyse des divergences ──")
    print(f"     • Cosinus vs Pearson : l'overlap devrait être élevé car Pearson")
    print(f"       est un cosinus sur ratings centrés. Les divergences viennent")
    print(f"       d'utilisateurs avec un biais de notation marqué (r̄_u très")
    print(f"       éloigné de la moyenne globale).")
    print(f"     • Cosinus/Pearson vs Jaccard : overlap potentiellement faible")
    print(f"       car Jaccard ignore les VALEURS des notes. Deux utilisateurs")
    print(f"       lisant les mêmes livres mais les appréciant différemment")
    print(f"       sont proches en Jaccard mais éloignés en Pearson.")
    print(f"     • Impact de l'activité : les utilisateurs très actifs tendent")
    print(f"       à avoir des top-10 plus stables entre mesures (plus d'items")
    print(f"       en commun → moins de bruit).")

print(f"\n✓ Comparaison terminée pour tous les échantillons.")

### 4) Distribution des similarités

For each measure:
- Plot the similarity distribution (histogram)
- Compute mean, median, standard deviation
- Identify outliers

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# 4) DISTRIBUTIONS DES SIMILARITÉS
#
# Pour chaque mesure, on analyse la distribution des valeurs de similarité
# stockées (c'est-à-dire celles au-dessus du seuil SIM_THRESHOLD).
#
# ── Pourquoi analyser la distribution ? ───────────────────────────────────
#
# La forme de la distribution informe sur le comportement de la mesure :
#   - Distribution concentrée près de 0 → la plupart des paires sont
#     faiblement similaires (normal pour des données très creuses)
#   - Queue droite épaisse → beaucoup de paires très similaires
#     (possible signal de communautés d'utilisateurs)
#   - Distribution bimodale → deux populations distinctes
#
# ── Détection des outliers ────────────────────────────────────────────────
#
# On utilise la règle de l'IQR (Interquartile Range) :
#   Q1 = 25e percentile, Q3 = 75e percentile
#   IQR = Q3 − Q1
#   Outlier haut  : sim > Q3 + 1.5 × IQR
#   Outlier bas   : sim < Q1 − 1.5 × IQR  (rare ici car seuil > 0)
#
# Les outliers hauts sont les paires exceptionnellement similaires —
# potentiellement des comptes dupliqués, des bots, ou de vrais
# jumeaux de goût.
# ══════════════════════════════════════════════════════════════════════════

SPLIT_DIRS = sorted(glob.glob("sample-*/splits"))

for split_dir in SPLIT_DIRS:
    split_path = Path(split_dir)
    sample_name = split_path.parent.name
    sim_dir = split_path / "similarities"

    print(f"\n{'═' * 70}")
    print(f"  Distributions des similarités : {sample_name}")
    print(f"{'═' * 70}")

    sim_files = {
        "Cosinus": "sim_cosine.npz",
        "Pearson": "sim_pearson.npz",
        "Jaccard": "sim_jaccard.npz",
    }

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f"Distribution des similarités — {sample_name}",
                 fontsize=14, fontweight="bold", y=1.02)

    colors = {"Cosinus": "#2196F3", "Pearson": "#4CAF50", "Jaccard": "#FF9800"}

    for ax_idx, (measure_name, fname) in enumerate(sim_files.items()):
        ax = axes[ax_idx]

        sim_mat = load_npz(sim_dir / fname)
        data = sim_mat.data.copy()

        if len(data) == 0:
            print(f"\n  {measure_name} : aucune paire stockée — distribution vide.")
            ax.text(0.5, 0.5, "Aucune donnée", ha="center", va="center",
                    transform=ax.transAxes, fontsize=12)
            continue

        # ── Statistiques descriptives ─────────────────────────────────

        mean_val = data.mean()
        median_val = np.median(data)
        std_val = data.std()
        min_val = data.min()
        max_val = data.max()

        q1 = np.percentile(data, 25)
        q3 = np.percentile(data, 75)
        iqr = q3 - q1
        outlier_high = q3 + 1.5 * iqr
        outlier_low = q1 - 1.5 * iqr
        n_outliers_high = (data > outlier_high).sum()
        n_outliers_low = (data < outlier_low).sum()

        print(f"\n  ── {measure_name} ──")
        print(f"     Nombre de paires stockées : {len(data):,}")
        print(f"     Moyenne                   : {mean_val:.4f}")
        print(f"     Médiane                   : {median_val:.4f}")
        print(f"     Écart-type                : {std_val:.4f}")
        print(f"     Min / Max                 : {min_val:.4f} / {max_val:.4f}")
        print(f"     Q1 / Q3                   : {q1:.4f} / {q3:.4f}")
        print(f"     IQR                       : {iqr:.4f}")
        print(f"     Seuil outlier haut        : {outlier_high:.4f}")
        print(f"     Outliers hauts            : {n_outliers_high:,} "
              f"({n_outliers_high / len(data) * 100:.2f}%)")
        if measure_name == "Pearson":
            print(f"     Seuil outlier bas         : {outlier_low:.4f}")
            print(f"     Outliers bas              : {n_outliers_low:,} "
                  f"({n_outliers_low / len(data) * 100:.2f}%)")

        # ── Histogramme ───────────────────────────────────────────────

        ax.hist(data, bins=100, color=colors[measure_name], alpha=0.7,
                edgecolor="white", linewidth=0.3)

        # Lignes de référence
        ax.axvline(mean_val, color="red", linestyle="--", linewidth=1.2,
                   label=f"Moyenne = {mean_val:.3f}")
        ax.axvline(median_val, color="darkred", linestyle=":", linewidth=1.2,
                   label=f"Médiane = {median_val:.3f}")
        ax.axvline(outlier_high, color="orange", linestyle="-.", linewidth=1,
                   label=f"Seuil outlier = {outlier_high:.3f}")

        ax.set_title(f"{measure_name}\n({len(data):,} paires)", fontsize=12)
        ax.set_xlabel("Similarité")
        ax.set_ylabel("Nombre de paires")
        ax.legend(fontsize=8, loc="upper right")
        ax.grid(axis="y", alpha=0.3)

    plt.tight_layout()
    plt.show()

    # ── Résumé comparatif ─────────────────────────────────────────────

    print(f"\n  ── Résumé comparatif ──")
    print(f"  {'Mesure':15s} │ {'N paires':>12s} │ {'Moyenne':>8s} │ {'Médiane':>8s} │ "
          f"{'Std':>8s} │ {'Outliers':>8s}")
    print(f"  {'─' * 15}─┼─{'─' * 12}─┼─{'─' * 8}─┼─{'─' * 8}─┼─{'─' * 8}─┼─{'─' * 8}")

    for measure_name, fname in sim_files.items():
        d = load_npz(sim_dir / fname).data
        if len(d) == 0:
            continue
        q1 = np.percentile(d, 25)
        q3 = np.percentile(d, 75)
        out_h = (d > q3 + 1.5 * (q3 - q1)).sum()
        print(f"  {measure_name:15s} │ {len(d):>12,} │ {d.mean():>8.4f} │ "
              f"{np.median(d):>8.4f} │ {d.std():>8.4f} │ {out_h:>8,}")

print(f"\n✓ Distributions analysées et visualisées.")

### 5) Visualisation des similarités

Heatmap of similarities for 30 randomly selected users.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# 5) HEATMAP DE SIMILARITÉ (30 UTILISATEURS)
#
# On sélectionne aléatoirement 30 utilisateurs et on calcule la matrice
# 30×30 de similarité pour chaque mesure. La visualisation en heatmap
# révèle la structure de voisinage :
#   - Blocs chauds sur la diagonale → groupes d'utilisateurs similaires
#   - Lignes/colonnes froides → utilisateurs « isolés » (peu de voisins)
#
# On applique un clustering hiérarchique (Ward) pour réordonner les
# lignes/colonnes et faire apparaître les communautés.
# ══════════════════════════════════════════════════════════════════════════

SEED_HEATMAP = 123
N_SAMPLE = 30

SPLIT_DIRS = sorted(glob.glob("sample-*/splits"))

for split_dir in SPLIT_DIRS:
    split_path = Path(split_dir)
    sample_name = split_path.parent.name
    sim_dir = split_path / "similarities"

    print(f"\n{'═' * 70}")
    print(f"  Heatmap de similarité : {sample_name}")
    print(f"{'═' * 70}")

    R_train = load_npz(split_path / "R_train.npz")
    user_ids = np.load(split_path / "user_ids.npy", allow_pickle=True)
    n_users = R_train.shape[0]

    # ── Sélection aléatoire de 30 utilisateurs ───────────────────────
    rng = np.random.RandomState(SEED_HEATMAP)
    sample_idx = rng.choice(n_users, size=N_SAMPLE, replace=False)
    sample_idx.sort()

    reviews_per_user = np.diff(R_train.indptr)
    print(f"\n  Utilisateurs sélectionnés : {N_SAMPLE} (seed={SEED_HEATMAP})")
    print(f"  Reviews : min={reviews_per_user[sample_idx].min()}, "
          f"moy={reviews_per_user[sample_idx].mean():.1f}, "
          f"max={reviews_per_user[sample_idx].max()}")

    sim_files = {
        "Cosinus": "sim_cosine.npz",
        "Pearson": "sim_pearson.npz",
        "Jaccard": "sim_jaccard.npz",
    }

    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    fig.suptitle(f"Heatmaps de similarité — {sample_name}\n"
                 f"(30 utilisateurs aléatoires, seed={SEED_HEATMAP})",
                 fontsize=14, fontweight="bold")

    cmaps = {"Cosinus": "Blues", "Pearson": "RdBu_r", "Jaccard": "Oranges"}

    for ax_idx, (measure_name, fname) in enumerate(sim_files.items()):
        ax = axes[ax_idx]

        sim_mat = load_npz(sim_dir / fname)
        sim_mat_T = sim_mat.T.tocsr()

        # ── Construction de la sous-matrice 30×30 ─────────────────────
        # Pour chaque paire (i, j) dans sample_idx, on récupère la
        # similarité depuis le triangle supérieur ou sa transposée.
        sub_matrix = np.zeros((N_SAMPLE, N_SAMPLE), dtype=np.float32)

        for i in range(N_SAMPLE):
            for j in range(i + 1, N_SAMPLE):
                u, v = sample_idx[i], sample_idx[j]
                # Assurer u < v pour lire le triangle supérieur
                a, b = min(u, v), max(u, v)
                val = sim_mat[a, b]
                sub_matrix[i, j] = val
                sub_matrix[j, i] = val

        # Diagonale = 1 (similarité avec soi-même)
        np.fill_diagonal(sub_matrix, 1.0)

        # ── Clustering hiérarchique pour réordonner ───────────────────
        # On convertit la similarité en distance : d = 1 − sim
        # (pour Pearson, on utilise 1 − |sim| car les valeurs peuvent
        # être négatives)
        if measure_name == "Pearson":
            dist_matrix = 1.0 - np.abs(sub_matrix)
        else:
            dist_matrix = 1.0 - sub_matrix

        np.fill_diagonal(dist_matrix, 0.0)
        dist_matrix = np.clip(dist_matrix, 0, None)

        # Conversion en forme condensée pour scipy
        from scipy.spatial.distance import squareform
        try:
            condensed = squareform(dist_matrix, checks=False)
            Z = linkage(condensed, method="ward")
            dendro = dendrogram(Z, no_plot=True)
            order = dendro["leaves"]
        except Exception:
            order = list(range(N_SAMPLE))

        # Réordonner la matrice selon le clustering
        sub_ordered = sub_matrix[np.ix_(order, order)]

        # ── Affichage du heatmap ──────────────────────────────────────
        if measure_name == "Pearson":
            vmin, vmax = -1, 1
            norm = TwoSlopeNorm(vmin=vmin, vcenter=0, vmax=vmax)
            im = ax.imshow(sub_ordered, cmap=cmaps[measure_name],
                           norm=norm, aspect="equal")
        else:
            im = ax.imshow(sub_ordered, cmap=cmaps[measure_name],
                           vmin=0, vmax=1, aspect="equal")

        ax.set_title(f"{measure_name}", fontsize=12, fontweight="bold")
        ax.set_xlabel("Utilisateur (réordonné)")
        ax.set_ylabel("Utilisateur (réordonné)")
        fig.colorbar(im, ax=ax, shrink=0.8, label="Similarité")

        # Statistiques de la sous-matrice (hors diagonale)
        mask = ~np.eye(N_SAMPLE, dtype=bool)
        vals = sub_ordered[mask]
        nonzero_vals = vals[vals != 0]

        print(f"\n  ── {measure_name} (sous-matrice 30×30) ──")
        print(f"     Paires non nulles : {len(nonzero_vals)} / {len(vals)}")
        if len(nonzero_vals) > 0:
            print(f"     Moyenne (non nul) : {nonzero_vals.mean():.4f}")
            print(f"     Max               : {nonzero_vals.max():.4f}")

    plt.tight_layout()
    plt.show()

print(f"\n✓ Heatmaps générées pour tous les échantillons.")

### 6) Discussion

Analysis of strengths and weaknesses of each similarity measure.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# 6) DISCUSSION : FORCES ET FAIBLESSES DES MESURES DE SIMILARITÉ
#
# Cette section analyse qualitativement les trois mesures selon quatre
# axes : sparsité, biais utilisateur, temps de calcul, et pertinence
# pour la recommandation de livres.
# ══════════════════════════════════════════════════════════════════════════

print("╔══════════════════════════════════════════════════════════════════════╗")
print("║  6) DISCUSSION : ANALYSE COMPARATIVE DES MESURES DE SIMILARITÉ     ║")
print("╚══════════════════════════════════════════════════════════════════════╝")

print("""
  ══════════════════════════════════════════════════════════════════════
  A) IMPACT DE LA SPARSITÉ
  ══════════════════════════════════════════════════════════════════════

  Notre matrice R_train a une sparsité > 99.8%, ce qui signifie que
  deux utilisateurs pris au hasard partagent très peu de livres.

  • Cosinus :
    Sensible à la sparsité. Quand deux utilisateurs n'ont que 1–2 livres
    en commun, le produit scalaire r_u · r_v repose sur très peu de
    termes. Une seule note élevée partagée peut artificiellement gonfler
    la similarité (ex: deux utilisateurs qui ont seulement noté 5★ le
    même best-seller → sim ≈ 1.0, alors qu'ils n'ont rien d'autre en
    commun).

  • Pearson :
    Encore plus sensible que le cosinus dans les cas extrêmes. Avec
    seulement 2 items en commun, la corrélation est soit +1, soit −1,
    soit indéfinie — aucune nuance possible. C'est pourquoi on impose
    un minimum d'items en commun (MIN_COMMON_PEARSON = 3). Le centrage
    par la moyenne aide à désambiguïser les profils, mais ne compense
    pas le manque de données.

  • Jaccard :
    Le PLUS ROBUSTE face à la sparsité. Il pénalise naturellement les
    faibles recouvrements : si u a noté 50 livres et v en a noté 40,
    mais seulement 2 en commun, Jaccard = 2/88 ≈ 0.023 — très faible.
    Pas de risque de similarité artificielle sur peu d'items.
    En revanche, Jaccard ignore les notes → il ne distingue pas un
    accord (les deux aiment) d'un désaccord (l'un aime, l'autre pas).

  ══════════════════════════════════════════════════════════════════════
  B) SENSIBILITÉ AU BIAIS DE NOTATION
  ══════════════════════════════════════════════════════════════════════

  Les utilisateurs n'utilisent pas l'échelle de 1 à 5 de la même façon :
    - « Généreux » : note moyenne r̄_u ≈ 4.5 (presque tout est bon)
    - « Sévère »   : note moyenne r̄_u ≈ 2.5 (rarement satisfait)
    - « Binaire »  : ne note que 1★ ou 5★ (aime ou déteste)

  • Cosinus :
    NE CORRIGE PAS le biais. Un utilisateur « généreux » (vecteur avec
    beaucoup de 4–5) aura un fort produit scalaire avec presque tout le
    monde → beaucoup de faux voisins. C'est le principal défaut du
    cosinus brut pour la recommandation.

  • Pearson :
    CORRIGE le biais. En centrant par r̄_u, un « généreux » avec r_u =
    [5,4,5,3] devient [+0.75, −0.25, +0.75, −1.25], et un « sévère »
    avec r_v = [3,2,3,1] devient [+0.75, −0.25, +0.75, −1.25].
    Les vecteurs centrés sont IDENTIQUES → Pearson = 1.0.
    C'est l'avantage majeur de Pearson : il capture les préférences
    RELATIVES (quels livres sont meilleurs/pires QUE LA MOYENNE de
    l'utilisateur).

  • Jaccard :
    INSENSIBLE au biais (il ignore les notes). Un utilisateur qui note
    tout à 5★ et un qui note tout à 1★ les mêmes livres auront un
    Jaccard très élevé, malgré des appréciations opposées.

  ══════════════════════════════════════════════════════════════════════
  C) TEMPS DE CALCUL ET SCALABILITÉ
  ══════════════════════════════════════════════════════════════════════

  • Cosinus :
    Le PLUS RAPIDE. sklearn.cosine_similarity exploite des routines
    BLAS (C/Fortran) optimisées pour le produit de matrices creuses.
    Complexité : O(|U| × nnz_avg × |U| / batch) par batch.
    Sur nos données (~10K users) : quelques secondes.

  • Pearson (adjusted cosine) :
    MÊME VITESSE que le cosinus. Le centrage est O(nnz) en une passe,
    puis on réutilise cosine_similarity sur la matrice centrée.
    Le filtre par nombre d'items en commun (R_binary @ R_binary.T)
    ajoute un produit matriciel creux supplémentaire, mais reste rapide.

  • Jaccard :
    RAPIDE grâce à la vectorisation. Le calcul d'intersection (produit
    de matrices binaires) est efficace en CSR. L'union se déduit par
    broadcasting numpy. Comparable en vitesse au cosinus.

  → Les trois mesures sont computables en minutes sur nos données.
    Pour des jeux 10× plus grands (100K+ users), le cosinus et
    l'adjusted cosine restent praticables ; Jaccard pourrait nécessiter
    un index inversé ou un échantillonnage pour éviter le produit
    matriciel complet.

  ══════════════════════════════════════════════════════════════════════
  D) PERTINENCE POUR LA RECOMMANDATION DE LIVRES
  ══════════════════════════════════════════════════════════════════════

  • Cosinus :
    Bon choix de BASE. Simple, rapide, et performant quand les
    utilisateurs ont des profils de notation similaires (même r̄_u).
    Moins adapté quand les biais de notation varient fortement.

  • Pearson :
    MEILLEUR CHOIX GLOBAL pour la recommandation de livres.
    - Corrige le biais → deux lecteurs avec les mêmes goûts mais des
      habitudes de notation différentes sont correctement identifiés.
    - Standard dans la littérature (Resnick et al. 1994, Herlocker 1999).
    - Les corrélations négatives peuvent être exploitées pour éviter
      de recommander des livres détestés par des « anti-voisins ».

  • Jaccard :
    Utile en COMPLÉMENT, pas seul. Jaccard capture « lisent-ils les
    mêmes livres ? » sans dire « les apprécient-ils ? ». Un Jaccard
    élevé avec un Pearson faible révèle des utilisateurs qui explorent
    le même genre mais divergent dans leurs goûts — information utile
    pour la diversification des recommandations.

  ── RECOMMANDATION FINALE ─────────────────────────────────────────────

  Pour un système de recommandation de livres sur ce jeu Amazon :

  1. Utiliser PEARSON comme mesure principale (corrige le biais,
     capture les préférences relatives).
  2. Utiliser JACCARD comme filtre préalable ou pondération :
     ne considérer comme voisins que les utilisateurs avec un Jaccard
     minimum (ex: > 0.05), ce qui garantit un recouvrement suffisant
     pour que la corrélation soit fiable.
  3. Garder COSINUS comme baseline de comparaison pour mesurer
     l'apport du centrage Pearson.
""")

print("✓ Discussion terminée.")

# 3.3 Tâche 2: Représentation en graphe

## 3.3.1 Construction du graphe biparti

### 1) Graphe biparti complet

Build G = (U ∪ I, E) where:
- U = set of user nodes
- I = set of book nodes
- E = set of edges (u, i) with weight w_{u,i} = r_{u,i} (the rating)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# 3.3.1 — CONSTRUCTION DU GRAPHE BIPARTI COMPLET
#
# Un graphe biparti G = (U ∪ I, E) modélise les interactions entre
# utilisateurs et livres. Chaque nœud appartient à l'un des deux
# ensembles (U ou I), et les arêtes ne relient que des nœuds de
# types différents (un utilisateur à un livre, jamais deux utilisateurs
# entre eux ni deux livres entre eux).
#
# ── Pourquoi un graphe biparti ? ──────────────────────────────────────────
#
# La matrice R (user × item) et le graphe biparti sont deux
# représentations ÉQUIVALENTES du même jeu de données :
#   - R[u, i] = w   ↔   arête (u, i) de poids w dans G
#   - Ligne u de R   ↔   voisins du nœud u dans G
#
# L'avantage du graphe : il donne accès à tout l'arsenal de la théorie
# des graphes (centralité, composantes connexes, clustering, chemins,
# communautés) qui révèle des structures invisibles dans la matrice.
#
# ── Stratégie de construction ─────────────────────────────────────────────
#
# Pour un échantillon avec ~500K reviews, le graphe aura :
#   |V| ≈ 10 000 + 44 000 = 54 000 nœuds
#   |E| ≈ 490 000 arêtes
#
# NetworkX stocke chaque nœud et arête comme un objet Python avec un
# dictionnaire d'attributs. Pour ~500K arêtes, cela consomme ~200 Mo.
#
# Optimisations appliquées :
#   1. Construction incrémentale depuis la matrice CSR (pas de DataFrame
#      intermédiaire) — on itère directement sur les entrées non nulles.
#   2. Préfixage des identifiants pour distinguer users et books :
#      "U_<idx>" et "I_<idx>" — évite les collisions si un user_id
#      ressemble à un parent_asin.
#   3. Attribut bipartite=0/1 sur chaque nœud pour que les fonctions
#      nx.bipartite.* les reconnaissent automatiquement.
# ══════════════════════════════════════════════════════════════════════════

SPLIT_DIRS = sorted(glob.glob("sample-*/splits"))

graphs = {}

for split_dir in SPLIT_DIRS:
    split_path = Path(split_dir)
    sample_name = split_path.parent.name

    print(f"\n{'═' * 70}")
    print(f"  Graphe biparti complet : {sample_name}")
    print(f"{'═' * 70}")

    # ── Chargement ────────────────────────────────────────────────────
    R_train = load_npz(split_path / "R_train.npz")
    user_ids = np.load(split_path / "user_ids.npy", allow_pickle=True)
    item_ids = np.load(split_path / "item_ids.npy", allow_pickle=True)

    n_users, n_items = R_train.shape
    n_ratings = R_train.nnz

    print(f"\n  ── Données source ──")
    print(f"     Matrice R_train : {n_users:,} users × {n_items:,} items")
    print(f"     Ratings (nnz)   : {n_ratings:,}")

    # ──────────────────────────────────────────────────────────────────
    # Étape 1 : Création des nœuds
    # ──────────────────────────────────────────────────────────────────
    # On ajoute les nœuds avec l'attribut bipartite=0 pour les users
    # et bipartite=1 pour les items (convention NetworkX).
    # L'attribut « original_id » conserve l'identifiant Amazon original.
    # ──────────────────────────────────────────────────────────────────

    t0 = time.perf_counter()

    G = nx.Graph()

    user_nodes = [(f"U_{i}", {"bipartite": 0, "original_id": str(user_ids[i])})
                  for i in range(n_users)]
    G.add_nodes_from(user_nodes)

    item_nodes = [(f"I_{j}", {"bipartite": 1, "original_id": str(item_ids[j])})
                  for j in range(n_items)]
    G.add_nodes_from(item_nodes)

    t_nodes = time.perf_counter() - t0

    print(f"\n  ── Étape 1 : Création des nœuds ──")
    print(f"     Nœuds utilisateurs (U) : {n_users:,}  (bipartite=0)")
    print(f"     Nœuds livres (I)       : {n_items:,}  (bipartite=1)")
    print(f"     Total |V| = |U| + |I|  : {n_users + n_items:,}")
    print(f"     Temps                   : {t_nodes * 1000:.0f} ms")

    # ──────────────────────────────────────────────────────────────────
    # Étape 2 : Création des arêtes (incrémentale depuis CSR)
    # ──────────────────────────────────────────────────────────────────
    # La matrice CSR stocke les données ligne par ligne :
    #   - R.indptr[u] à R.indptr[u+1] : plage des données de la ligne u
    #   - R.indices[...] : indices de colonnes (livres)
    #   - R.data[...]    : valeurs (ratings)
    #
    # On construit la liste d'arêtes pondérées en une seule passe sur
    # la structure CSR, sans créer de DataFrame intermédiaire.
    # ──────────────────────────────────────────────────────────────────

    t1 = time.perf_counter()

    # Extraction vectorisée des triplets (user_idx, item_idx, rating)
    # depuis la matrice CSR — plus rapide qu'une boucle Python.
    coo = R_train.tocoo()
    edges = [(f"U_{u}", f"I_{i}", float(w))
             for u, i, w in zip(coo.row, coo.col, coo.data)]
    G.add_weighted_edges_from(edges)

    t_edges = time.perf_counter() - t1

    print(f"\n  ── Étape 2 : Création des arêtes ──")
    print(f"     Arêtes ajoutées |E| : {G.number_of_edges():,}")
    print(f"     Poids = rating      : min={R_train.data.min():.1f}, "
          f"max={R_train.data.max():.1f}, moy={R_train.data.mean():.2f}")
    print(f"     Temps               : {t_edges:.1f}s")

    # ──────────────────────────────────────────────────────────────────
    # Étape 3 : Validation (sanity checks)
    # ──────────────────────────────────────────────────────────────────

    n_nodes_expected = n_users + n_items
    n_edges_expected = n_ratings

    assert G.number_of_nodes() == n_nodes_expected, \
        f"Erreur : {G.number_of_nodes()} nœuds ≠ {n_nodes_expected} attendus"
    assert G.number_of_edges() == n_edges_expected, \
        f"Erreur : {G.number_of_edges()} arêtes ≠ {n_edges_expected} attendues"

    user_set = {n for n, d in G.nodes(data=True) if d["bipartite"] == 0}
    item_set = {n for n, d in G.nodes(data=True) if d["bipartite"] == 1}
    assert len(user_set) == n_users
    assert len(item_set) == n_items
    assert nx.is_bipartite(G), "Le graphe n'est pas biparti !"

    t_total = time.perf_counter() - t0

    print(f"\n  ── Étape 3 : Validation ──")
    print(f"     ✓ |V| = {G.number_of_nodes():,} (attendu : {n_nodes_expected:,})")
    print(f"     ✓ |E| = {G.number_of_edges():,} (attendu : {n_edges_expected:,})")
    print(f"     ✓ |U| = {len(user_set):,}, |I| = {len(item_set):,}")
    print(f"     ✓ Graphe biparti : {nx.is_bipartite(G)}")
    print(f"     Temps total       : {t_total:.1f}s")

    print(f"\n  {'─' * 66}")
    print(f"  RÉSUMÉ — {sample_name}")
    print(f"  {'─' * 66}")
    print(f"     |U| (utilisateurs) : {n_users:,}")
    print(f"     |I| (livres)       : {n_items:,}")
    print(f"     |V| = |U| + |I|   : {G.number_of_nodes():,}")
    print(f"     |E| (interactions) : {G.number_of_edges():,}")
    print(f"     Degré moyen        : {2 * G.number_of_edges() / G.number_of_nodes():.2f}")
    print(f"     Densité bipartie   : {G.number_of_edges() / (n_users * n_items) * 100:.4f}%")
    print(f"  {'─' * 66}")

    graphs[sample_name] = {
        "G": G,
        "user_set": user_set,
        "item_set": item_set,
        "user_ids": user_ids,
        "item_ids": item_ids,
        "n_users": n_users,
        "n_items": n_items,
    }

print(f"\n✓ {len(graphs)} graphe(s) biparti(s) construit(s) et stocké(s) dans `graphs`.")

### 2) Sous-graphe reduit

Create a subgraph containing:
- The 30 most active users
- The 50 most popular books
- The corresponding edges

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# SOUS-GRAPHE RÉDUIT POUR LA VISUALISATION
#
# Le graphe complet (~55 000 nœuds, ~490 000 arêtes) est trop dense
# pour être visualisé lisiblement. On extrait un sous-graphe contenant :
#   - Les 30 utilisateurs les plus actifs (plus haut degré dans G)
#   - Les 50 livres les plus populaires (plus haut degré dans G)
#   - Les arêtes entre ces nœuds uniquement
#
# Ce sous-graphe représente le « cœur » du réseau : les power users
# et les best-sellers qui concentrent l'essentiel des interactions.
#
# ── Pourquoi 30 users × 50 books ? ───────────────────────────────────────
#
# Compromis entre lisibilité et représentativité :
#   - 80 nœuds max → le graphe reste lisible avec des labels
#   - Les 30 top users ont beaucoup de livres en commun parmi les 50
#     top books → le sous-graphe sera connecté et dense
#   - On peut observer des motifs (cliques, hubs, ponts) impossibles
#     à voir sur le graphe complet
# ══════════════════════════════════════════════════════════════════════════

TOP_USERS = 30
TOP_BOOKS = 50

subgraphs = {}

for sample_name, gdata in graphs.items():
    G = gdata["G"]
    user_set = gdata["user_set"]
    item_set = gdata["item_set"]

    print(f"\n{'═' * 70}")
    print(f"  Sous-graphe réduit : {sample_name}")
    print(f"{'═' * 70}")

    # ──────────────────────────────────────────────────────────────────
    # Étape 1 : Calcul de l'activité (degré) par type de nœud
    # ──────────────────────────────────────────────────────────────────
    # Le degré d'un nœud dans le graphe biparti = nombre d'interactions.
    # Pour un utilisateur : degré = nombre de livres notés.
    # Pour un livre : degré = nombre de lecteurs qui l'ont noté.
    # ──────────────────────────────────────────────────────────────────

    user_degrees = {n: G.degree(n) for n in user_set}
    item_degrees = {n: G.degree(n) for n in item_set}

    top_users = sorted(user_degrees, key=user_degrees.get, reverse=True)[:TOP_USERS]
    top_books = sorted(item_degrees, key=item_degrees.get, reverse=True)[:TOP_BOOKS]

    print(f"\n  ── Étape 1 : Sélection des nœuds ──")
    print(f"     Top {TOP_USERS} utilisateurs (par activité/degré) :")
    print(f"       Degré min : {user_degrees[top_users[-1]]:,}")
    print(f"       Degré max : {user_degrees[top_users[0]]:,}")
    print(f"       Degré moy : {np.mean([user_degrees[u] for u in top_users]):.1f}")
    print(f"     Top {TOP_BOOKS} livres (par popularité/degré) :")
    print(f"       Degré min : {item_degrees[top_books[-1]]:,}")
    print(f"       Degré max : {item_degrees[top_books[0]]:,}")
    print(f"       Degré moy : {np.mean([item_degrees[b] for b in top_books]):.1f}")

    # ──────────────────────────────────────────────────────────────────
    # Étape 2 : Extraction du sous-graphe induit
    # ──────────────────────────────────────────────────────────────────
    # nx.subgraph(G, nodes) retourne une VIEW (pas une copie) sur G.
    # On copie pour avoir un graphe indépendant modifiable.
    # ──────────────────────────────────────────────────────────────────

    selected_nodes = set(top_users) | set(top_books)
    G_sub = G.subgraph(selected_nodes).copy()

    n_sub_users = len(set(top_users) & set(G_sub.nodes()))
    n_sub_books = len(set(top_books) & set(G_sub.nodes()))

    print(f"\n  ── Étape 2 : Sous-graphe induit ──")
    print(f"     Nœuds sélectionnés : {len(selected_nodes):,} "
          f"({n_sub_users} users + {n_sub_books} books)")
    print(f"     Arêtes dans le sous-graphe : {G_sub.number_of_edges():,}")

    max_edges_sub = n_sub_users * n_sub_books
    density_sub = G_sub.number_of_edges() / max_edges_sub if max_edges_sub > 0 else 0

    print(f"     Arêtes max possibles : {max_edges_sub:,}")
    print(f"     Densité bipartie     : {density_sub * 100:.2f}%")
    print(f"     Degré moyen          : {2 * G_sub.number_of_edges() / G_sub.number_of_nodes():.2f}")

    # ──────────────────────────────────────────────────────────────────
    # Étape 3 : Validation
    # ──────────────────────────────────────────────────────────────────

    sub_user_set = {n for n, d in G_sub.nodes(data=True) if d["bipartite"] == 0}
    sub_item_set = {n for n, d in G_sub.nodes(data=True) if d["bipartite"] == 1}

    assert len(sub_user_set) == n_sub_users
    assert len(sub_item_set) == n_sub_books
    assert nx.is_bipartite(G_sub)

    for u, v in G_sub.edges():
        assert (u in sub_user_set and v in sub_item_set) or \
               (v in sub_user_set and u in sub_item_set), \
            f"Arête invalide : ({u}, {v}) ne relie pas un user à un book"

    print(f"\n  ── Étape 3 : Validation ──")
    print(f"     ✓ Biparti : {nx.is_bipartite(G_sub)}")
    print(f"     ✓ {n_sub_users} users, {n_sub_books} books")
    print(f"     ✓ Toutes les arêtes relient user ↔ book")

    subgraphs[sample_name] = {
        "G_sub": G_sub,
        "top_users": top_users,
        "top_books": top_books,
        "sub_user_set": sub_user_set,
        "sub_item_set": sub_item_set,
    }

print(f"\n✓ {len(subgraphs)} sous-graphe(s) réduit(s) construit(s).")

### 3) Visualisation du sous-graphe

Bipartite layout with:
- Node sizes proportional to degree
- Different colors for users vs books
- Edge thickness proportional to rating weight

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# VISUALISATION DU SOUS-GRAPHE BIPARTI
#
# On utilise un layout biparti où les utilisateurs sont placés à gauche
# et les livres à droite, afin de bien mettre en évidence la structure
# bipartie du réseau.
#
# ── Encodage visuel ───────────────────────────────────────────────────────
#
# • Couleur : bleu pour les utilisateurs, orange pour les livres
# • Taille : proportionnelle au degré dans le sous-graphe
#   (un nœud avec plus de connexions est plus gros)
# • Arêtes : gris transparent pour ne pas surcharger visuellement
#   (l'épaisseur est proportionnelle au poids/rating)
# • Layout : users à gauche (x=0), books à droite (x=2),
#   répartis verticalement par degré décroissant (hubs en haut)
# ══════════════════════════════════════════════════════════════════════════

for sample_name, sgdata in subgraphs.items():
    G_sub = sgdata["G_sub"]
    sub_user_set = sgdata["sub_user_set"]
    sub_item_set = sgdata["sub_item_set"]

    print(f"\n{'═' * 70}")
    print(f"  Visualisation : {sample_name}")
    print(f"{'═' * 70}")

    # ──────────────────────────────────────────────────────────────────
    # Layout biparti personnalisé
    # ──────────────────────────────────────────────────────────────────
    # Users à x=0, books à x=2. Dans chaque colonne, les nœuds sont
    # ordonnés par degré décroissant (hubs en haut) pour lisibilité.
    # ──────────────────────────────────────────────────────────────────

    users_sorted = sorted(sub_user_set, key=lambda n: G_sub.degree(n), reverse=True)
    books_sorted = sorted(sub_item_set, key=lambda n: G_sub.degree(n), reverse=True)

    pos = {}
    n_u = len(users_sorted)
    n_b = len(books_sorted)
    for i, u in enumerate(users_sorted):
        pos[u] = (0, -i)
    for j, b in enumerate(books_sorted):
        pos[b] = (2, -j * n_u / max(n_b, 1))

    # ──────────────────────────────────────────────────────────────────
    # Calcul des tailles et couleurs
    # ──────────────────────────────────────────────────────────────────

    degrees = dict(G_sub.degree())

    node_colors = []
    node_sizes = []
    for n in G_sub.nodes():
        if n in sub_user_set:
            node_colors.append("#2196F3")
            node_sizes.append(50 + degrees[n] * 8)
        else:
            node_colors.append("#FF9800")
            node_sizes.append(50 + degrees[n] * 8)

    edge_weights = [G_sub[u][v].get("weight", 1.0) for u, v in G_sub.edges()]
    max_weight = max(edge_weights) if edge_weights else 1
    edge_widths = [0.3 + 1.5 * w / max_weight for w in edge_weights]

    # ──────────────────────────────────────────────────────────────────
    # Tracé
    # ──────────────────────────────────────────────────────────────────

    fig, ax = plt.subplots(1, 1, figsize=(14, max(10, n_u * 0.35)))

    nx.draw_networkx_edges(G_sub, pos, ax=ax,
                           edge_color="#CCCCCC", alpha=0.4,
                           width=edge_widths)

    nx.draw_networkx_nodes(G_sub, pos, ax=ax,
                           node_color=node_colors,
                           node_size=node_sizes,
                           edgecolors="white", linewidths=0.5)

    labels = {n: n.split("_")[1] for n in G_sub.nodes()}
    nx.draw_networkx_labels(G_sub, pos, labels=labels, ax=ax,
                            font_size=6, font_color="black")

    user_patch = mpatches.Patch(color="#2196F3", label=f"Utilisateurs ({len(sub_user_set)})")
    book_patch = mpatches.Patch(color="#FF9800", label=f"Livres ({len(sub_item_set)})")
    ax.legend(handles=[user_patch, book_patch], loc="upper right", fontsize=10)

    ax.set_title(f"Graphe biparti réduit — {sample_name}\n"
                 f"({len(sub_user_set)} users × {len(sub_item_set)} books, "
                 f"{G_sub.number_of_edges()} arêtes)",
                 fontsize=13, fontweight="bold")
    ax.axis("off")

    plt.tight_layout()
    out_path = f"{sample_name}_bipartite_subgraph.png"
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.show()

    print(f"  ✓ Figure sauvegardée : {out_path}")
    print(f"     Nœuds affichés  : {G_sub.number_of_nodes()}")
    print(f"     Arêtes affichées : {G_sub.number_of_edges()}")

print(f"\n✓ Visualisations générées.")

## 3.3.2 Analyse du graphe

### 1) Métriques globales

*On the full graph*
- Number of nodes: |V| = |U| + |I|
- Number of edges: |E|
- Average degree: d̄ = 2|E| / |V|
- Density: δ = |E| / (|U| × |I|)
- Degree distribution (log-log scale)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# 3.3.2 — MÉTRIQUES GLOBALES DU GRAPHE BIPARTI
#
# On calcule les métriques structurelles du graphe biparti complet
# pour caractériser le réseau d'interactions utilisateur-livre.
#
# ── Métriques calculées ───────────────────────────────────────────────────
#
# 1. |V| = |U| + |I| : nombre total de nœuds
# 2. |E| : nombre d'arêtes (interactions)
# 3. d̄ = 2|E| / |V| : degré moyen global
#    (chaque arête contribue +1 au degré de chaque extrémité → facteur 2)
# 4. δ = |E| / (|U| × |I|) : densité bipartie
#    (proportion de cases remplies dans la matrice U × I)
# 5. Distribution des degrés (séparée users/books) en échelle log-log
#    pour détecter un comportement en loi de puissance (power law)
#
# ── Loi de puissance (power law) ──────────────────────────────────────────
#
# Si P(k) ∝ k^{-γ}, la distribution des degrés suit une loi de puissance.
# En échelle log-log, cela se traduit par une droite de pente −γ.
# C'est une propriété universelle des réseaux sociaux et commerciaux :
#   - Quelques « hubs » (power users ou best-sellers) ont un très haut
#     degré
#   - La majorité des nœuds ont un degré faible
#   - La distribution a une « queue lourde » (heavy tail)
# ══════════════════════════════════════════════════════════════════════════

for sample_name, gdata in graphs.items():
    G = gdata["G"]
    user_set = gdata["user_set"]
    item_set = gdata["item_set"]
    n_users = gdata["n_users"]
    n_items = gdata["n_items"]

    print(f"\n{'═' * 70}")
    print(f"  Métriques globales : {sample_name}")
    print(f"{'═' * 70}")

    n_V = G.number_of_nodes()
    n_E = G.number_of_edges()
    avg_degree = 2 * n_E / n_V
    density = n_E / (n_users * n_items)

    user_degs = np.array([G.degree(u) for u in user_set])
    item_degs = np.array([G.degree(i) for i in item_set])

    print(f"\n  ── Métriques de base ──")
    print(f"     |U| (utilisateurs)            : {n_users:,}")
    print(f"     |I| (livres)                  : {n_items:,}")
    print(f"     |V| = |U| + |I|              : {n_V:,}")
    print(f"     |E| (arêtes)                  : {n_E:,}")
    print(f"     Degré moyen d̄ = 2|E|/|V|     : {avg_degree:.2f}")
    print(f"     Densité δ = |E|/(|U|×|I|)    : {density:.6f} ({density * 100:.4f}%)")

    print(f"\n  ── Degrés par type de nœud ──")
    print(f"     Utilisateurs :")
    print(f"       min={user_degs.min()}, Q1={np.percentile(user_degs, 25):.0f}, "
          f"médiane={np.median(user_degs):.0f}, "
          f"Q3={np.percentile(user_degs, 75):.0f}, max={user_degs.max()}")
    print(f"       moyenne={user_degs.mean():.1f}, écart-type={user_degs.std():.1f}")
    print(f"     Livres :")
    print(f"       min={item_degs.min()}, Q1={np.percentile(item_degs, 25):.0f}, "
          f"médiane={np.median(item_degs):.0f}, "
          f"Q3={np.percentile(item_degs, 75):.0f}, max={item_degs.max()}")
    print(f"       moyenne={item_degs.mean():.1f}, écart-type={item_degs.std():.1f}")

    # ──────────────────────────────────────────────────────────────────
    # Distribution des degrés en échelle log-log
    # ──────────────────────────────────────────────────────────────────
    # P(k) = nombre de nœuds de degré k.
    # En log-log, une loi de puissance apparaît comme une droite.
    # On estime l'exposant γ par régression linéaire sur log(k) vs log(P(k)).
    # ──────────────────────────────────────────────────────────────────

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f"Distribution des degrés (log-log) — {sample_name}",
                 fontsize=13, fontweight="bold")

    for ax, degs, label, color in [
        (axes[0], user_degs, "Utilisateurs", "#2196F3"),
        (axes[1], item_degs, "Livres", "#FF9800"),
    ]:
        deg_counts = Counter(degs)
        ks = np.array(sorted(deg_counts.keys()))
        counts = np.array([deg_counts[k] for k in ks])

        ax.scatter(ks, counts, s=15, alpha=0.6, color=color, edgecolors="none")
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.set_xlabel("Degré k", fontsize=11)
        ax.set_ylabel("Nombre de nœuds P(k)", fontsize=11)
        ax.set_title(f"{label} (n={len(degs):,})", fontsize=11)
        ax.grid(True, alpha=0.3, which="both")

        # Régression linéaire en log-log pour estimer γ
        log_k = np.log10(ks[ks > 0].astype(float))
        log_c = np.log10(counts[ks > 0].astype(float))
        if len(log_k) > 2:
            coeffs = np.polyfit(log_k, log_c, 1)
            gamma = -coeffs[0]
            fit_line = 10 ** np.polyval(coeffs, log_k)
            ax.plot(ks[ks > 0], fit_line, "--", color="red", alpha=0.7,
                    label=f"Pente ≈ −{gamma:.2f}")
            ax.legend(fontsize=9)

            print(f"\n     Distribution {label} :")
            print(f"       Exposant estimé γ ≈ {gamma:.2f}")
            if 2.0 < gamma < 3.5:
                print(f"       → Compatible avec une loi de puissance (γ typique : 2–3)")
            else:
                print(f"       → Pente hors de la plage typique [2, 3.5] — distribution")
                print(f"         possiblement log-normale ou exponentielle tronquée")

    plt.tight_layout()
    plt.show()

print(f"\n✓ Métriques globales calculées.")

### 2) Centralités des Livres

Book degree centrality: C_d(i) = deg(i) / |U|

- Identify the 20 books with highest centrality
- Compare with the most popular books (by number of reviews)
- Analyze differences

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# CENTRALITÉ DES LIVRES (Book Degree Centrality)
#
# La centralité de degré d'un livre i dans un graphe biparti mesure
# la fraction d'utilisateurs qui ont noté ce livre :
#
#   C_d(i) = deg(i) / |U|
#
# Interprétation :
#   C_d(i) = 1.0 → tous les utilisateurs ont noté ce livre
#   C_d(i) = 0.001 → seul 0.1% des utilisateurs l'ont noté
#
# ── Centralité vs Popularité ──────────────────────────────────────────────
#
# Dans le graphe COMPLET, la centralité est exactement proportionnelle
# au nombre de reviews (popularité). Les classements sont identiques.
#
# Dans le SOUS-GRAPHE, la centralité est calculée par rapport aux |U|
# du graphe complet, mais le degré est mesuré dans le sous-graphe
# (seules les arêtes des 30 top users sont présentes). Un livre
# populaire globalement peut avoir peu de connexions dans le sous-graphe
# si ses lecteurs ne sont pas parmi les 30 top users.
# ══════════════════════════════════════════════════════════════════════════

TOP_N = 20

for sample_name, gdata in graphs.items():
    G = gdata["G"]
    user_set = gdata["user_set"]
    item_set = gdata["item_set"]
    n_users = gdata["n_users"]
    item_ids = gdata["item_ids"]
    sgdata = subgraphs[sample_name]
    G_sub = sgdata["G_sub"]
    sub_item_set = sgdata["sub_item_set"]

    print(f"\n{'═' * 70}")
    print(f"  Centralité des livres : {sample_name}")
    print(f"{'═' * 70}")

    # ──────────────────────────────────────────────────────────────────
    # Centralité dans le sous-graphe
    # C_d(i) = deg_sub(i) / |U_full|
    # On divise par |U| du graphe COMPLET (convention de la formule).
    # ──────────────────────────────────────────────────────────────────

    centrality_sub = {}
    for node in sub_item_set:
        deg = G_sub.degree(node)
        centrality_sub[node] = deg / n_users

    top_centrality = sorted(centrality_sub.items(), key=lambda x: x[1], reverse=True)[:TOP_N]

    # Popularité dans le graphe complet (degré global)
    popularity_full = {node: G.degree(node) for node in item_set}
    top_popularity = sorted(popularity_full.items(), key=lambda x: x[1], reverse=True)[:TOP_N]

    # ── Tableau : Top 20 par centralité (sous-graphe) ─────────────────

    print(f"\n  ── Top {TOP_N} livres par centralité C_d(i) dans le sous-graphe ──")
    print(f"     (C_d = deg_sub / |U_full|, |U_full| = {n_users:,})")
    print(f"\n  {'Rang':>5s} │ {'Nœud':>10s} │ {'deg(sub)':>9s} │ {'C_d(i)':>9s} │ "
          f"{'deg(full)':>10s} │ {'ASIN':>15s}")
    print(f"  {'─' * 5}─┼─{'─' * 10}─┼─{'─' * 9}─┼─{'─' * 9}─┼─{'─' * 10}─┼─{'─' * 15}")

    for rank, (node, cd) in enumerate(top_centrality, 1):
        deg_sub = G_sub.degree(node)
        deg_full = G.degree(node)
        idx = int(node.split("_")[1])
        asin = str(item_ids[idx])
        print(f"  {rank:>5d} │ {node:>10s} │ {deg_sub:>9d} │ {cd:>9.6f} │ "
              f"{deg_full:>10,} │ {asin:>15s}")

    # ── Tableau : Top 20 par popularité (graphe complet) ──────────────

    print(f"\n  ── Top {TOP_N} livres par popularité dans le graphe complet ──")
    print(f"\n  {'Rang':>5s} │ {'Nœud':>10s} │ {'deg(full)':>10s} │ {'C_d(full)':>10s} │ {'ASIN':>15s}")
    print(f"  {'─' * 5}─┼─{'─' * 10}─┼─{'─' * 10}─┼─{'─' * 10}─┼─{'─' * 15}")

    for rank, (node, deg) in enumerate(top_popularity, 1):
        cd_full = deg / n_users
        idx = int(node.split("_")[1])
        asin = str(item_ids[idx])
        print(f"  {rank:>5d} │ {node:>10s} │ {deg:>10,} │ {cd_full:>10.6f} │ {asin:>15s}")

    # ── Overlap entre les deux classements ────────────────────────────

    set_centrality = {node for node, _ in top_centrality}
    set_popularity = {node for node, _ in top_popularity}
    overlap = set_centrality & set_popularity

    print(f"\n  ── Comparaison centralité (sous-graphe) vs popularité (complet) ──")
    print(f"     Top-{TOP_N} en commun          : {len(overlap)}/{TOP_N}")
    print(f"     Seulement dans centralité  : {len(set_centrality - set_popularity)}")
    print(f"     Seulement dans popularité  : {len(set_popularity - set_centrality)}")

    print(f"\n  ── Explication des différences ──")
    print(f"     Le sous-graphe ne contient que les 30 top users et 50 top books.")
    print(f"     Un livre populaire globalement (noté par 500+ utilisateurs) peut")
    print(f"     avoir peu de connexions dans le sous-graphe si ses lecteurs ne")
    print(f"     font pas partie des 30 power users.")
    print(f"     Inversement, un livre avec une popularité modérée peut dominer")
    print(f"     le sous-graphe s'il est le favori des power users.")
    print(f"     → La centralité du sous-graphe capture la popularité PARMI LES")
    print(f"       GROS LECTEURS, pas dans la population générale.")

print(f"\n✓ Analyse de centralité terminée.")

### 3) Coéfficient de clustering *&* 4) Composantes connexes

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# COEFFICIENT DE CLUSTERING ET COMPOSANTES CONNEXES
#
# ── 3) Coefficient de clustering ──────────────────────────────────────────
#
# Le coefficient de clustering classique (Watts-Strogatz) est TOUJOURS 0
# pour un graphe biparti : il n'existe aucun triangle car il n'y a pas
# d'arête user-user ni book-book. On utilise donc le coefficient de
# clustering BIPARTI (Latapy, Magnien & Del Vecchio, 2008).
#
# Pour un nœud u avec voisins v1, v2, ..., vk :
#   cc(u) = fraction des paires de voisins (vi, vj) qui ont au moins
#           un voisin commun dans l'AUTRE partition.
#
# Interprétation pour la recommandation :
#   - Clustering élevé chez un user → ses livres sont fréquemment lus
#     ensemble par d'autres lecteurs → préférences « mainstream »
#   - Clustering faible → goûts éclectiques, livres rarement co-notés
#
# On calcule le clustering sur le SOUS-GRAPHE (le graphe complet serait
# trop lent : O(|E|²) pour les projections).
#
# ── 4) Composantes connexes ──────────────────────────────────────────────
#
# Un graphe peut avoir des « îlots » déconnectés : des groupes
# d'utilisateurs et de livres sans aucun lien avec le reste du réseau.
#
# Implications pour la recommandation :
#   - On ne peut PAS recommander entre composantes déconnectées
#   - Chaque composante forme un « silo » indépendant
#   - Les composantes de taille 1 = nœuds isolés (cold start)
# ══════════════════════════════════════════════════════════════════════════

for sample_name, gdata in graphs.items():
    G = gdata["G"]
    user_set = gdata["user_set"]
    item_set = gdata["item_set"]
    sgdata = subgraphs[sample_name]
    G_sub = sgdata["G_sub"]
    sub_user_set = sgdata["sub_user_set"]
    sub_item_set = sgdata["sub_item_set"]

    print(f"\n{'═' * 70}")
    print(f"  Clustering & composantes : {sample_name}")
    print(f"{'═' * 70}")

    # ==================================================================
    # 3) COEFFICIENT DE CLUSTERING BIPARTI (sur le sous-graphe)
    # ==================================================================

    print(f"\n  ══════════════════════════════════════════════════════════════")
    print(f"  3) COEFFICIENT DE CLUSTERING BIPARTI")
    print(f"  ══════════════════════════════════════════════════════════════")
    print(f"     Calculé sur le sous-graphe ({G_sub.number_of_nodes()} nœuds, "
          f"{G_sub.number_of_edges()} arêtes)")

    cc_dict = bipartite.clustering(G_sub)

    cc_users = [cc_dict[n] for n in sub_user_set]
    cc_books = [cc_dict[n] for n in sub_item_set]
    cc_all = list(cc_dict.values())

    avg_cc = np.mean(cc_all)
    avg_cc_users = np.mean(cc_users) if cc_users else 0
    avg_cc_books = np.mean(cc_books) if cc_books else 0

    print(f"\n  ── Résultats ──")
    print(f"     Clustering moyen global        : {avg_cc:.4f}")
    print(f"     Clustering moyen (users seuls) : {avg_cc_users:.4f}")
    print(f"     Clustering moyen (books seuls) : {avg_cc_books:.4f}")

    print(f"\n     Détail par utilisateur :")
    print(f"       min={min(cc_users):.4f}, médiane={np.median(cc_users):.4f}, "
          f"max={max(cc_users):.4f}")
    print(f"     Détail par livre :")
    print(f"       min={min(cc_books):.4f}, médiane={np.median(cc_books):.4f}, "
          f"max={max(cc_books):.4f}")

    print(f"\n  ── Interprétation ──")
    if avg_cc > 0.5:
        print(f"     → Clustering ÉLEVÉ ({avg_cc:.2f}) : les voisins d'un nœud tendent")
        print(f"       à être eux-mêmes connectés. Les power users lisent les mêmes")
        print(f"       best-sellers → forte structure de co-lecture, favorable au")
        print(f"       filtrage collaboratif.")
    elif avg_cc > 0.2:
        print(f"     → Clustering MODÉRÉ ({avg_cc:.2f}) : structure de co-lecture présente")
        print(f"       mais pas dominante. Mix de goûts mainstream et éclectiques.")
    else:
        print(f"     → Clustering FAIBLE ({avg_cc:.2f}) : les voisins d'un nœud sont")
        print(f"       rarement connectés entre eux. Profils diversifiés, peu de")
        print(f"       « communautés de lecture » clairement définies dans ce sous-graphe.")

    # ==================================================================
    # 4) COMPOSANTES CONNEXES
    # ==================================================================

    print(f"\n  ══════════════════════════════════════════════════════════════")
    print(f"  4) COMPOSANTES CONNEXES")
    print(f"  ══════════════════════════════════════════════════════════════")

    # ── Graphe complet ────────────────────────────────────────────────

    components_full = list(nx.connected_components(G))
    n_comp_full = len(components_full)
    sizes_full = sorted([len(c) for c in components_full], reverse=True)
    largest_full = sizes_full[0]

    print(f"\n  ── Graphe complet ({G.number_of_nodes():,} nœuds) ──")
    print(f"     Composantes connexes           : {n_comp_full:,}")
    print(f"     Taille de la plus grande       : {largest_full:,} nœuds "
          f"({largest_full / G.number_of_nodes() * 100:.1f}%)")

    if n_comp_full > 1:
        print(f"     Tailles des 5 plus grandes     : {sizes_full[:5]}")
        n_isolated = sum(1 for s in sizes_full if s == 1)
        n_small = sum(1 for s in sizes_full if s <= 5)
        print(f"     Nœuds isolés (taille = 1)      : {n_isolated:,}")
        print(f"     Petites composantes (≤ 5)      : {n_small:,}")
    else:
        print(f"     → Le graphe est ENTIÈREMENT CONNEXE (une seule composante).")
        print(f"       Tout utilisateur est relié à tout livre par un chemin.")

    # ── Sous-graphe ───────────────────────────────────────────────────

    components_sub = list(nx.connected_components(G_sub))
    n_comp_sub = len(components_sub)
    sizes_sub = sorted([len(c) for c in components_sub], reverse=True)

    print(f"\n  ── Sous-graphe réduit ({G_sub.number_of_nodes()} nœuds) ──")
    print(f"     Composantes connexes           : {n_comp_sub:,}")
    print(f"     Taille de la plus grande       : {sizes_sub[0]:,} nœuds")

    if n_comp_sub > 1:
        print(f"     Tailles des composantes        : {sizes_sub}")
    else:
        print(f"     → Le sous-graphe est ENTIÈREMENT CONNEXE.")

    print(f"\n  ── Interprétation ──")
    print(f"     • Un graphe entièrement connexe signifie que le filtrage")
    print(f"       collaboratif peut potentiellement recommander n'importe quel")
    print(f"       livre à n'importe quel utilisateur (via des voisins transitifs).")
    print(f"     • Les composantes déconnectées créent des « silos » :")
    print(f"       on ne peut pas recommander entre composantes.")
    print(f"     • Les petites composantes correspondent souvent à des niches")
    print(f"       ultra-spécialisées ou à du cold start (nouveaux users/livres).")

print(f"\n✓ Analyse clustering et composantes terminée.")

### 5) Analyse et Interprétation

- What do these metrics reveal about reader behavior?
- How could this information improve recommendations?
- Compare with expected properties of a social network.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# 5) ANALYSE ET INTERPRÉTATION DU GRAPHE BIPARTI
#
# Discussion structurée des métriques du graphe en relation avec le
# comportement des lecteurs et les systèmes de recommandation.
# ══════════════════════════════════════════════════════════════════════════

print("╔══════════════════════════════════════════════════════════════════════╗")
print("║  5) ANALYSE ET INTERPRÉTATION DU GRAPHE BIPARTI                    ║")
print("╚══════════════════════════════════════════════════════════════════════╝")

print("""
  ══════════════════════════════════════════════════════════════════════
  A) CE QUE LES MÉTRIQUES RÉVÈLENT SUR LE COMPORTEMENT DES LECTEURS
  ══════════════════════════════════════════════════════════════════════

  1. Distribution des degrés en loi de puissance :

     La distribution des degrés (users et books) suit approximativement
     une loi de puissance P(k) ∝ k^{-γ}. Cela signifie :

     - Quelques « super-lecteurs » notent des centaines de livres
       (hubs du réseau). Ce sont les power users qui alimentent le
       système de recommandation en signal exploitable.
     - La majorité des utilisateurs n'ont noté que 10–30 livres
       (longue queue). Leurs profils sont épars → recommandation
       plus difficile, mais c'est justement pour eux que le système
       a le plus de valeur.
     - Quelques best-sellers sont notés par des milliers d'utilisateurs.
       Ils dominent la popularité mais n'apportent pas de signal
       discriminant (tout le monde les a lus).
     - La majorité des livres n'ont que 5–20 notes (long tail du
       catalogue). C'est là que la recommandation a le plus d'impact :
       faire découvrir des livres de niche pertinents.

  2. Densité très faible (< 0.2%) :

     Chaque utilisateur n'a noté qu'une infime fraction du catalogue.
     Implications :
     - Le filtrage collaboratif repose sur très peu d'observations
       par paire d'utilisateurs → les similarités sont bruitées.
     - Les approches par factorisation matricielle (SVD, NMF, ALS)
       sont mieux adaptées que les méthodes de voisinage direct
       pour extrapoler dans les zones vides de la matrice.

  3. Composante connexe géante :

     Si le graphe est (quasi-)connexe, il existe un chemin entre
     presque n'importe quelle paire (user, book). Cela valide la
     faisabilité du filtrage collaboratif : les préférences se
     « propagent » dans le réseau via les voisins communs. Les
     éventuelles petites composantes isolées correspondent à des
     niches ultra-spécialisées ou à du cold start.

  4. Clustering dans le sous-graphe :

     Les power users et les best-sellers forment un noyau dense
     où les voisins partagent beaucoup de livres. C'est le
     « mainstream core » du réseau : les recommandations y sont
     faciles et fiables. En dehors de ce noyau, les connexions
     sont plus éparses → la qualité se dégrade.

  ══════════════════════════════════════════════════════════════════════
  B) COMMENT UTILISER CES INFORMATIONS POUR AMÉLIORER LES
     RECOMMANDATIONS
  ══════════════════════════════════════════════════════════════════════

  1. Livres centraux (hubs) comme recommandations de cold start :
     Pour un nouvel utilisateur sans historique, recommander les livres
     à haute centralité est raisonnable : ce sont les livres les plus
     « connecteurs » du réseau, statistiquement les plus susceptibles
     de plaire.

  2. Composantes pour la segmentation :
     Si le graphe a des composantes ou des communautés distinctes,
     on peut segmenter les utilisateurs par « univers de lecture »
     (fiction, science, cuisine...) et entraîner des modèles
     spécialisés par segment plutôt qu'un modèle unique global.

  3. Degré comme pondération de confiance :
     Un utilisateur avec un degré élevé (beaucoup de reviews) fournit
     un signal plus riche → ses similarités sont plus fiables. On
     peut pondérer sa contribution dans les prédictions par son degré
     (ou log(deg)) pour donner plus de poids aux profils riches.

  4. Clustering pour la diversification :
     Un utilisateur dans un cluster dense reçoit naturellement des
     recommandations redondantes (tous ses voisins lisent les mêmes
     livres). On peut utiliser le clustering pour détecter ce cas et
     intentionnellement diversifier en allant chercher des livres
     hors du cluster.

  ══════════════════════════════════════════════════════════════════════
  C) COMPARAISON AVEC LES PROPRIÉTÉS ATTENDUES D'UN RÉSEAU SOCIAL
  ══════════════════════════════════════════════════════════════════════

  Les réseaux sociaux classiques (Facebook, Twitter) ont des propriétés
  bien étudiées. Comparons :

  ┌──────────────────────┬───────────────────┬──────────────────────┐
  │ Propriété            │ Réseau social     │ Notre graphe biparti │
  ├──────────────────────┼───────────────────┼──────────────────────┤
  │ Type de graphe       │ Uniparti          │ Biparti (U, I)       │
  │ Distribution degrés  │ Power law (γ≈2-3) │ Power law (similaire)│
  │ Composante géante    │ > 90% des nœuds   │ ~100% (après filtrage│
  │                      │                   │ des profils rares)   │
  │ Clustering           │ Élevé (0.1–0.5)   │ Variable (biparti =  │
  │                      │                   │ pas de triangles     │
  │                      │                   │ classiques)          │
  │ Petit monde          │ ≤ 6 sauts         │ Attendu ~4-8 sauts   │
  │ (diamètre)           │                   │ (user→book→user→...) │
  │ Densité              │ Faible (<0.01%)   │ Très faible (<0.2%)  │
  │ Assortativité        │ Positive          │ Mixte (biparti)      │
  └──────────────────────┴───────────────────┴──────────────────────┘

  Points communs :
    - Distribution « scale-free » avec quelques hubs et une longue queue.
    - Composante connexe géante → le réseau est fonctionnel pour la
      propagation d'information (recommandations).
    - Faible densité → les connexions sont sélectives, pas aléatoires.

  Différences clés :
    - Bipartition stricte → pas de triangles classiques, le clustering
      doit être mesuré via des métriques biparties spécifiques.
    - Les « amitiés » sont implicites (co-lecture) plutôt qu'explicites
      → la relation user-user n'existe que via la projection du graphe
      biparti (deux users sont « amis » s'ils ont noté les mêmes livres).
    - L'assortativité est plus complexe dans un graphe biparti : les
      hubs utilisateurs (power readers) tendent à noter les hubs livres
      (best-sellers), ce qui crée un « rich-get-richer » bidirectionnel.
""")

print("✓ Discussion terminée.")

# 3.4 Tâche 3 - Représentation en graphe

## 3.4.1 Construction graphe biparti

In [ ]:
# TODO: Construire graphe biparti avec NetworkX
pass

## 2.2 Analyse du graphe
- Degré moyen
- Densité
- Centralité
- Clustering
- Composantes connexes


In [ ]:
# TODO: Calcul métriques graphe
pass

# Tâche 3 - Regroupement des utilisateurs
## 3.1 K-Means et détermination de K


In [ ]:
# TODO: Appliquer KMeans pour K = 3..8
pass

## 3.2 Analyse des clusters
- Taille
- Centres
- Moyennes
- Top livres
- Visualisation PCA / t-SNE


In [ ]:
# TODO: Analyse clusters + visualisation 2D
pass

# Tâche 4 - Prédiction des évaluations
## 4.1 Baselines


In [ ]:
# TODO: Baseline moyenne globale
# TODO: Baseline moyenne par livre
pass

## 4.2 k-NN collaboratif basé utilisateur


In [ ]:
# TODO: Implémentation k-NN
pass

## 4.3 Analyse des performances
- RMSE
- MAE
- Temps d'exécution


In [ ]:
# TODO: Tableau comparatif
pass

# Tâche 5 - Discussion et analyse critique
- Synthèse des résultats
- Limitations
- Défis de volumétrie
- Perspectives d'amélioration
